## Cellule 1 — Installation

In [1]:
%pip install -q amplpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 27.6 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, sys

# ---- UUID depuis les Secrets Kaggle -----------------------------------------
UUID = None
try:
    from kaggle_secrets import UserSecretsClient
    UUID = UserSecretsClient().get_secret("AMPL_LICENSE_UUID")
    print("licence lue depuis les Secrets Kaggle")
except Exception as exc:
    print(f"Secrets Kaggle indisponibles ({type(exc).__name__}).")
    UUID = os.environ.get("AMPL_LICENSE_UUID")
    if UUID:
        print("licence lue depuis la variable d'environnement AMPL_LICENSE_UUID")

if not UUID:
    raise RuntimeError(
        "AMPL_LICENSE_UUID introuvable.\n"
        "  Kaggle : Add-ons -> Secrets -> Add a new secret\n"
        "           Label = AMPL_LICENSE_UUID, Value = ton UUID AMPL\n"
        "           puis coche la case pour l'attacher a ce notebook.\n"
        "  Ne colle jamais l'UUID directement dans le notebook.")

# ---- modules AMPL ------------------------------------------------------------
from amplpy import AMPL, modules
modules.install(["ampl", "cplex"])          # noyau AMPL + solveur CPLEX
modules.activate(UUID)

ampl = AMPL()
ampl.option["solver"] = "cplex"
print("AMPL initialise |", ampl.get_option("version"))
print("solveur         |", ampl.option["solver"])

licence lue depuis les Secrets Kaggle
AMPL initialise | AMPL Version 20260809 (Linux-6.17.0-1020-azure, 64-bit)
Licensed to AMPL for Academics - azza jhidri <ru846414@dal.ca>.
Temporary license expires 20260924.
Using license file "/usr/local/lib/python3.12/dist-packages/ampl_module_base/bin/ampl.lic".

solveur         | cplex


## Cellule 3 — Écriture de `model.mod`

Le modèle complet. Chaque bloc de contraintes porte **le numéro de l'équation du
PDF** qu'il code, et les blocs exclus disent pourquoi ils le sont.

In [3]:
MODEL_MOD = r'''
# =============================================================================
#  model.mod  --  Dynamic Newsvendor-Based Production and Deterioration-Aware
#                 Inventory Control.  Modele deterministe des Eqs (9)-(43) de
#                 sujet.pdf, sections 3.2 et 3.3.
#
#  PERIMETRE.  Ce fichier code UNIQUEMENT le modele mathematique. Aucun element
#  d'apprentissage par renforcement n'y figure : ni Eq (46), ni Eq (47), ni la
#  section 3.4, ni CVaR/DRO.
#
#  HORS PERIMETRE, ET POURQUOI  (detail dans AUDIT_equations.md) :
#    Eqs (2)-(7)   repere newsvendor classique. Le PDF dit lui-meme qu'il "is not
#                  imposed on the learned policy". L'imposer a CPLEX reviendrait a
#                  recopier une heuristique. Calcule APRES la resolution.
#    Eqs (32)-(34) declencheur, validite du plan, affectation executee. NON codees
#                  dans le MILP : elles sont pilotees en Python AUTOUR du modele
#                  (cellule 9c). Une fenetre MILP = un PLAN ENGAGE ; le declencheur
#                  et la reparation sont des evenements d'EXECUTION, qui dependent
#                  d'informations non disponibles au moment ou la fenetre est
#                  resolue. Les coder dans le MILP reviendrait a donner au solveur
#                  la connaissance de la perturbation future : exactement la fuite
#                  que la version corrigee elimine.
#    Eqs (44)-(45) temps de recuperation. Metriques post-resolution.
#
#  CONVENTIONS D'INDICES (identiques au PDF) :
#    g = 1..G          produit
#    j = 1..Jg[g]      stade de traitement, ordonne
#    a = 0..abar[g,j]  classe d'age. a GRAND = PLUS VIEUX (= perime en premier)
#    e = 1..E          machine
#    n = 1..N          epoque de revue
#    h = 1..H          intervalle d'execution
#
#  AJOUTS DE LA VERSION CORRIGEE (marques [C1]..[C3]). Tous sont NEUTRES par
#  defaut : sans les nouvelles donnees, le modele se comporte exactement comme
#  la version validee.
#    [C1] PrevAsg  report de l'affectation executee d'une fenetre a la suivante
#    [C2] REPAIR   fenetre de reparation intra-epoque (V deja engage)
#    [C3] FixAsg   epinglage d'affectations, reserve aux tests
# =============================================================================

# -----------------------------------------------------------------------------
#  ENSEMBLES ET INDICES  (PDF section 3.2)
# -----------------------------------------------------------------------------
param G  > 0 integer;               # |script G|      : nombre de produits
param E  > 0 integer;               # nombre de machines
param N  > 0 integer;               # |script N|      : epoques de revue
param H  > 0 integer;               # |script H_n|    : intervalles par epoque

set PRODUCTS  := 1..G;
set EQUIP     := 1..E;
set EPOCHS    := 1..N;
set INTERVALS := 1..H;

# Ensembles plats, entierement fournis par data.dat. Cette forme evite les
# ensembles indexes imbriques et rend le fichier de donnees generable sans risque.
set GJ  dimen 2;                    # (g,j)     : stades de g,  j dans script J_g
set GJA dimen 3;                    # (g,j,a)   : classes d'age, a dans script A_gj
set GJE dimen 3;                    # (g,j,e)   : e dans script E_gj  (machines eligibles)
set GE  dimen 2;                    # (g,e)     : e peut traiter g a un stade unique
set GGE dimen 3;                    # (g,gp,e)  : changement de gp vers g possible sur e

set GA  dimen 2;                    # (g,a)     : classes d'age du STADE FINAL de g
param AMAX >= 0 integer;            # plus grande classe d'age, tous produits confondus

param jf{PRODUCTS} integer;         # J_g : stade final du produit g
param abar{GJ}     integer;         # abar_gj : classe d'age maximale admissible

# -----------------------------------------------------------------------------
#  PARAMETRES ECONOMIQUES  (PDF section 3.2)
# -----------------------------------------------------------------------------
param mu{GJE}      >= 0;            # mu_gje    : cout unitaire de traitement
param alpha{PRODUCTS, 0..AMAX} >= 0; # alpha_ga  : cout de possession par classe d'age
param beta{PRODUCTS}   >= 0;        # beta_g    : penalite par unite de demande non servie
param rho{PRODUCTS, EPOCHS} >= 0;   # rho_gn    : revenu par unite servie
param delta{GJ}    >= 0;            # delta_gj  : cout de rebut / destruction
param kappa{GGE}   >= 0;            # kappa_gg'e: cout de changement de serie
param eta_idle     >= 0;            # eta_idle  : poids de la capacite inutilisee, Eq (42)
param eta_serv     >= 0;            # eta_serv  : poids de la penalite de service, Eq (43)

# -----------------------------------------------------------------------------
#  PARAMETRES DE CAPACITE ET DE PROCEDE
# -----------------------------------------------------------------------------
param tau{GJE}     >  0;            # tau_gje   : temps unitaire de traitement
param Cbar{EQUIP, INTERVALS, EPOCHS} >= 0;   # Cbar_ehn : capacite nominale
param avail{EQUIP, INTERVALS, EPOCHS} binary;# a_ehn    : disponibilite (scenario)
param upsilon{GGE} >= 0;            # upsilon_gg'e : temps de changement de serie
param Rbar{INTERVALS, EPOCHS} >= 0; # Rbar_hn   : ressource partagee disponible
param Wcap{INTERVALS, EPOCHS} >= 0 integer;  # floor(Wbar_hn) : main-d'oeuvre
param psi{GJE}     >= 0;            # psi_gje   : consommation de ressource partagee
param ell{PRODUCTS}  >= 0;          # ell_g     : lot minimal de lancement
param umax{PRODUCTS} >= 0;          # u_g       : lancement maximal
param mfrak{PRODUCTS, EPOCHS} >= 0; # m_gn      : materiel d'entree disponible
param Delta{INTERVALS, EPOCHS} > 0; # Delta_hn  : duree de l'intervalle

# -----------------------------------------------------------------------------
#  DETERIORATION  --  Eqs (10)-(11) PRECALCULEES
#
#  lambda_gjahn = lam0 + lam1*a + lam2*Temp + lam3*Hum + lam4*Dev          Eq (10)
#  Gamma_gjahn  = exp(-lambda * Delta_hn),  tronquee dans [0,1]            Eq (11)
#
#  Gam est fourni comme TABLEAU NUMERIQUE : aucune exponentielle ne subsiste
#  dans le modele, conformement a la consigne. Le calcul est fait en NumPy dans
#  la cellule 4 du notebook, a partir des conditions de stockage OBSERVEES
#  (mode rolling horizon) ou du scenario complet (mode oracle).
# -----------------------------------------------------------------------------
param Gam{GJA, INTERVALS, EPOCHS} >= 0, <= 1;

# -----------------------------------------------------------------------------
#  DEMANDE ET CONDITIONS INITIALES
#
#  D est la demande utilisee par la resolution :
#     - mode "rolling horizon"  : mediane de la prevision F_gn (equitable vs PPO)
#     - mode "oracle"           : demande realisee D_gn (information parfaite)
#  Le choix est fait dans la cellule 4 et ecrit dans data.dat.
#
#  ATTENTION : le PDF ne donne AUCUNE condition initiale. S0 et L0 sont repris de
#  Env.reset du notebook PPO et signales comme tels dans l'audit.
# -----------------------------------------------------------------------------
param D{PRODUCTS, EPOCHS}  >= 0;    # D_gn
param S0{GJA}              >= 0;    # S_{gja,h=1,n=1}
param L0{PRODUCTS}         >= 0;    # L_{g,0}
param SLmin{PRODUCTS} >= 0, <= 1;   # SL^min_g, cible de service
param eps0 > 0;                     # epsilon_0 de l'Eq (29)

# -----------------------------------------------------------------------------
#  [C1]  REPORT DE L'AFFECTATION EXECUTEE  --  Eq (36) au premier intervalle
#
#  Sans ce parametre, chaque nouvelle fenetre rolling horizon "oublie" ce que la
#  machine faisait a la fin de la fenetre precedente : le changement de serie
#  n'est ni facture (kappa) ni impute a la capacite (upsilon). PrevAsg[g,e] = 1
#  signifie : la machine e traitait le produit g a l'intervalle IMMEDIATEMENT
#  precedant (h=1, n=1) de cette fenetre.
#  Valeur par defaut 0 pour TOUS les couples => machine inactive avant la fenetre
#  => aucun changement facture, exactement comme dans la version d'origine.
# -----------------------------------------------------------------------------
param PrevAsg{GE} binary default 0;

# -----------------------------------------------------------------------------
#  [C2]  MODE REPARATION INTRA-EPOQUE  --  Eqs (32)-(34), pilotees en Python
#
#  REPAIR = 1 : la fenetre couvre les intervalles RESTANTS d'une epoque deja
#  commencee. Le lancement V a ete engage au debut de l'epoque et n'est plus une
#  decision ; seul le solde restant K0 peut encore etre traite. Les bornes de
#  l'Eq (9) ont deja ete satisfaites a l'engagement et ne sont donc pas re-imposees
#  (K0 < ell_g est parfaitement normal en cours d'epoque).
# -----------------------------------------------------------------------------
param REPAIR binary default 0;
param K0{PRODUCTS}   >= 0 default 0;   # solde de lancement restant, Eq (21)
param Opn0{PRODUCTS} binary default 0; # O_gn deja engage au debut de l'epoque

# -----------------------------------------------------------------------------
#  [C3]  EPINGLAGE D'AFFECTATIONS  --  reserve au test 3 du notebook
#  FixAsg[g,e,h,n] dans {0,1} epingle Asg ; -1 (defaut) = libre.
# -----------------------------------------------------------------------------
param FixAsg{GE, INTERVALS, EPOCHS} default -1;

# -----------------------------------------------------------------------------
#  CONSTANTES BIG-M  --  RESSERREES, derivees des donnees (audit section 2.5)
#
#  Le PDF fournit un unique "M suffisamment grand" (Mfrak = 1e7 dans le notebook).
#  Utilise tel quel, la relaxation lineaire est inexploitable. On le remplace par :
#     Mp    = Cbar / tau        borne de debit issue de l'Eq (35) seule
#     Mfefo = u_g * H * N       borne de flux cumule sur une classe d'age
#  Facteur de resserrement mesure : environ 50 000.
# -----------------------------------------------------------------------------
param Mp{GJE, INTERVALS, EPOCHS} > 0;
param Mfefo{GJ} > 0;

param ENFORCE_FEFO binary default 1;      # 0 = mesurer le cout de la regle FEFO

# =============================================================================
#  VARIABLES DE DECISION  --  Eqs (40)-(41)
# =============================================================================
var Rel{PRODUCTS, EPOCHS}  >= 0;                    # V_gn   lancement
var Opn{PRODUCTS, EPOCHS}  binary;                  # O_gn   indicateur de lancement
var Kbal{PRODUCTS, INTERVALS, EPOCHS} >= 0;         # K_ghn  solde de lancement
var Prod{GJE, INTERVALS, EPOCHS} >= 0;              # P_gjehn quantite traitee
var Asg{GE, INTERVALS, EPOCHS} binary;              # A_gehn affectation
var Chg{GGE, INTERVALS, EPOCHS} binary;             # C_gg'ehn changement de serie
var Stk{GJA, INTERVALS, EPOCHS} >= 0;               # S_gjahn stock par age
var Wdr{GJA, INTERVALS, EPOCHS} >= 0;               # U_gjahn retrait
var Det{GJA, INTERVALS, EPOCHS} >= 0;               # Q_gjahn quantite deterioree
var Exp{GJ,  INTERVALS, EPOCHS} >= 0;               # X_gjhn  quantite perimee
var Spre{GJA, EPOCHS} >= 0;                         # Stilde_gjan stock pre-demande
var Wdem{GA, EPOCHS} >= 0;                          # U^dem_gan retrait de demande
var Ful{PRODUCTS, EPOCHS}  >= 0;                    # F_gn   demande satisfaite
var Bkl{PRODUCTS, EPOCHS}  >= 0;                    # L_gn   backlog
var Idl{EQUIP, INTERVALS, EPOCHS} >= 0;             # Z_ehn  capacite inutilisee

# --- variables auxiliaires des deux parties positives, Eqs (29) et (43) -------
#  Aucune n'est binaire : voir la justification sous SHORTFALL_DEF.
var DLpos{PRODUCTS, EPOCHS} >= 0;                   # (L_n - L_{n-1})^+
var Short{PRODUCTS, EPOCHS} >= 0;                   # [SL^min - SL]^+

# --- binaires de la regle FEFO ------------------------------------------------
var FullI{GJA, INTERVALS, EPOCHS} binary;           # classe d'age entierement consommee
var FullD{GA, EPOCHS} binary;                       # idem, retraits de demande

# =============================================================================
#  OBJECTIF  --  Eqs (42) et (43)
# =============================================================================
#  Eq (42) : cout operatoire d'une epoque, somme sur toutes les epoques.
#  Eq (43) : J = sum_n G_n + eta_serv * sum [SL^min - SL]^+ .
#            Le terme eta_rec * H^rec est ABSENT de l'objectif : eta_rec est
#            DEFINI SYMBOLIQUEMENT par l'Eq (43) mais sa valeur numerique n'est
#            donnee ni par le PDF ni par le notebook PPO. H^rec est donc calcule
#            APRES la resolution, en sensibilite sur w_conf (Eq 45), et n'entre
#            pas dans le critere optimise. Voir AUDIT_equations.md section 5.
# -----------------------------------------------------------------------------
var CostProcessing    = sum{(g,j,e) in GJE, h in INTERVALS, n in EPOCHS}
                          mu[g,j,e] * Prod[g,j,e,h,n];
var CostChangeover    = sum{(g,gp,e) in GGE, h in INTERVALS, n in EPOCHS}
                          kappa[g,gp,e] * Chg[g,gp,e,h,n];
var CostHolding       = sum{(g,j,a) in GJA, h in INTERVALS, n in EPOCHS}
                          alpha[g,a] * Stk[g,j,a,h,n];
var CostBacklog       = sum{g in PRODUCTS, n in EPOCHS} beta[g] * Bkl[g,n];
var CostDeterioration = sum{(g,j,a) in GJA, h in INTERVALS, n in EPOCHS}
                          delta[g,j] * Det[g,j,a,h,n];
var CostDisposal      = sum{(g,j) in GJ, h in INTERVALS, n in EPOCHS}
                          delta[g,j] * Exp[g,j,h,n];
var Revenue           = sum{g in PRODUCTS, n in EPOCHS} rho[g,n] * Ful[g,n];
param term_value {GJ} default 0;   # valeur terminale du stock en fin de fenetre
var TerminalValue =
    sum{(g,j,a) in GJA: j < jf[g]} term_value[g,j] * Spre[g,j,a,N]
  + sum{(g,a) in GA} term_value[g,jf[g]] * (Spre[g,jf[g],a,N] - Wdem[g,a,N]);

var CostIdle          = eta_idle * sum{e in EQUIP, h in INTERVALS, n in EPOCHS}
                          Idl[e,h,n];
var ServicePenalty    = eta_serv * sum{g in PRODUCTS, n in EPOCHS} Short[g,n];

minimize TotalCost:
    CostProcessing + CostChangeover + CostHolding + CostBacklog
    + CostDeterioration + CostDisposal - Revenue - TerminalValue + CostIdle + ServicePenalty;

# =============================================================================
#  CONTRAINTES
# =============================================================================

# --- Eq (9) : ensemble admissible de lancement, forme exacte de l'Eq (8) ------
#  V_gn dans {0} U [ell_g, min{u_g, m_gn}] : ensemble NON CONVEXE, rendu exact
#  par le binaire O_gn. Le PDF fournit lui-meme cette formulation.
#  [C2] En mode reparation V n'est plus une decision : les bornes ont deja ete
#  imposees a l'engagement, et le solde restant K0 peut legitimement etre < ell.
subject to RELEASE_LO{g in PRODUCTS, n in EPOCHS: REPAIR = 0}:
    Rel[g,n] >= ell[g] * Opn[g,n];
subject to RELEASE_HI{g in PRODUCTS, n in EPOCHS: REPAIR = 0}:
    Rel[g,n] <= umax[g] * Opn[g,n];
subject to RELEASE_MAT{g in PRODUCTS, n in EPOCHS: REPAIR = 0}:
    Rel[g,n] <= mfrak[g,n];

# --- [C2] Eqs (19)-(21) en mode reparation : V = solde restant ---------------
subject to REPAIR_FIX_REL{g in PRODUCTS: REPAIR = 1}:
    Rel[g,1] = K0[g];
subject to REPAIR_FIX_OPN{g in PRODUCTS: REPAIR = 1}:
    Opn[g,1] = Opn0[g];

# --- [C3] epinglage d'affectations, tests uniquement ------------------------
subject to FIX_ASG{(g,e) in GE, h in INTERVALS, n in EPOCHS: FixAsg[g,e,h,n] >= 0}:
    Asg[g,e,h,n] = FixAsg[g,e,h,n];

# --- Eqs (19)-(21) : solde de lancement du premier stade ---------------------
subject to K_INIT{g in PRODUCTS, n in EPOCHS}:                         # Eq (19)
    Kbal[g,1,n] = Rel[g,n];
subject to K_CAP{g in PRODUCTS, h in INTERVALS, n in EPOCHS}:          # Eq (20)
    sum{(gg,jj,e) in GJE: gg = g and jj = 1} Prod[gg,jj,e,h,n] <= Kbal[g,h,n];
subject to K_BALANCE{g in PRODUCTS, h in INTERVALS, n in EPOCHS: h < H}:# Eq (21)
    Kbal[g,h+1,n] = Kbal[g,h,n]
                  - sum{(gg,jj,e) in GJE: gg = g and jj = 1} Prod[gg,jj,e,h,n];

# --- Eq (12) : quantite deterioree -------------------------------------------
#  Gam etant un parametre, l'egalite est lineaire.
subject to DETERIORATED{(g,j,a) in GJA, h in INTERVALS, n in EPOCHS}:
    Det[g,j,a,h,n] = (1 - Gam[g,j,a,h,n]) * Stk[g,j,a,h,n];

# --- Eq (13) : le retrait ne peut exceder la quantite survivante -------------
subject to WITHDRAW_CAP{(g,j,a) in GJA, h in INTERVALS, n in EPOCHS}:
    Wdr[g,j,a,h,n] <= Gam[g,j,a,h,n] * Stk[g,j,a,h,n];

# --- Eq (14) : aucun retrait du stade final pendant les intervalles ----------
subject to NO_FINAL_WITHDRAW{(g,j,a) in GJA, h in INTERVALS, n in EPOCHS: j = jf[g]}:
    Wdr[g,j,a,h,n] = 0;

# --- Eq (15) : le materiel fraichement traite entre en classe d'age 0 --------
subject to AGE_ZERO{(g,j) in GJ, h in INTERVALS, n in EPOCHS: h < H}:
    Stk[g,j,0,h+1,n] = sum{(gg,jj,e) in GJE: gg = g and jj = j} Prod[gg,jj,e,h,n];

# --- Eq (16) : vieillissement de ce qui survit et n'est pas retire -----------
subject to AGE_ADVANCE{(g,j,a) in GJA, h in INTERVALS, n in EPOCHS:
                       h < H and a < abar[g,j]}:
    Stk[g,j,a+1,h+1,n] = Gam[g,j,a,h,n] * Stk[g,j,a,h,n] - Wdr[g,j,a,h,n];

# --- Eq (17) : peremption au-dela de la classe d'age maximale ----------------
subject to EXPIRY{(g,j) in GJ, h in INTERVALS, n in EPOCHS}:
    Exp[g,j,h,n] = Gam[g,j,abar[g,j],h,n] * Stk[g,j,abar[g,j],h,n]
                 - Wdr[g,j,abar[g,j],h,n];

# --- Eq (18) : conservation des flux entre stades consecutifs ----------------
subject to INTERSTAGE{(g,j) in GJ, h in INTERVALS, n in EPOCHS: j >= 2}:
    sum{(gg,jj,e) in GJE: gg = g and jj = j} Prod[gg,jj,e,h,n]
  = sum{(gg,jj,a) in GJA: gg = g and jj = j-1} Wdr[gg,jj,a,h,n];

# --- Eqs (22)-(23) : stock pre-demande a l'intervalle terminal ---------------
subject to TILDE_AGE_ZERO{(g,j) in GJ, n in EPOCHS}:                   # Eq (22)
    Spre[g,j,0,n] = sum{(gg,jj,e) in GJE: gg = g and jj = j} Prod[gg,jj,e,H,n];
subject to TILDE_AGE_ADV{(g,j,a) in GJA, n in EPOCHS: a < abar[g,j]}:  # Eq (23)
    Spre[g,j,a+1,n] = Gam[g,j,a,H,n] * Stk[g,j,a,H,n] - Wdr[g,j,a,H,n];

# --- Eq (24) : report des stades intermediaires vers l'epoque suivante -------
subject to CARRY_INTER{(g,j,a) in GJA, n in EPOCHS: n < N and j < jf[g]}:
    Stk[g,j,a,1,n+1] = Spre[g,j,a,n];

# --- Eq (25) : satisfaction de la demande, le "min" relaxe en deux bornes ----
#  RELAXATION JUSTIFIEE, pas une egalite. Augmenter F d'une unite fait varier
#  l'objectif de -rho (revenu) - beta (backlog) - (holding + peremption evites),
#  soit une quantite strictement negative avec rho = 10 et beta = 5. L'optimum
#  sature donc toujours l'une des deux bornes et le min est atteint.
#  La cellule 7 VERIFIE numeriquement F = min{.,.} apres chaque resolution.
subject to FULFIL_DEM{g in PRODUCTS, n in EPOCHS}:
    Ful[g,n] <= (if n = 1 then L0[g] else Bkl[g,n-1]) + D[g,n];
subject to FULFIL_STK{g in PRODUCTS, n in EPOCHS}:
    Ful[g,n] <= sum{(gg,a) in GA: gg = g} Spre[gg,jf[gg],a,n];

# --- Eq (26) : retraits de demande, par classe d'age -------------------------
subject to DEM_WITHDRAW{g in PRODUCTS, n in EPOCHS}:
    sum{(gg,a) in GA: gg = g} Wdem[gg,a,n] = Ful[g,n];
subject to DEM_CAP{(g,a) in GA, n in EPOCHS}:
    Wdem[g,a,n] <= Spre[g,jf[g],a,n];

# --- Eq (27) : report du stade final apres prelevement de la demande ---------
subject to CARRY_FINAL{(g,a) in GA, n in EPOCHS: n < N}:
    Stk[g,jf[g],a,1,n+1] = Spre[g,jf[g],a,n] - Wdem[g,a,n];

# --- Eq (28) : bilan de backlog ----------------------------------------------
subject to BACKLOG{g in PRODUCTS, n in EPOCHS}:
    Bkl[g,n] = (if n = 1 then L0[g] else Bkl[g,n-1]) + D[g,n] - Ful[g,n];

# --- Eqs (29) + (43) : les deux parties positives, SANS aucun binaire --------
#  SL_gn = 1 - (L_n - L_{n-1})^+ / (D + eps0)                            Eq (29)
#  penalite = eta_serv * [SL^min - SL]^+                                 Eq (43)
#
#  EXACTITUDE. DLpos n'apparait que dans SHORTFALL_DEF avec un coefficient
#  1/(D+eps0) > 0, et Short n'apparait que dans l'objectif avec eta_serv > 0.
#  Dans une MINIMISATION, les deux prennent donc leur plus petite valeur
#  admissible, c'est-a-dire exactement max(0, .). Aucune approximation.
subject to DLPOS_DEF{g in PRODUCTS, n in EPOCHS}:
    DLpos[g,n] >= Bkl[g,n] - (if n = 1 then L0[g] else Bkl[g,n-1]);
subject to SHORTFALL_DEF{g in PRODUCTS, n in EPOCHS}:
    Short[g,n] >= SLmin[g] - 1 + DLpos[g,n] / (D[g,n] + eps0);

# --- Eq (30) : lien entre quantite traitee et affectation -------------------
#  Mp est resserre (Cbar/tau), PAS le Mfrak = 1e7 du notebook.
subject to LINK_PA{(g,j,e) in GJE, h in INTERVALS, n in EPOCHS}:
    Prod[g,j,e,h,n] <= Mp[g,j,e,h,n] * Asg[g,e,h,n];

# --- Eq (31) : au plus un produit par machine, et machine disponible --------
subject to ONE_ITEM{e in EQUIP, h in INTERVALS, n in EPOCHS}:
    sum{(gg,ee) in GE: ee = e} Asg[gg,ee,h,n] <= avail[e,h,n];

# --- Eq (35) : capacite machine, traitement + changement de serie -----------
subject to CAPACITY{e in EQUIP, h in INTERVALS, n in EPOCHS}:
    sum{(gg,jj,ee) in GJE: ee = e} tau[gg,jj,ee] * Prod[gg,jj,ee,h,n]
  + sum{(gg,gp,ee) in GGE: ee = e} upsilon[gg,gp,ee] * Chg[gg,gp,ee,h,n]
  <= Cbar[e,h,n] * avail[e,h,n];

# --- Eq (36) : activation du changement de serie ----------------------------
#  (h-, n-) est l'intervalle precedent sur l'horizon APLATI :
#     h > 1          ->  (h-1, n)
#     h = 1, n > 1   ->  (H, n-1)
#     h = 1, n = 1   ->  [C1] l'intervalle precedent appartient a la FENETRE
#                       PRECEDENTE : il est fourni par PrevAsg.
#  Un intervalle a vide brise la chaine : si l'une des deux affectations vaut 0,
#  le membre de droite est <= 0 et aucun changement n'est facture. Conforme au
#  texte du PDF (p. 10) et a _project_throughput du notebook.
subject to CHANGEOVER_IN{(g,gp,e) in GGE, h in INTERVALS, n in EPOCHS: h >= 2}:
    Chg[g,gp,e,h,n] >= Asg[g,e,h,n] + Asg[gp,e,h-1,n] - 1;
subject to CHANGEOVER_CROSS{(g,gp,e) in GGE, n in EPOCHS: n >= 2}:
    Chg[g,gp,e,1,n] >= Asg[g,e,1,n] + Asg[gp,e,H,n-1] - 1;
subject to CHANGEOVER_PREV{(g,gp,e) in GGE}:                            # [C1]
    Chg[g,gp,e,1,1] >= Asg[g,e,1,1] + PrevAsg[gp,e] - 1;

# --- Eq (37) : ressource partagee -------------------------------------------
subject to SHARED_RES{h in INTERVALS, n in EPOCHS}:
    sum{(gg,jj,ee) in GJE} psi[gg,jj,ee] * Prod[gg,jj,ee,h,n] <= Rbar[h,n];

# --- Eq (38) : main-d'oeuvre -------------------------------------------------
#  Wcap = floor(Wbar) : Wbar est reel dans les scenarios mais sum(A) est entier.
#  Le notebook applique le meme floor dans _project_throughput.
subject to WORKFORCE{h in INTERVALS, n in EPOCHS}:
    sum{(gg,ee) in GE} Asg[gg,ee,h,n] <= Wcap[h,n];

# --- Eq (39) : capacite inutilisee ------------------------------------------
subject to IDLE_CAP{e in EQUIP, h in INTERVALS, n in EPOCHS}:
    Idl[e,h,n] = Cbar[e,h,n] * avail[e,h,n]
               - sum{(gg,jj,ee) in GJE: ee = e} tau[gg,jj,ee] * Prod[gg,jj,ee,h,n]
               - sum{(gg,gp,ee) in GGE: ee = e} upsilon[gg,gp,ee] * Chg[gg,gp,ee,h,n];

# --- Regle FEFO  --  Eqs (18) et (26) ---------------------------------------
#  La FEFO est une REGLE D'EXECUTION, pas un optimum : avec alpha constant et un
#  delta identique pour la deterioration et la peremption, rien dans l'objectif
#  ne force a retirer le plus vieux d'abord. Elle doit donc etre IMPOSEE.
#
#  FullI[g,j,a,h,n] = 1  <=>  la classe d'age a est entierement consommee.
#  On ne peut retirer de la classe a que si la classe a+1 (PLUS VIEILLE) est
#  entierement consommee.
subject to FEFO_FULL{(g,j,a) in GJA, h in INTERVALS, n in EPOCHS:
                     ENFORCE_FEFO = 1 and j < jf[g]}:
    Gam[g,j,a,h,n] * Stk[g,j,a,h,n] - Wdr[g,j,a,h,n]
        <= Mfefo[g,j] * (1 - FullI[g,j,a,h,n]);
subject to FEFO_ORDER{(g,j,a) in GJA, h in INTERVALS, n in EPOCHS:
                      ENFORCE_FEFO = 1 and j < jf[g] and a < abar[g,j]}:
    Wdr[g,j,a,h,n] <= Mfefo[g,j] * FullI[g,j,a+1,h,n];

subject to FEFO_DEM_FULL{(g,a) in GA, n in EPOCHS: ENFORCE_FEFO = 1}:
    Spre[g,jf[g],a,n] - Wdem[g,a,n] <= Mfefo[g,jf[g]] * (1 - FullD[g,a,n]);
subject to FEFO_DEM_ORDER{(g,a) in GA, n in EPOCHS:
                          ENFORCE_FEFO = 1 and a < abar[g,jf[g]]}:
    Wdem[g,a,n] <= Mfefo[g,jf[g]] * FullD[g,a+1,n];

# --- Conditions initiales  (ABSENTES DU PDF, reprises de Env.reset) ---------
subject to INIT_STOCK{(g,j,a) in GJA}:
    Stk[g,j,a,1,1] = S0[g,j,a];
'''

with open('model.mod', 'w') as f:
    f.write(MODEL_MOD)
print(f'model.mod ecrit : {len(MODEL_MOD.splitlines())} lignes')
print('   ajouts de la version corrigee, tous NEUTRES par defaut :')
print('     [C1] param PrevAsg{GE} binary default 0   + CHANGEOVER_PREV')
print('     [C2] param REPAIR / K0 / Opn0             + REPAIR_FIX_REL/OPN')
print('     [C3] param FixAsg{GE,INTERVALS,EPOCHS}    + FIX_ASG   (tests seulement)')

model.mod ecrit : 421 lignes
   ajouts de la version corrigee, tous NEUTRES par defaut :
     [C1] param PrevAsg{GE} binary default 0   + CHANGEOVER_PREV
     [C2] param REPAIR / K0 / Opn0             + REPAIR_FIX_REL/OPN
     [C3] param FixAsg{GE,INTERVALS,EPOCHS}    + FIX_ASG   (tests seulement)


In [4]:
# Imports. Aucun torch : ce notebook ne contient aucun apprentissage.
import math, time, json
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional, Sequence

import numpy as np
import pandas as pd
from scipy.stats import gamma as gamma_dist

np.set_printoptions(precision=3, suppress=True, linewidth=140)
pd.set_option("display.width", 180)
pd.set_option("display.precision", 3)

EPS_NUM = 1e-9
RUNTIME_CHECKS = True
try:
    display
except NameError:
    display = print


@dataclass
class Instance:
    """Sets and indices of §3.2."""
    G: int                          # |G|  number of products
    Jg: np.ndarray                  # (G,)  number of stages J_g per product
    E: int                          # |E|  number of equipment units
    stage_of: np.ndarray            # (G,E) stage served by e for product g, -1 if unused
    eligible: np.ndarray            # (G,E) bool: derived from stage_of >= 0
    abar: np.ndarray                # (G,Jmax) max admissible age class \bar a_{gj}
    N: int                          # |N|  review epochs
    H: int                          # |H_n| execution intervals per epoch (constant here)

    def __post_init__(self):
        self.Jmax = int(self.Jg.max())
        self.Amax = int(self.abar.max())
        self.A = self.Amax + 1                                   # age classes 0..Amax
        self.Upsilon = self.N * self.H                           # Upsilon, Eq (47)

        # valid_gj[g,j]  : stage j exists for product g
        self.valid_gj = np.zeros((self.G, self.Jmax), dtype=bool)
        for g in range(self.G):
            self.valid_gj[g, : self.Jg[g]] = True

        # valid_gja[g,j,a] : age class a is admissible for (g,j)
        self.valid_gja = np.zeros((self.G, self.Jmax, self.A), dtype=bool)
        for g in range(self.G):
            for j in range(self.Jg[g]):
                self.valid_gja[g, j, : self.abar[g, j] + 1] = True

        # final stage index of each product (paper's J_g)
        self.final_stage = self.Jg - 1

        # E_{gj} as a boolean mask [G, Jmax, E]
        self.eligible = self.stage_of >= 0
        self.Egj = np.zeros((self.G, self.Jmax, self.E), dtype=bool)
        for g in range(self.G):
            for e in range(self.E):
                j = int(self.stage_of[g, e])
                if j >= 0:
                    assert j < self.Jg[g], (
                        f"equipment {e} serves stage {j} which product {g} does not have")
                    self.Egj[g, j, e] = True

        # per-equipment option list for the micro head: index 0 = idle, 1..G = product g-1
        self.equip_options = [
            np.array([True] + [bool(self.eligible[g, e]) for g in range(self.G)])
            for e in range(self.E)
        ]

    def iota_to_nh(self, iota: int) -> Tuple[int, int]:
        """Eq (48): flattened index iota -> (n, h), both 0-indexed here."""
        return iota // self.H, iota % self.H

    def nh_to_iota(self, n: int, h: int) -> int:
        """Eq (47)-(48) inverse."""
        return n * self.H + h


@dataclass
class Params:
    """Parameters of §3.2. Shapes are stated per field."""
    # --- economic ---------------------------------------------------------
    mu: np.ndarray            # (G,Jmax,E)   unit processing cost      mu_{gje}
    alpha: np.ndarray         # (G,A)        carrying cost by age      alpha_{ga}
    beta: np.ndarray          # (G,)         unmet-demand penalty      beta_g
    rho: np.ndarray           # (G,N)        service value / revenue   rho_{gn}
    delta: np.ndarray         # (G,Jmax)     disposal / write-off      delta_{gj}
    sigma: np.ndarray         # (G,Jmax)     salvage / rework value    sigma_{gj}
    kappa: np.ndarray         # (G,G,E)      changeover cost g'->g     kappa_{gg'e}
    theta_service: np.ndarray # (G,)         backlog sensitivity       theta^service_g
    eta_idle: float           # idle-capacity cost                     eta_idle
    eta_serv: float           # service-shortfall penalty              eta_serv
    eta_viol: float           # feasibility-correction penalty         eta_viol

    # --- capacity & process ----------------------------------------------
    tau: np.ndarray           # (G,Jmax,E)   processing time / unit    tau_{gje}
    Cbar: np.ndarray          # (E,H,N)      nominal capacity          \bar C_{ehn}
    upsilon: np.ndarray       # (G,G,E)      setup time g'->g          upsilon_{gg'e}
    psi: np.ndarray           # (G,Jmax,E)   shared-resource rate      psi_{gje}
    ell: np.ndarray           # (G,)         min release               ell_g
    umax: np.ndarray          # (G,)         max release               u_g
    Mfrak: float              # big-M                                  \mathfrak{M}

    # --- deterioration ----------------------------------------------------
    lam0: np.ndarray          # (G,Jmax)     baseline hazard           lambda^0_{gj}
    lam1: np.ndarray          # (G,Jmax)     age sensitivity           lambda^1_{gj}
    lam2: np.ndarray          # (G,Jmax)     temperature sensitivity   lambda^2_{gj}
    lam3: np.ndarray          # (G,Jmax)     humidity sensitivity      lambda^3_{gj}
    lam4: np.ndarray          # (G,Jmax)     storage-deviation sens.   lambda^4_{gj}
    omega: np.ndarray         # (G,Jmax)     readiness coefficient     omega_{gj} in [0,1]
    Delta: np.ndarray         # (H,N)        interval duration         Delta_{hn}

    # --- service / newsvendor --------------------------------------------
    SLmin: np.ndarray         # (G,)         target min service level  SL^min_g
    epsilon: float            # fractile truncation, 0 < epsilon < 1/2
    eps0: float               # Eq (29) division guard

    # --- discounting -------------------------------------------------------
    gamma_disc: float         # gamma in (0,1], Eq (42)

# >>> INSTANCE DATA
# 3 produits, 4 machines, routes fixes reprises de Ryan (d'apres Ozdamar &
# Barbarosoglu, 1999). Une epoque de revue = un jour ; un intervalle
# d'execution = un poste de 8 h ; horizon 28 jours (Wang et al., 2025).
# Chaque valeur porte sa source en commentaire.

TARGET_TIME_UTIL = 0.625      # 1/CAT avec CAT = 1.6, scenario "loose" d'Ozdamar (1999)
TARGET_RESOURCE_UTIL = 0.625  # meme regle appliquee au froid


def build_instance(seed: int = 0, N: int = 28) -> Tuple[Instance, Params, dict]:
    # N is a PARAMETER so that a ladder of reduced-horizon instances can be built for
    # the exact-solver comparison; the reference study reports CPLEX hitting its time
    # limit from about T = 10 macro-periods upward, so the full N = 28 instance is out
    # of reach for an exact solver and smaller instances are required.
    G, E, H = 3, 4, 3                     # 28 jours x 3 postes -> Upsilon = 84
    Jg = np.array([3, 3, 4], dtype=int)
    Jmax = int(Jg.max())

    # ---- routes fixes ; machines 0-indexees (e0..e3 = e1..e4 du papier) ----
    #   g1 : e3 -> e2 -> e1        g2 : e2 -> e4 -> e3        g3 : e1 -> e3 -> e2 -> e4
    stage_of = -np.ones((G, E), dtype=int)
    stage_of[0, 2], stage_of[0, 1], stage_of[0, 0] = 0, 1, 2
    stage_of[1, 1], stage_of[1, 3], stage_of[1, 2] = 0, 1, 2
    stage_of[2, 0], stage_of[2, 2], stage_of[2, 1], stage_of[2, 3] = 0, 1, 2, 3

    # ---- classes d'age : durees de vie 3/4/5 jours = 9/12/15 postes --------
    #   Temizoz et al. (2025), EJOR 324, p.111 : product lifetime m_l = {3,4,5}
    #   Horloge unique : meme \bar a a toutes les etapes d'un produit.
    abar_g = np.array([3, 4, 5]) * H - 1                       # 8, 11, 14
    abar = np.zeros((G, Jmax), dtype=int)
    for g in range(G):
        abar[g, : Jg[g]] = abar_g[g]

    inst = Instance(G=G, Jg=Jg, E=E, stage_of=stage_of,
                    eligible=(stage_of >= 0), abar=abar, N=N, H=H)
    A = inst.A

    # ---- demande nominale --------------------------------------------------
    #   Ozdamar & Barbarosoglu (1999), p.820 : niveau de base N(100, .)
    #   Gamma(k=4, theta=25) -> moyenne 100, CV 0.50 (Nomura et al., 2025)
    nominal_demand = np.full(G, 100.0)

    # ---- temps de traitement (min/unite -> h/unite) -------------------------
    #   Calibres pour un goulot a 62.5 % (CAT = 1.6 d'Ozdamar) a demande nominale.
    tau_min = np.full((G, Jmax, E), np.nan)
    tau_min[0, 0, 2], tau_min[0, 1, 1], tau_min[0, 2, 0] = 2.6, 3.2, 2.8
    tau_min[1, 0, 1], tau_min[1, 1, 3], tau_min[1, 2, 2] = 3.0, 2.4, 3.4
    tau_min[2, 0, 0], tau_min[2, 1, 2], tau_min[2, 2, 1], tau_min[2, 3, 3] = 3.6, 3.0, 2.8, 2.2
    tau = np.nan_to_num(tau_min / 60.0, nan=1e6)               # ineligible -> infini

    Delta = np.full((H, N), 8.0)                               # poste de 8 h
    Cbar = np.full((E, H, N), 8.0)                             # machine dispo tout le poste

    # ---- couts de traitement et froid, repartis au prorata de tau -----------
    #   cout total d'une unite finie = 3 (Wang 2025 ; Nomura 2025)
    #   froid = 0.15 kWh/unite (15 kWh/tonne, adapte de Zhao et al. 2018 ; 1 unite = 10 kg)
    mu = np.zeros((G, Jmax, E))
    psi = np.zeros((G, Jmax, E))
    for g in range(G):
        route = [(j, int(np.where(stage_of[g] == j)[0][0])) for j in range(Jg[g])]
        tot = sum(tau_min[g, j, e] for j, e in route)
        for j, e in route:
            share = tau_min[g, j, e] / tot
            mu[g, j, e] = 3.00 * share
            psi[g, j, e] = 0.15 * share

    # ---- changements de serie ----------------------------------------------
    upsilon = np.zeros((G, G, E))                              # 30 min = 0.5 h
    kappa = np.zeros((G, G, E))                                # declare
    for e in range(E):
        for g in range(G):
            for gp in range(G):
                if g != gp:
                    upsilon[g, gp, e] = 0.5
                    kappa[g, gp, e] = 10.0

    # ---- parametres economiques --------------------------------------------
    #   Wang et al. (2025) Omega ; Nomura et al. (2025) Appl. Sci. 15:2421 ;
    #   Yan, Chen & Yu (2025) POM 34(11) pour le salvage.
    alpha = np.ones((G, A))                                    # possession = 1, sans effet d'age
    beta = np.full(G, 5.0)                                     # rupture / backlog
    rho = np.full((G, N), 10.0)                                # revenu par unite servie
    delta = np.zeros((G, Jmax)); sigma = np.zeros((G, Jmax))
    for g in range(G):
        delta[g, : Jg[g]] = 7.0                                # peremption / rebut
        sigma[g, : Jg[g]] = 4.0                                # recuperation
    theta_service = np.full(G, 0.05)                           # 1 jour de retard double la penalite

    # ---- deterioration -------------------------------------------------------
    #   lam0, lam1 : Weibull de Wang et al. (2025) recalee au pas du poste
    #   lam2       : loi de Wu et al. (2025), Q10 = 3, calee a la classe d'age mediane
    #   lam3, lam4 : desactives, aucun coefficient publie
    lam0_g = np.array([0.000421, 0.000547, 0.000643])          # /h
    lam1_g = np.array([0.000132, 0.000131, 0.000124])          # /h/classe d'age
    lam2_g = np.array([0.000177, 0.000240, 0.000290])          # /h/degC d'ecart
    lam0 = np.zeros((G, Jmax)); lam1 = np.zeros((G, Jmax)); lam2 = np.zeros((G, Jmax))
    lam3 = np.zeros((G, Jmax)); lam4 = np.zeros((G, Jmax)); omega = np.zeros((G, Jmax))
    for g in range(G):
        for j in range(Jg[g]):
            lam0[g, j], lam1[g, j], lam2[g, j] = lam0_g[g], lam1_g[g], lam2_g[g]
            omega[g, j] = 0.0 if j == Jg[g] - 1 else 1.0       # en-cours pleinement compte

    # ---- bornes de lancement -------------------------------------------------
    ell = np.full(G, 40.0)                                     # declare : 1/4 de poste
    umax = np.full(G, 200.0)                                   # 2x la demande (Kara & Dogan)

    prm = Params(
        mu=mu, alpha=alpha, beta=beta, rho=rho, delta=delta, sigma=sigma, kappa=kappa,
        theta_service=theta_service,
        eta_idle=1.0, eta_serv=1.0, eta_viol=1.0,              # poids a 1, cf. Table 2
        tau=tau, Cbar=Cbar, upsilon=upsilon, psi=psi, ell=ell, umax=umax, Mfrak=1e7,
        lam0=lam0, lam1=lam1, lam2=lam2, lam3=lam3, lam4=lam4, omega=omega, Delta=Delta,
        SLmin=np.full(G, 0.90),                                # Wu et al. (2025)
        epsilon=0.01, eps0=1e-6, gamma_disc=0.99,
    )

    Rbar_nominal = float(nominal_demand.sum() * 0.15 / H / TARGET_RESOURCE_UTIL)
    nominal = dict(demand=nominal_demand, Wbar=4.0, Rbar=Rbar_nominal,
                   Wbar_profile=np.array([4.0, 4.0, 4.0]),     # effectif complet sur les 3 postes
                   machine_hours_per_epoch=float(E * H * 8.0))

    # ---- controle de calibration ---------------------------------------------
    load = np.zeros(E)
    for g in range(G):
        for j in range(Jg[g]):
            e = int(np.where(stage_of[g] == j)[0][0])
            load[e] += nominal_demand[g] * tau_min[g, j, e]
    cap_day = H * 480.0
    print("charge machine (min/jour) :")
    for e in range(E):
        print(f"   e{e}: {load[e]:6.0f} min  ->  {100 * load[e] / cap_day:5.1f} %")
    print(f"   goulot {100 * load.max() / cap_day:.1f} %  (cible 62.5 % = CAT 1.6)")
    return inst, prm, nominal


INST, PRM, NOMINAL = build_instance()
print(f"\n|G|={INST.G}  J_g={INST.Jg}  |E|={INST.E}  |N|={INST.N}  |H_n|={INST.H}  "
      f"Upsilon={INST.Upsilon}  classes d'age={INST.A}")
print(f"routes stage_of[g,e] =\n{INST.stage_of}")
print(f"\\bar R = {NOMINAL['Rbar']:.1f} kWh/poste,  \\bar W profil = {NOMINAL['Wbar_profile']}")
print(f"tau in [{PRM.tau[PRM.tau < 1e5].min() * 60:.1f}, "
      f"{PRM.tau[PRM.tau < 1e5].max() * 60:.1f}] min/unite")


@dataclass
class Scenario:
    """A fixed realisation zeta of all exogenous processes over the horizon."""
    kind: str                      # "routine" or a disruption family name
    demand_mean: np.ndarray        # (G,N)   m_{gn}: mean of \mathfrak{F}_{gn}
    demand_cv: np.ndarray          # (G,)    cv of \mathfrak{F}_{gn}
    demand: np.ndarray             # (G,N)   \mathscr{D}^zeta_{gn}, drawn from \mathfrak{F}
    avail: np.ndarray              # (E,H,N) \mathfrak{a}^zeta_{ehn} in {0,1}
    temp: np.ndarray               # (H,N)   Temp^zeta_{hn}  (deviation from nominal, C)
    hum: np.ndarray                # (H,N)   Hum^zeta_{hn}   (deviation from nominal, %RH)
    dev: np.ndarray                # (H,N)   Dev^zeta_{hn}   in {0,1}
    Rbar: np.ndarray               # (H,N)   \bar R_{hn}
    Wbar: np.ndarray               # (H,N)   \bar W_{hn}
    material: np.ndarray           # (G,N)   \mathfrak{m}^zeta_{gn}
    weight: float = 1.0            # varpi_zeta (routine) or \widehat\varpi_zeta (disruption)

    # ---- \mathfrak{F}_{gn}: Gamma(shape=1/cv^2, scale=mean*cv^2) ----------
    def _shape_scale(self, g: int, n: int) -> Tuple[float, float]:
        cv = self.demand_cv[g]
        k = 1.0 / (cv ** 2)
        theta = max(self.demand_mean[g, n], 1e-6) * cv ** 2
        return k, theta

    def _ks(self) -> Tuple[np.ndarray, np.ndarray]:
        """Vectorised (shape, scale) arrays of \\mathfrak{F}_{gn}, cached."""
        if getattr(self, "_ks_cache", None) is None:
            k = 1.0 / self.demand_cv ** 2                                  # (G,)
            theta = np.maximum(self.demand_mean, 1e-6) * (self.demand_cv ** 2)[:, None]
            object.__setattr__(self, "_ks_cache", (k, theta))
        return self._ks_cache

    def quantile(self, g: int, n: int, chi: float) -> float:
        """\\mathfrak{F}^{-1}_{gn}(chi | s_n), scalar form. Used by Eqs (3) and (51)."""
        k, theta = self._shape_scale(g, n)
        return float(gamma_dist.ppf(np.clip(chi, 1e-6, 1 - 1e-6), a=k, scale=theta))

    def quantile_vec(self, n: int, chi: np.ndarray) -> np.ndarray:
        """\\mathfrak{F}^{-1}_{gn}(chi_g | s_n) for all g at once (one vectorised call)."""
        k, theta = self._ks()
        return gamma_dist.ppf(np.clip(chi, 1e-6, 1 - 1e-6), a=k, scale=theta[:, n])

    def forecast_descriptor(self, n: int) -> np.ndarray:
        """Fixed-dimensional descriptor of \\mathfrak{F}_{gn} handed to the policy (§3.8.2).

        "In implementation, \\mathfrak{F}_{gn}(. | s_n) is supplied to the policy through a
        fixed-dimensional forecast descriptor containing the information required to
        evaluate the corresponding demand quantiles; the neural network is not provided
        with an abstract distribution object."  The realised demand \\mathscr{D}_{gn} is
        never part of it (checked by `assert_no_demand_leak`).
        """
        desc = getattr(self, "_desc_cache", None)
        if desc is None:
            k, theta = self._ks()                                  # (G,), (G,N)
            G, N = theta.shape
            out = np.zeros((N, G, 5))
            out[:, :, 0] = (k[:, None] * theta).T                          # mean
            out[:, :, 1] = (np.sqrt(k)[:, None] * theta).T                 # std
            for c, q in enumerate([0.25, 0.50, 0.90], start=2):            # quantiles
                out[:, :, c] = gamma_dist.ppf(q, a=k[:, None], scale=theta).T
            desc = out.reshape(N, G * 5)
            object.__setattr__(self, "_desc_cache", desc)
        return desc[n]


# --- routine ranges: the boundary that separates R from D --------------------
ROUTINE_AVAIL_MIN_FRACTION = 0.75   # >= 75% of units available in every interval
ROUTINE_WORKFORCE_MIN = None        # filled in by make_scenario_generator


class ScenarioGenerator:
    """Draws zeta from R (training) or from D (post-training stress testing)."""

    def __init__(self, inst: Instance, prm: Params, nominal: dict, seed: int = 0):
        self.inst, self.prm = inst, prm
        self.rng = np.random.default_rng(seed)
        G, E, N, H = inst.G, inst.E, inst.N, inst.H

        # >>> INSTANCE DATA: nominal demand profile and resource envelopes
        self.base_demand = np.asarray(nominal["demand"], dtype=float)
        self.season_amp = 0.15 + 0.15 * self.rng.random(G)
        self.season_phase = 2 * math.pi * self.rng.random(G)
        self.cv_range = (0.18, 0.32)

        self.Rbar_nominal = float(nominal["Rbar"])
        self.Wbar_nominal = float(nominal["Wbar"])

        # routine floors (used both to generate R and to assert the R/D separation)
        self.routine_avail_floor = int(math.ceil(ROUTINE_AVAIL_MIN_FRACTION * E))
        self.Wbar_profile = np.asarray(
            nominal.get("Wbar_profile", np.full(inst.H, self.Wbar_nominal)), dtype=float)
        self.routine_workforce_floor = np.maximum(
            1.0, np.round(0.85 * self.Wbar_profile))          # (H,) plancher par poste
        self.routine_Rbar_floor = 0.85 * self.Rbar_nominal

    # ------------------------------------------------------------------ R ---
    def sample_routine(self) -> Scenario:
        inst, prm, rng = self.inst, self.prm, self.rng
        G, E, N, H = inst.G, inst.E, inst.N, inst.H

        n_idx = np.arange(N)
        season = 1.0 + self.season_amp[:, None] * np.sin(
            2 * math.pi * n_idx[None, :] / max(N, 1) * 2.0 + self.season_phase[:, None])
        level = 1.0 + 0.10 * rng.standard_normal((G, 1))            # slow level shift
        noise = 1.0 + 0.06 * rng.standard_normal((G, N))            # forecast drift
        demand_mean = np.clip(self.base_demand[:, None] * season * level * noise, 5.0, None)
        demand_cv = rng.uniform(*self.cv_range, size=G)

        demand = np.zeros((G, N))
        for g in range(G):
            k = 1.0 / demand_cv[g] ** 2
            demand[g] = rng.gamma(shape=k, scale=demand_mean[g] / k)

        # normal operating uncertainty: minor, bounded equipment unavailability
        avail = np.ones((E, H, N), dtype=float)
        for h in range(H):
            for n in range(N):
                down = rng.random(E) < 0.06
                if down.sum() > E - self.routine_avail_floor:       # keep inside R
                    idx = np.where(down)[0]
                    rng.shuffle(idx)
                    down[idx[E - self.routine_avail_floor:]] = False
                avail[down, h, n] = 0.0

        # routine storage conditions: small deviations, NO cold-chain failure
        temp = 0.45 * rng.standard_normal((H, N))
        hum = 1.60 * rng.standard_normal((H, N))
        dev = np.zeros((H, N))                                      # Dev == 0 on R

        Rbar = self.Rbar_nominal * (1.0 + 0.05 * rng.standard_normal((H, N)))
        Rbar = np.maximum(Rbar, self.routine_Rbar_floor)
        Wbar = np.tile(self.Wbar_profile[:, None], (1, N))          # profil (4,4,2)
        Wbar -= (rng.random((H, N)) < 0.10).astype(float)           # occasional -1
        Wbar = np.maximum(Wbar, self.routine_workforce_floor[:, None])

        material = np.full((G, N), 1e6)                             # non-binding on R

        return Scenario("routine", demand_mean, demand_cv, demand, avail,
                        temp, hum, dev, Rbar, Wbar, material)

    # ------------------------------------------------------------------ D ---
    def sample_disruption(self, family: str) -> Scenario:
        """Severe out-of-sample scenarios. NEVER sampled during training."""
        assert family in ("line_failure", "workforce_shortage", "cold_chain")
        sc = self.sample_routine()
        sc.kind = family
        rng, inst = self.rng, self.inst
        E, H, N = inst.E, inst.H, inst.N

        n0 = rng.integers(N // 4, max(N // 4 + 1, 3 * N // 4))
        span = 2                                   # 2 epoques (design du banc)
        window = slice(int(n0), min(N, int(n0) + span))

        if family == "line_failure":
            k = max(1, int(round(0.5 * E)))
            hit = rng.choice(E, size=k, replace=False)
            sc.avail[np.ix_(hit, range(H), range(window.start, window.stop))] = 0.0
        elif family == "workforce_shortage":
            sc.Wbar[:, window] = np.maximum(
                1.0, np.floor(0.5 * self.Wbar_profile))[:, None]   # Ivanov 2021 : 50 %
        elif family == "cold_chain":
            sc.dev[:, window] = 1.0                                 # Dev = 1
            sc.temp[:, window] += 15.0                              # 23 C vs 8 C nominal
        return sc

    # ---------------------------------------------------------- guarantees ---
    def assert_routine(self, sc: Scenario) -> None:
        """§3.8.5: the three explicit tests that keep D out of R."""
        E = self.inst.E
        assert sc.kind == "routine"
        assert np.all(sc.dev == 0.0), "routine scenario activated the cold-chain channel"
        assert np.all(sc.avail.sum(axis=0) >= self.routine_avail_floor), \
            "routine scenario imposed a production-line failure state outside R"
        assert np.all(sc.Wbar >= self.routine_workforce_floor[:, None]), \
            "routine scenario imposed a workforce-shortage state outside R"


GEN = ScenarioGenerator(INST, PRM, NOMINAL, seed=1)

# --- the exclusion tests of §3.8.5 -------------------------------------------
for _ in range(200):
    GEN.assert_routine(GEN.sample_routine())
_viol = {"line_failure": 0, "workforce_shortage": 0, "cold_chain": 0}
for fam in _viol:
    for _ in range(20):
        d = GEN.sample_disruption(fam)
        try:
            d2 = Scenario(**{**d.__dict__}); d2.kind = "routine"
            GEN.assert_routine(d2)
        except AssertionError:
            _viol[fam] += 1
print("routine scenarios pass all three exclusion tests (200/200)")
print("disruption scenarios correctly rejected by those tests:", _viol, "of 20 each")

_sc = GEN.sample_routine()
print(f"\nsample routine scenario: demand mean/product = "
      f"{_sc.demand_mean.mean(axis=1).round(1)},  cv = {_sc.demand_cv.round(3)}")
print(f"  \\bar R nominal = {GEN.Rbar_nominal:.1f}, \\bar W nominal = {GEN.Wbar_nominal}, "
      f"routine floors: avail>={GEN.routine_avail_floor}, W>={GEN.routine_workforce_floor}")

# --- Eqs (7)-(8), verbatim de la cellule 10 du notebook PPO ---
def project_release(x: float, ell: float, umax: float, material: float) -> Tuple[float, int]:
    """Proj_{V_{gn}}[x], Eqs (7)-(8). Returns (release, binary indicator O_{gn}) of Eq (9)."""
    hi = min(umax, material)
    if hi < ell:                       # V = {0}                       Eq (8), first case
        return 0.0, 0
    if x <= 0.0:
        return 0.0, 0
    if x < ell:                        # nearest point of {0} U [ell, hi]
        return (0.0, 0) if (x - 0.0) <= (ell - x) else (ell, 1)
    return (min(x, hi), 1)

charge machine (min/jour) :
   e0:    640 min  ->   44.4 %
   e1:    900 min  ->   62.5 %
   e2:    900 min  ->   62.5 %
   e3:    460 min  ->   31.9 %
   goulot 62.5 %  (cible 62.5 % = CAT 1.6)

|G|=3  J_g=[3 3 4]  |E|=4  |N|=28  |H_n|=3  Upsilon=84  classes d'age=15
routes stage_of[g,e] =
[[ 2  1  0 -1]
 [-1  0  2  1]
 [ 0  2  1  3]]
\bar R = 24.0 kWh/poste,  \bar W profil = [4. 4. 4.]
tau in [2.2, 3.6] min/unite
routine scenarios pass all three exclusion tests (200/200)
disruption scenarios correctly rejected by those tests: {'line_failure': 20, 'workforce_shortage': 20, 'cold_chain': 20} of 20 each

sample routine scenario: demand mean/product = [101.   80.4  95.4],  cv = [0.23  0.201 0.283]
  \bar R nominal = 24.0, \bar W nominal = 4.0, routine floors: avail>=3, W>=[3. 3. 3.]


In [5]:
"""Generateur de data.dat AMPL  --  VERSION CORRIGEE.

Ecrit les donnees au format LISTE (tous les indices puis la valeur), qui est la
forme la plus sure pour des parametres indexes sur des ensembles creux de dimension
2 ou 3. AMPL utilise l'indexation 1-based pour g, j, e, h, n ; les classes d'age
restent 0-based, comme dans le PDF (a = 0 pour le materiel frais).

AJOUTS DE LA VERSION CORRIGEE (aucune suppression) :
  * PrevAsg[g,e]  : affectation reellement executee au dernier intervalle precedent,
                    reportee d'une fenetre a la suivante -> Eq (36) au 1er intervalle.
  * REPAIR, K0, Opn0 : mode "reparation intra-epoque" (Eqs 32-34), ou le lancement
                    V n'est plus une decision mais un solde deja engage.
  * FixAsg[g,e,h,n] : epinglage d'affectations, utilise UNIQUEMENT par les tests.
"""
import numpy as np


def _fmt(x):
    return f"{float(x):.10g}"


def build_dims(INST, PRM, G=None, E=None, N=None, H=None, abar_cap=None):
    """Construit le dictionnaire de dimensions/ensembles a partir de INST et PRM
    du notebook PPO. Les arguments optionnels tronquent l'instance (petite instance
    de verification). Indices Python 0-based ; la conversion en 1-based a lieu a
    l'ecriture."""
    G = INST.G if G is None else G
    E = INST.E if E is None else E
    N = INST.N if N is None else N
    H = INST.H if H is None else H
    Jg = [int(INST.Jg[g]) for g in range(G)]
    abar = INST.abar[:G, :].copy()
    if abar_cap is not None:
        abar = np.minimum(abar, abar_cap)
    Egj = [[[e for e in range(E) if int(INST.stage_of[g, e]) == j] for j in range(Jg[g])]
           for g in range(G)]
    return dict(
        G=G, E=E, N=N, H=H, Jg=Jg, abar=abar, Egj=Egj,
        eligible=INST.eligible[:G, :E], stage_of=INST.stage_of[:G, :E],
        tau=PRM.tau[:G], mu=PRM.mu[:G], psi=PRM.psi[:G],
        kappa=PRM.kappa[:G, :G, :E], upsilon=PRM.upsilon[:G, :G, :E],
        Cbar=PRM.Cbar[:E, :H, :N], Delta=PRM.Delta[:H, :N],
        alpha=PRM.alpha[:G], beta=PRM.beta[:G], rho=PRM.rho[:G, :N],
        delta=PRM.delta[:G], ell=PRM.ell[:G], umax=PRM.umax[:G],
        SLmin=PRM.SLmin[:G], eps0=PRM.eps0,
        eta_idle=PRM.eta_idle, eta_serv=PRM.eta_serv,
        lam0=PRM.lam0[:G], lam1=PRM.lam1[:G], lam2=PRM.lam2[:G],
        lam3=PRM.lam3[:G], lam4=PRM.lam4[:G])


def scenario_dict(sc, G, E, N, H):
    return dict(avail=sc.avail[:E, :H, :N], temp=sc.temp[:H, :N], hum=sc.hum[:H, :N],
                dev=sc.dev[:H, :N], Rbar=sc.Rbar[:H, :N], Wbar=sc.Wbar[:H, :N],
                material=sc.material[:G, :N])


def survival(d, sc):
    """Gam[g,j,a,h,n], Eqs (10)-(11), calcule ICI et non dans le modele."""
    Gam = {}
    for g in range(d["G"]):
        for j in range(d["Jg"][g]):
            for a in range(d["abar"][g, j] + 1):
                lam = (d["lam0"][g, j] + d["lam1"][g, j] * a
                       + d["lam2"][g, j] * sc["temp"] + d["lam3"][g, j] * sc["hum"]
                       + d["lam4"][g, j] * sc["dev"])                     # (H,N)
                Gm = np.clip(np.exp(-np.maximum(lam, 0.0) * d["Delta"]), 0.0, 1.0)
                for h in range(d["H"]):
                    for n in range(d["N"]):
                        Gam[g, j, a, h, n] = float(Gm[h, n])
    return Gam


def survival_array(d, temp, hum, dev, Delta):
    """Meme calcul, renvoye en tableau (G,Jm,Am,H). temp/hum/dev/Delta : (H,)."""
    G, H = d["G"], len(np.atleast_1d(Delta))
    Jm, Am = max(d["Jg"]), int(np.max(d["abar"])) + 1
    temp, hum, dev = np.asarray(temp), np.asarray(hum), np.asarray(dev)
    Delta = np.asarray(Delta, dtype=float)
    Gam = np.ones((G, Jm, Am, H))
    for g in range(G):
        for j in range(d["Jg"][g]):
            for a in range(d["abar"][g, j] + 1):
                lam = (d["lam0"][g, j] + d["lam1"][g, j] * a + d["lam2"][g, j] * temp
                       + d["lam3"][g, j] * hum + d["lam4"][g, j] * dev)     # (H,)
                Gam[g, j, a, :] = np.clip(np.exp(-np.maximum(lam, 0.0) * Delta), 0.0, 1.0)
    return Gam


def initial_state(d, sc_full):
    """Conditions initiales ABSENTES DU PDF, reprises de Env.reset du notebook PPO.
    N'utilise que demand_mean (la PREVISION), jamais la demande realisee."""
    G, Jm = d["G"], max(d["Jg"])
    Am = int(np.max(d["abar"])) + 1
    S0 = np.zeros((G, Jm, Am))
    for g in range(G):
        jf = d["Jg"][g] - 1
        per = 0.5 * sc_full.demand_mean[g, 0] / max(1, min(2, d["abar"][g, jf] + 1))
        for a in range(min(2, d["abar"][g, jf] + 1)):
            S0[g, jf, a] = per
    return S0, np.zeros(G)


def write_dat(path, d, sc, demand, S0, L0, enforce_fefo=1,
              prev_asg=None, repair=0, K0=None, Opn0=None, fix_asg=None):
    """Ecrit data.dat.

    prev_asg : (G,E) 0/1  -- affectation executee a l'intervalle precedant le
               PREMIER intervalle de la fenetre.  None => aucune (comportement
               identique a la version d'origine).
    repair   : 1 => V n'est plus une decision, il vaut K0 (solde restant).
    K0, Opn0 : (G,) -- solde de lancement restant et indicateur deja engage.
    fix_asg  : dict {(g,e,h,n): 0/1} en indices PYTHON 0-based -- epinglage
               d'affectations, utilise uniquement par les tests.
    """
    G, E, N, H = d["G"], d["E"], d["N"], d["H"]
    Jg, abar, Egj = d["Jg"], d["abar"], d["Egj"]
    jf = [Jg[g] - 1 for g in range(G)]
    Gam = survival(d, sc)
    AMAX = int(np.max(abar))
    L = []
    w = L.append

    w("# =============================================================================")
    w("# data.dat  --  genere automatiquement depuis le notebook PPO.")
    w("# Indices AMPL : g,j,e,h,n en 1-based ; classes d'age a en 0-based (PDF).")
    w("# AUCUNE valeur n'est inventee : tout provient de INST, PRM ou du scenario.")
    w("# =============================================================================")
    w(f"param G := {G};")
    w(f"param E := {E};")
    w(f"param N := {N};")
    w(f"param H := {H};")
    w(f"param AMAX := {AMAX};")
    w(f"param eps0 := {_fmt(d['eps0'])};")
    w(f"param eta_idle := {_fmt(d['eta_idle'])};")
    w(f"param eta_serv := {_fmt(d['eta_serv'])};")
    w(f"param ENFORCE_FEFO := {int(enforce_fefo)};")
    w(f"param REPAIR := {int(repair)};")
    w("")
    w("#  PARAMETRES DEFINIS SYMBOLIQUEMENT MAIS NON CHIFFRES DANS sujet.pdf :")
    w("#    eta_rec  -- apparait dans l'Eq (43) (poids du terme de recuperation)")
    w("#    w_conf   -- apparait dans l'Eq (45) (fenetre de confirmation)")
    w("#  Les deux sont DEFINIS par le papier ; seule leur valeur numerique n'est")
    w("#  pas fournie. eta_rec * H^rec reste donc hors de l'objectif tant qu'une")
    w("#  valeur n'est pas choisie, et H^rec est reporte apres resolution en")
    w("#  sensibilite sur w_conf.")
    w("")

    # ---- ensembles ----------------------------------------------------------
    w("set GJ := " + " ".join(f"({g+1},{j+1})" for g in range(G)
                              for j in range(Jg[g])) + ";")
    w("set GJA := " + " ".join(f"({g+1},{j+1},{a})" for g in range(G)
                               for j in range(Jg[g])
                               for a in range(abar[g, j] + 1)) + ";")
    w("set GJE := " + " ".join(f"({g+1},{j+1},{e+1})" for g in range(G)
                               for j in range(Jg[g]) for e in Egj[g][j]) + ";")
    w("set GE := " + " ".join(f"({g+1},{e+1})" for g in range(G) for e in range(E)
                              if d["eligible"][g, e]) + ";")
    w("set GGE := " + " ".join(f"({g+1},{gp+1},{e+1})"
                               for g in range(G) for gp in range(G) if gp != g
                               for e in range(E)
                               if d["eligible"][g, e] and d["eligible"][gp, e]) + ";")
    w("set GA := " + " ".join(f"({g+1},{a})" for g in range(G)
                              for a in range(abar[g, jf[g]] + 1)) + ";")
    w("")

    # ---- parametres indexes -------------------------------------------------
    w("param jf := " + " ".join(f"{g+1} {jf[g]+1}" for g in range(G)) + ";")
    w("param abar := " + " ".join(f"{g+1} {j+1} {int(abar[g,j])}"
                                  for g in range(G) for j in range(Jg[g])) + ";")
    w("param beta := " + " ".join(f"{g+1} {_fmt(d['beta'][g])}" for g in range(G)) + ";")
    w("param ell := " + " ".join(f"{g+1} {_fmt(d['ell'][g])}" for g in range(G)) + ";")
    w("param umax := " + " ".join(f"{g+1} {_fmt(d['umax'][g])}" for g in range(G)) + ";")
    w("param SLmin := " + " ".join(f"{g+1} {_fmt(d['SLmin'][g])}" for g in range(G)) + ";")
    w("param L0 := " + " ".join(f"{g+1} {_fmt(L0[g])}" for g in range(G)) + ";")
    w("")

    # ---- report d'affectation entre fenetres  (Eq 36 au 1er intervalle) -----
    if prev_asg is not None:
        rows = [f"{g+1} {e+1} {int(round(float(prev_asg[g, e])))}"
                for g in range(G) for e in range(E) if d["eligible"][g, e]]
        w("param PrevAsg := " + " ".join(rows) + ";")
        w("")
    if repair:
        assert K0 is not None and Opn0 is not None, "mode reparation : K0 et Opn0 requis"
        w("param K0 := " + " ".join(f"{g+1} {_fmt(K0[g])}" for g in range(G)) + ";")
        w("param Opn0 := " + " ".join(f"{g+1} {int(round(float(Opn0[g])))}"
                                      for g in range(G)) + ";")
        w("")
    if fix_asg:
        w("param FixAsg := " + " ".join(
            f"{g+1} {e+1} {h+1} {n+1} {int(v)}" for (g, e, h, n), v in
            sorted(fix_asg.items())) + ";")
        w("")

    def blk(name, rows):
        w(f"param {name} :=")
        for r in rows:
            w("  " + r)
        w(";")
        w("")

    blk("alpha", [f"{g+1} {a} {_fmt(d['alpha'][g, a])}"
                  for g in range(G) for a in range(AMAX + 1)])
    blk("rho", [f"{g+1} {n+1} {_fmt(d['rho'][g, n])}"
                for g in range(G) for n in range(N)])
    blk("term_value", [f"{g+1} {j+1} {_fmt(d['rho'][g, N-1] * (j + 1) / Jg[g])}"
                          for g in range(G) for j in range(Jg[g])])
    blk("delta", [f"{g+1} {j+1} {_fmt(d['delta'][g, j])}"
                  for g in range(G) for j in range(Jg[g])])
    blk("mu", [f"{g+1} {j+1} {e+1} {_fmt(d['mu'][g, j, e])}"
               for g in range(G) for j in range(Jg[g]) for e in Egj[g][j]])
    blk("tau", [f"{g+1} {j+1} {e+1} {_fmt(d['tau'][g, j, e])}"
                for g in range(G) for j in range(Jg[g]) for e in Egj[g][j]])
    blk("psi", [f"{g+1} {j+1} {e+1} {_fmt(d['psi'][g, j, e])}"
                for g in range(G) for j in range(Jg[g]) for e in Egj[g][j]])
    blk("kappa", [f"{g+1} {gp+1} {e+1} {_fmt(d['kappa'][g, gp, e])}"
                  for g in range(G) for gp in range(G) if gp != g for e in range(E)
                  if d["eligible"][g, e] and d["eligible"][gp, e]])
    blk("upsilon", [f"{g+1} {gp+1} {e+1} {_fmt(d['upsilon'][g, gp, e])}"
                    for g in range(G) for gp in range(G) if gp != g for e in range(E)
                    if d["eligible"][g, e] and d["eligible"][gp, e]])
    blk("Cbar", [f"{e+1} {h+1} {n+1} {_fmt(d['Cbar'][e, h, n])}"
                 for e in range(E) for h in range(H) for n in range(N)])
    blk("avail", [f"{e+1} {h+1} {n+1} {int(round(float(sc['avail'][e, h, n])))}"
                  for e in range(E) for h in range(H) for n in range(N)])
    blk("Delta", [f"{h+1} {n+1} {_fmt(d['Delta'][h, n])}"
                  for h in range(H) for n in range(N)])
    blk("Rbar", [f"{h+1} {n+1} {_fmt(sc['Rbar'][h, n])}"
                 for h in range(H) for n in range(N)])
    # Wcap = floor(Wbar) : voir audit section 4.3
    blk("Wcap", [f"{h+1} {n+1} {int(np.floor(sc['Wbar'][h, n] + 1e-9))}"
                 for h in range(H) for n in range(N)])
    blk("mfrak", [f"{g+1} {n+1} {_fmt(sc['material'][g, n])}"
                  for g in range(G) for n in range(N)])
    blk("D", [f"{g+1} {n+1} {_fmt(demand[g, n])}"
              for g in range(G) for n in range(N)])
    blk("S0", [f"{g+1} {j+1} {a} {_fmt(S0[g, j, a])}"
               for g in range(G) for j in range(Jg[g])
               for a in range(abar[g, j] + 1)])
    blk("Gam", [f"{g+1} {j+1} {a} {h+1} {n+1} {_fmt(Gam[g, j, a, h, n])}"
                for g in range(G) for j in range(Jg[g])
                for a in range(abar[g, j] + 1)
                for h in range(H) for n in range(N)])
    # big-M resserres, audit section 2.5
    blk("Mp", [f"{g+1} {j+1} {e+1} {h+1} {n+1} "
               f"{_fmt(d['Cbar'][e, h, n] / d['tau'][g, j, e])}"
               for g in range(G) for j in range(Jg[g]) for e in Egj[g][j]
               for h in range(H) for n in range(N)])
    blk("Mfefo", [f"{g+1} {j+1} {_fmt(d['umax'][g] * H * N)}"
                  for g in range(G) for j in range(Jg[g])])

    txt = "\n".join(L) + "\n"
    with open(path, "w") as f:
        f.write(txt)
    return txt


# =============================================================================
#  Jeux de scenarios, IDENTIQUES a ceux du notebook PPO
# =============================================================================
EVAL_GEN   = ScenarioGenerator(INST, PRM, NOMINAL, seed=98765)
STRESS_GEN = ScenarioGenerator(INST, PRM, NOMINAL, seed=424242)
EVAL_SCENARIOS = [EVAL_GEN.sample_routine() for _ in range(24)]
DISRUPTION_FAMILIES = ("line_failure", "workforce_shortage", "cold_chain")
print(f"{len(EVAL_SCENARIOS)} scenarios routiniers tenus a l'ecart (seed 98765)")


def forecast_median(sc, G, N):
    """Mediane q50 de la prevision, exactement la valeur que recoit PPO dans son
    descripteur a 5 composantes. Utilisee en mode rolling horizon.
    N'utilise QUE demand_mean et demand_cv : jamais la demande realisee."""
    med = np.zeros((G, N))
    for n in range(N):
        desc = sc.forecast_descriptor(n).reshape(-1, 5)      # [mean, std, q25, q50, q90]
        med[:, n] = desc[:G, 3]
    return med


# =============================================================================
#  Instance JOUET : verifie la syntaxe AMPL en quelques secondes
# =============================================================================
SC0 = EVAL_SCENARIOS[0]
d_toy  = build_dims(INST, PRM, G=2, E=4, N=2, H=2, abar_cap=3)
sc_toy = scenario_dict(SC0, d_toy["G"], d_toy["E"], d_toy["N"], d_toy["H"])
S0_toy, L0_toy = initial_state(d_toy, SC0)
write_dat("data_toy.dat", d_toy, sc_toy, SC0.demand[:2, :2], S0_toy, L0_toy)
print("data_toy.dat ecrit  (G=2, E=4, N=2, H=2, abar<=3)")

# =============================================================================
#  Instance COMPLETE, mode ORACLE (demande realisee connue d'avance)
#
#  ATTENTION. Ce mode n'est PAS la comparaison equitable : il connait toutes les
#  realisations. Il sert de BORNE INFERIEURE de cout, jamais de concurrent de PPO.
#  Le nom a employer dans le papier est
#      "CPLEX offline perfect-information oracle"
#  et il ne doit jamais figurer dans la meme colonne que le rolling horizon.
# =============================================================================
d_full  = build_dims(INST, PRM)
sc_full = scenario_dict(SC0, d_full["G"], d_full["E"], d_full["N"], d_full["H"])
S0_full, L0_full = initial_state(d_full, SC0)
write_dat("data_full_oracle.dat", d_full, sc_full, SC0.demand, S0_full, L0_full)
print("data_full_oracle.dat ecrit  (G=3, E=4, N=28, H=3)")

_nGJA = sum(d_full["abar"][g, j] + 1 for g in range(d_full["G"])
            for j in range(d_full["Jg"][g]))
print(f"\ntaille attendue de l'instance complete :")
print(f"   |GJA| = {_nGJA}   ->  S, U, Q : environ {3*_nGJA*d_full['H']*d_full['N']:,} "
      f"variables continues")
print(f"   binaires FEFO : environ {_nGJA*d_full['H']*d_full['N']:,}")
print("   L'oracle 28 epoques avec FEFO est un GROS MILP : limite de temps et gap.")
print("   La comparaison equitable est le rolling horizon CORRIGE (cellules 9a-9f).")

24 scenarios routiniers tenus a l'ecart (seed 98765)
data_toy.dat ecrit  (G=2, E=4, N=2, H=2, abar<=3)
data_full_oracle.dat ecrit  (G=3, E=4, N=28, H=3)

taille attendue de l'instance complete :
   |GJA| = 123   ->  S, U, Q : environ 30,996 variables continues
   binaires FEFO : environ 10,332
   L'oracle 28 epoques avec FEFO est un GROS MILP : limite de temps et gap.
   La comparaison equitable est le rolling horizon CORRIGE (cellules 9a-9f).


In [6]:
def load(dat_path, model_path="model.mod"):
    a = AMPL()
    a.eval("option solver_msg 0;") 
    a.option["solver"] = "cplex"
    a.read(model_path)
    a.read_data(dat_path)
    return a


print("--- chargement de l'instance JOUET ---")
ampl_toy = load("data_toy.dat")
print("model.mod et data_toy.dat charges sans erreur de syntaxe.")
print(f"   G={int(ampl_toy.get_parameter('G').value())}  "
      f"E={int(ampl_toy.get_parameter('E').value())}  "
      f"N={int(ampl_toy.get_parameter('N').value())}  "
      f"H={int(ampl_toy.get_parameter('H').value())}")

--- chargement de l'instance JOUET ---
model.mod et data_toy.dat charges sans erreur de syntaxe.
   G=2  E=4  N=2  H=2


## Cellule 6 — Résolution avec CPLEX

Affiche le statut, la valeur de l'objectif, le temps, le MIP gap, le nombre de
variables et le nombre de contraintes.

In [7]:
import time


def solve(a, timelimit=600, mipgap=1e-4, verbose=True, label=""):
    """Resout avec CPLEX et renvoie un dictionnaire de statistiques."""
    a.option["cplex_options"] = (f"timelimit={timelimit} mipgap={mipgap} "
                                 f"return_mipgap=3")
    t0 = time.time()
    a.solve()
    dt = time.time() - t0

    nvar = int(a.get_value("_nvars"))
    ncon = int(a.get_value("_ncons"))
    obj  = a.get_objective("TotalCost").value()
    stat = a.get_value("solve_result")
    try:
        gap = float(a.get_value("TotalCost.relmipgap"))
    except Exception:
        gap = float("nan")

    if verbose:
        print("=" * 74)
        print(f"CPLEX  {label}")
        print("=" * 74)
        print(f"  statut de resolution   : {stat}")
        print(f"  objectif J (Eqs 42+43) : {obj:,.2f}")
        print(f"  temps de resolution    : {dt:.2f} s")
        print(f"  MIP gap relatif        : {gap:.3%}" if gap == gap
              else "  MIP gap relatif        : n/d")
        print(f"  variables              : {nvar:,}")
        print(f"  contraintes            : {ncon:,}")
        print("=" * 74)
    return dict(label=label, status=stat, objective=obj, seconds=dt, mipgap=gap,
                n_variables=nvar, n_constraints=ncon)


STATS_TOY = solve(ampl_toy, timelimit=120, label="instance jouet")
assert "solved" in str(STATS_TOY["status"]).lower(), \
    f"instance jouet non resolue : {STATS_TOY['status']}"
print("\nLa syntaxe AMPL et la formulation sont valides. On peut passer a la suite.")

CPLEX 22.2.0:   lim:time = 120
  mip:gap = 0.0001
  mip:return_gap = 3

suffix absmipgap OUT;
suffix relmipgap OUT;
CPLEX  instance jouet
  statut de resolution   : solved
  objectif J (Eqs 42+43) : -1,531.94
  temps de resolution    : 0.07 s
  MIP gap relatif        : 0.000%
  variables              : 610
  contraintes            : 722

La syntaxe AMPL et la formulation sont valides. On peut passer a la suite.


In [8]:
def extract(a, d):
    """Recupere les variables AMPL et les remet en tableaux NumPy 0-based."""
    G, E, N, H = d["G"], d["E"], d["N"], d["H"]
    Jm, Am = max(d["Jg"]), int(np.max(d["abar"])) + 1

    def arr(name, shape, age_pos=None):
        """age_pos : position (0-based) de l'indice de classe d'age, qui reste
        0-based dans AMPL ; tous les autres indices sont 1-based."""
        out = np.zeros(shape)
        df = a.get_variable(name).get_values().to_pandas()
        for idx, row in df.iterrows():
            key = idx if isinstance(idx, tuple) else (idx,)
            k = []
            for p, v in enumerate(key):
                v = int(v)
                k.append(v if (age_pos is not None and p == age_pos) else v - 1)
            out[tuple(k)] = float(row.iloc[0])
        return out

    return dict(
        V=arr("Rel", (G, N)), O=arr("Opn", (G, N)), K=arr("Kbal", (G, H, N)),
        P=arr("Prod", (G, Jm, E, H, N)), A=arr("Asg", (G, E, H, N)),
        Ch=arr("Chg", (G, G, E, H, N)),
        S=arr("Stk", (G, Jm, Am, H, N), age_pos=2),
        U=arr("Wdr", (G, Jm, Am, H, N), age_pos=2),
        Q=arr("Det", (G, Jm, Am, H, N), age_pos=2),
        X=arr("Exp", (G, Jm, H, N)),
        St=arr("Spre", (G, Jm, Am, N), age_pos=2),
        Ud=arr("Wdem", (G, Am, N), age_pos=1),
        F=arr("Ful", (G, N)), L=arr("Bkl", (G, N)), Z=arr("Idl", (E, H, N)),
        DL=arr("DLpos", (G, N)), Sh=arr("Short", (G, N)))


def cost_terms(a):
    """Decomposition de l'Eq (42) + la penalite de service de l'Eq (43)."""
    names = ["CostProcessing", "CostChangeover", "CostHolding", "CostBacklog",
             "CostDeterioration", "CostDisposal", "Revenue", "CostIdle",
             "ServicePenalty"]
    out = {}
    for nm in names:
        out[nm] = float(a.get_variable(nm).get_values().to_list()[0])
    out["TotalCost"] = a.get_objective("TotalCost").value()
    return out


SOL_TOY   = extract(ampl_toy, d_toy)
TERMS_TOY = cost_terms(ampl_toy)

print("Decomposition du cout, Eq (42) + penalite de service Eq (43)\n")
for k, v in TERMS_TOY.items():
    mark = "  (soustrait)" if k == "Revenue" else ""
    print(f"   {k:20s} {v:14,.2f}{mark}")

print(f"\nlancements V_gn (Eq 9) :\n{np.round(SOL_TOY['V'], 2)}")
print(f"\nindicateurs O_gn :\n{SOL_TOY['O'].astype(int)}")
print(f"\ndemande satisfaite F_gn (Eq 25) :\n{np.round(SOL_TOY['F'], 2)}")
print(f"\nbacklog L_gn (Eq 28) :\n{np.round(SOL_TOY['L'], 2)}")
print(f"\nquantites perimees X (Eq 17), total = {SOL_TOY['X'].sum():.2f}")
print(f"capacite inutilisee Z (Eq 39), total = {SOL_TOY['Z'].sum():.2f}")
print(f"changements de serie (Eq 36), total = {SOL_TOY['Ch'].sum():.0f}")

Decomposition du cout, Eq (42) + penalite de service Eq (43)

   CostProcessing             1,108.96
   CostChangeover                20.00
   CostHolding                  952.50
   CostBacklog                  887.49
   CostDeterioration             25.88
   CostDisposal                  -0.00
   Revenue                    2,258.43  (soustrait)
   CostIdle                      57.71
   ServicePenalty                 1.51
   TotalCost                 -1,531.94

lancements V_gn (Eq 9) :
[[200. 200.]
 [200. 200.]]

indicateurs O_gn :
[[1 1]
 [1 1]]

demande satisfaite F_gn (Eq 25) :
[[ 44.37 145.68]
 [ 35.78   0.  ]]

backlog L_gn (Eq 28) :
[[ 43.52   0.  ]
 [ 16.14 117.85]]

quantites perimees X (Eq 17), total = -0.00
capacite inutilisee Z (Eq 39), total = 57.71
changements de serie (Eq 36), total = 2


In [9]:
"""
Tests de validation post-resolution des Eqs (9)-(43).

Ce module ne connait NI PuLP NI AMPL : il prend la solution sous forme de tableaux
NumPy et verifie les equations une par une. Le meme fichier est utilise ici (solution
CPLEX directe) et dans CPLEX_Baseline.ipynb (solution AMPL), ce qui garantit que les
deux chemins sont juges par exactement le meme code.

Formes attendues dans `sol` :
    V  (G,N)          O  (G,N)            K  (G,H,N)
    P  (G,Jmax,E,H,N) A  (G,E,H,N)        Ch (G,G,E,H,N)
    S  (G,Jmax,A,H,N) U  (G,Jmax,A,H,N)   Q  (G,Jmax,A,H,N)
    X  (G,Jmax,H,N)   St (G,Jmax,A,N)     Ud (G,A,N)
    F  (G,N)          L  (G,N)            Z  (E,H,N)
"""
import numpy as np
import pandas as pd


def _survival(d, sc):
    G, N, H = d["G"], d["N"], d["H"]
    Jm, Am = max(d["Jg"]), int(np.max(d["abar"])) + 1
    Gam = np.ones((G, Jm, Am, H, N))
    for g in range(G):
        for j in range(d["Jg"][g]):
            for a in range(d["abar"][g, j] + 1):
                lam = (d["lam0"][g, j] + d["lam1"][g, j] * a
                       + d["lam2"][g, j] * sc["temp"] + d["lam3"][g, j] * sc["hum"]
                       + d["lam4"][g, j] * sc["dev"])                 # (H,N)
                Gam[g, j, a] = np.clip(np.exp(-np.maximum(lam, 0.0) * d["Delta"]), 0, 1)
    return Gam


def validate(d, sc, demand, S0, L0, sol, tol=1e-6):
    """Renvoie (DataFrame des tests, nombre de violations)."""
    G, E, N, H = d["G"], d["E"], d["N"], d["H"]
    Jg, abar, Egj = d["Jg"], d["abar"], d["Egj"]
    jf = {g: Jg[g] - 1 for g in range(G)}
    Gam = _survival(d, sc)
    V, O, K = sol["V"], sol["O"], sol["K"]
    P, A, Ch = sol["P"], sol["A"], sol["Ch"]
    S, U, Q, X = sol["S"], sol["U"], sol["Q"], sol["X"]
    St, Ud, F, L, Z = sol["St"], sol["Ud"], sol["F"], sol["L"], sol["Z"]
    rows = []

    def rec(eq, name, worst, ok_if_le=True):
        rows.append(dict(equation=eq, test=name, ecart_max=float(worst),
                         verdict="OK" if (worst <= tol) else "VIOLATION"))

    # -- 1. non-negativite  Eq (41) ----------------------------------------
    worst = 0.0
    for nm in ("V", "K", "P", "S", "U", "Q", "X", "St", "Ud", "F", "L", "Z"):
        worst = max(worst, float(np.max(np.maximum(-sol[nm], 0.0))))
    rec("(41)", "non-negativite de toutes les variables continues", worst)

    # -- 2. domaines binaires  Eq (40) -------------------------------------
    worst = 0.0
    for nm in ("O", "A", "Ch"):
        v = sol[nm]
        worst = max(worst, float(np.max(np.minimum(np.abs(v), np.abs(v - 1.0)))))
    rec("(40)", "O, A, C prennent des valeurs binaires", worst)

    # -- 3. bornes de lancement  Eq (9) ------------------------------------
    w = 0.0
    for g in range(G):
        for n in range(N):
            w = max(w, d["ell"][g] * O[g, n] - V[g, n],
                    V[g, n] - d["umax"][g] * O[g, n],
                    V[g, n] - sc["material"][g, n])
    rec("(9)", "ell*O <= V <= umax*O  et  V <= materiel", w)

    # -- 4. solde de lancement  Eqs (19)-(21) ------------------------------
    w19 = max(abs(K[g, 0, n] - V[g, n]) for g in range(G) for n in range(N))
    rec("(19)", "K[g,1,n] = V[g,n]", w19)
    w20 = w21 = 0.0
    for g in range(G):
        for n in range(N):
            for h in range(H):
                p0 = sum(P[g, 0, e, h, n] for e in Egj[g][0])
                w20 = max(w20, p0 - K[g, h, n])
                if h < H - 1:
                    w21 = max(w21, abs(K[g, h + 1, n] - (K[g, h, n] - p0)))
    rec("(20)", "traitement du 1er stade <= solde restant", w20)
    rec("(21)", "evolution du solde de lancement", w21)

    # -- 5. deterioration  Eq (12) -----------------------------------------
    w = 0.0
    for g in range(G):
        for j in range(Jg[g]):
            for a in range(abar[g, j] + 1):
                w = max(w, float(np.max(np.abs(
                    Q[g, j, a] - (1.0 - Gam[g, j, a]) * S[g, j, a]))))
    rec("(12)", "Q = (1 - Gamma) * S", w)

    # -- 6. retrait <= survivant  Eq (13) ----------------------------------
    w = 0.0
    for g in range(G):
        for j in range(Jg[g]):
            for a in range(abar[g, j] + 1):
                w = max(w, float(np.max(U[g, j, a] - Gam[g, j, a] * S[g, j, a])))
    rec("(13)", "0 <= U <= Gamma * S", w)

    # -- 7. coherence des stocks par age  Eqs (15)-(16) --------------------
    w15 = w16 = 0.0
    for g in range(G):
        for j in range(Jg[g]):
            for n in range(N):
                for h in range(H - 1):
                    w15 = max(w15, abs(S[g, j, 0, h + 1, n]
                                       - sum(P[g, j, e, h, n] for e in Egj[g][j])))
                    for a in range(abar[g, j]):
                        w16 = max(w16, abs(S[g, j, a + 1, h + 1, n]
                                           - (Gam[g, j, a, h, n] * S[g, j, a, h, n]
                                              - U[g, j, a, h, n])))
    rec("(15)", "material frais -> classe d'age 0", w15)
    rec("(16)", "vieillissement : S[a+1,h+1] = Gamma*S[a,h] - U[a,h]", w16)

    # -- 8. peremption  Eq (17) --------------------------------------------
    w = 0.0
    for g in range(G):
        for j in range(Jg[g]):
            ab = abar[g, j]
            w = max(w, float(np.max(np.abs(
                X[g, j] - (Gam[g, j, ab] * S[g, j, ab] - U[g, j, ab])))))
    rec("(17)", "peremption au-dela de l'age maximal", w)

    # -- 9. conservation des flux inter-stades  Eq (18) --------------------
    w = 0.0
    for g in range(G):
        for j in range(1, Jg[g]):
            for h in range(H):
                for n in range(N):
                    lhs = sum(P[g, j, e, h, n] for e in Egj[g][j])
                    rhs = sum(U[g, j - 1, a, h, n] for a in range(abar[g, j - 1] + 1))
                    w = max(w, abs(lhs - rhs))
    rec("(18)", "conservation des flux : production stade j = retraits stade j-1", w)

    # -- 10. satisfaction de la demande  Eq (25) ---------------------------
    w_le = w_min = 0.0
    for g in range(G):
        for n in range(N):
            Lp = L0[g] if n == 0 else L[g, n - 1]
            a1 = Lp + demand[g, n]
            a2 = sum(St[g, jf[g], a, n] for a in range(abar[g, jf[g]] + 1))
            w_le = max(w_le, F[g, n] - a1, F[g, n] - a2)
            w_min = max(w_min, abs(F[g, n] - min(a1, a2)))
    rec("(25)", "F <= min{L+D, stock} (les deux bornes)", w_le)
    rec("(25)", "le MIN est effectivement atteint (relaxation justifiee)", w_min)

    # -- 11. retraits de la demande  Eq (26) -------------------------------
    w26 = w26c = 0.0
    for g in range(G):
        for n in range(N):
            w26 = max(w26, abs(sum(Ud[g, a, n] for a in range(abar[g, jf[g]] + 1))
                               - F[g, n]))
            for a in range(abar[g, jf[g]] + 1):
                w26c = max(w26c, Ud[g, a, n] - St[g, jf[g], a, n])
    rec("(26)", "somme des retraits de demande = F", w26)
    rec("(26)", "retrait par classe <= stock pre-demande", w26c)

    # -- 12. bilan de backlog  Eq (28) -------------------------------------
    w = 0.0
    for g in range(G):
        for n in range(N):
            Lp = L0[g] if n == 0 else L[g, n - 1]
            w = max(w, abs(L[g, n] - (Lp + demand[g, n] - F[g, n])))
    rec("(28)", "equilibre du backlog", w)

    # -- 13. FEFO  Eqs (18) et (26) ----------------------------------------
    #  Regle : on ne retire d'une classe a que si toutes les classes PLUS VIEILLES
    #  (a' > a) sont entierement consommees.
    w = 0.0
    for g in range(G):
        for j in range(Jg[g]):
            if j == jf[g]:
                continue
            for h in range(H):
                for n in range(N):
                    for a in range(abar[g, j]):
                        if U[g, j, a, h, n] > tol:            # on touche la classe a
                            reste = sum(Gam[g, j, ap, h, n] * S[g, j, ap, h, n]
                                        - U[g, j, ap, h, n]
                                        for ap in range(a + 1, abar[g, j] + 1))
                            w = max(w, reste)                 # doit etre nul
    rec("(18)", "FEFO inter-stades : le plus vieux retire en premier", w)
    w = 0.0
    for g in range(G):
        ab = abar[g, jf[g]]
        for n in range(N):
            for a in range(ab):
                if Ud[g, a, n] > tol:
                    reste = sum(St[g, jf[g], ap, n] - Ud[g, ap, n]
                                for ap in range(a + 1, ab + 1))
                    w = max(w, reste)
    rec("(26)", "FEFO demande : le plus vieux retire en premier", w)

    # -- 14. lien P <-> affectation  Eq (30) -------------------------------
    w = 0.0
    for g in range(G):
        for j in range(Jg[g]):
            for e in Egj[g][j]:
                for h in range(H):
                    for n in range(N):
                        if A[g, e, h, n] < 0.5:
                            w = max(w, P[g, j, e, h, n])
    rec("(30)", "P = 0 si la machine n'est pas affectee au produit", w)

    # -- 15. une seule tache par machine  Eq (31) --------------------------
    w = 0.0
    for e in range(E):
        for h in range(H):
            for n in range(N):
                w = max(w, sum(A[g, e, h, n] for g in range(G) if d["eligible"][g, e])
                        - sc["avail"][e, h, n])
    rec("(31)", "au plus un produit par machine, et machine disponible", w)

    # -- 16. capacite machine  Eq (35)  et capacite inutilisee  Eq (39) ----
    wcap = widle = 0.0
    for e in range(E):
        for h in range(H):
            for n in range(N):
                proc = sum(d["tau"][g, j, e] * P[g, j, e, h, n]
                           for g in range(G) for j in range(Jg[g]) if e in Egj[g][j])
                setp = sum(d["upsilon"][g, gp, e] * Ch[g, gp, e, h, n]
                           for g in range(G) for gp in range(G) if gp != g
                           and d["eligible"][g, e] and d["eligible"][gp, e])
                cap = d["Cbar"][e, h, n] * sc["avail"][e, h, n]
                wcap = max(wcap, proc + setp - cap)
                widle = max(widle, abs(Z[e, h, n] - (cap - proc - setp)))
    rec("(35)", "temps de traitement + changement <= capacite disponible", wcap)
    rec("(39)", "capacite inutilisee Z bien calculee", widle)

    # -- 17. changement de serie  Eq (36) ----------------------------------
    w = 0.0
    for g in range(G):
        for gp in range(G):
            if gp == g:
                continue
            for e in range(E):
                if not (d["eligible"][g, e] and d["eligible"][gp, e]):
                    continue
                for h in range(H):
                    for n in range(N):
                        if h > 0:
                            hm, nm = h - 1, n
                        elif n > 0:
                            hm, nm = H - 1, n - 1
                        else:
                            continue
                        w = max(w, (A[g, e, h, n] + A[gp, e, hm, nm] - 1)
                                - Ch[g, gp, e, h, n])
    rec("(36)", "changement active quand deux affectations consecutives different", w)

    # -- 18. ressource partagee  Eq (37) -----------------------------------
    w = 0.0
    for h in range(H):
        for n in range(N):
            w = max(w, sum(d["psi"][g, j, e] * P[g, j, e, h, n]
                           for g in range(G) for j in range(Jg[g]) for e in Egj[g][j])
                    - sc["Rbar"][h, n])
    rec("(37)", "consommation de ressource partagee <= capacite", w)

    # -- 19. main-d'oeuvre  Eq (38) ----------------------------------------
    w = 0.0
    for h in range(H):
        for n in range(N):
            w = max(w, sum(A[g, e, h, n] for g in range(G) for e in range(E)
                           if d["eligible"][g, e]) - np.floor(sc["Wbar"][h, n] + 1e-9))
    rec("(38)", "affectations actives <= main-d'oeuvre disponible", w)

    # -- 20. report d'epoque  Eqs (24) et (27) -----------------------------
    w24 = w27 = 0.0
    for g in range(G):
        for n in range(N - 1):
            for j in range(Jg[g]):
                if j < jf[g]:
                    for a in range(abar[g, j] + 1):
                        w24 = max(w24, abs(S[g, j, a, 0, n + 1] - St[g, j, a, n]))
            for a in range(abar[g, jf[g]] + 1):
                w27 = max(w27, abs(S[g, jf[g], a, 0, n + 1]
                                   - (St[g, jf[g], a, n] - Ud[g, a, n])))
    rec("(24)", "report des stades intermediaires vers l'epoque suivante", w24)
    rec("(27)", "report du stade final apres prelevement de la demande", w27)

    df = pd.DataFrame(rows)
    return df, int((df["verdict"] == "VIOLATION").sum())


def service_levels(d, demand, L0, L):
    """SL_gn de l'Eq (29), calcule apres la resolution."""
    G, N = d["G"], d["N"]
    SL = np.zeros((G, N))
    for g in range(G):
        for n in range(N):
            Lp = L0[g] if n == 0 else L[g, n - 1]
            SL[g, n] = 1.0 - max(L[g, n] - Lp, 0.0) / (demand[g, n] + d["eps0"])
    return SL


DF_TOY, NBAD_TOY = validate(d_toy, sc_toy, SC0.demand[:2, :2], S0_toy, L0_toy,
                            SOL_TOY, tol=1e-6)
import pandas as pd
pd.set_option("display.width", 200, "display.max_rows", 80,
              "display.max_colwidth", 66)
print(DF_TOY.to_string(index=False))
print(f"\n{len(DF_TOY)} tests   |   violations : {NBAD_TOY}")
if NBAD_TOY:
    print("\nDES CONTRAINTES SONT VIOLEES. Ne pas exploiter ces resultats.")
    display(DF_TOY[DF_TOY["verdict"] == "VIOLATION"])
else:
    print("Toutes les equations sont satisfaites a 1e-6 pres.")

SL_TOY = service_levels(d_toy, SC0.demand[:2, :2], L0_toy, SOL_TOY["L"])
print(f"\nniveau de service SL_gn (Eq 29) :\n{np.round(SL_TOY, 4)}")
print(f"SL moyen = {SL_TOY.mean():.4f}   |   cible SL^min = {d_toy['SLmin']}")

equation                                                             test  ecart_max verdict
    (41)                 non-negativite de toutes les variables continues  2.842e-14      OK
    (40)                            O, A, C prennent des valeurs binaires  0.000e+00      OK
     (9)                          ell*O <= V <= umax*O  et  V <= materiel  0.000e+00      OK
    (19)                                                K[g,1,n] = V[g,n]  0.000e+00      OK
    (20)                         traitement du 1er stade <= solde restant  0.000e+00      OK
    (21)                                  evolution du solde de lancement  7.105e-15      OK
    (12)                                              Q = (1 - Gamma) * S  3.241e-09      OK
    (13)                                              0 <= U <= Gamma * S  3.241e-09      OK
    (15)                                 material frais -> classe d'age 0  0.000e+00      OK
    (16)              vieillissement : S[a+1,h+1] = Gamma*S[a,h] - U[a

In [10]:
def diagnose_infeasible(a):
    """Demande a CPLEX un IIS et affiche les contraintes en cause."""
    a.option["cplex_options"] = "iisfind=1"
    a.solve()
    res = str(a.get_value("solve_result"))
    print(f"statut : {res}")
    if "infeasible" not in res.lower():
        print("Le modele n'est pas infaisable : rien a diagnostiquer.")
        return None
    rows = []
    for name, con in a.get_constraints():
        try:
            df = con.get_values().to_pandas()
        except Exception:
            continue
        if "iis" in [c.lower() for c in df.columns]:
            col = [c for c in df.columns if c.lower() == "iis"][0]
            bad = df[df[col].astype(str).str.lower() != "non"]
            for idx in bad.index:
                rows.append(dict(contrainte=name, indice=str(idx)))
    out = pd.DataFrame(rows)
    if len(out):
        print(f"\n{len(out)} contraintes dans l'IIS :\n")
        display(out.head(60))
        print("\nPistes de correction, par ordre de frequence :")
        print("  CAPACITY / WORKFORCE  -> capacite ou main-d'oeuvre insuffisante pour")
        print("     servir la demande : verifier avail, Wcap et Cbar du scenario.")
        print("  RELEASE_LO            -> ell_g > min{u_g, m_gn} : lot minimal")
        print("     inatteignable, l'Eq (8) impose alors V = 0 (premier cas).")
        print("  INIT_STOCK / CARRY_*  -> incoherence des conditions initiales.")
        print("  FEFO_*                -> big-M trop petit : verifier Mfefo.")
    else:
        print("Aucune information IIS renvoyee par CPLEX.")
    return out


# decommente si la resolution echoue :
# IIS = diagnose_infeasible(ampl_toy)
print("Fonction de diagnostic prete (a appeler seulement en cas d'infaisabilite).")

Fonction de diagnostic prete (a appeler seulement en cas d'infaisabilite).


## Cellule 8 — Export `cplex_results.csv`

Format **long** (`scenario, mode, epoch, product, metric, value`), choisi pour que la
comparaison avec les sorties PPO se fasse par une simple jointure, sans avoir à
réconcilier des largeurs de colonnes différentes. Un second fichier agrégé, une ligne
par scénario, reprend les mêmes colonnes que le tableau de comparaison du notebook PPO.

In [11]:
def results_long(sol, d, demand, L0, scenario_id, mode, terms, stats):
    """Une ligne par (epoque, produit, metrique). Colonnes stables."""
    G, N = d["G"], d["N"]
    SL = service_levels(d, demand, L0, sol["L"])
    rows = []
    per_gn = {
        "release_V_Eq9":        sol["V"],
        "open_O_Eq9":           sol["O"],
        "fulfilled_F_Eq25":     sol["F"],
        "backlog_L_Eq28":       sol["L"],
        "demand_D":             demand,
        "service_level_Eq29":   SL,
    }
    for name, arr_ in per_gn.items():
        for g in range(G):
            for n in range(N):
                rows.append(dict(scenario=scenario_id, mode=mode, epoch=n + 1,
                                 product=g + 1, metric=name,
                                 value=float(arr_[g, n])))
    # quantites agregees par epoque
    for g in range(G):
        for n in range(N):
            rows.append(dict(scenario=scenario_id, mode=mode, epoch=n + 1,
                             product=g + 1, metric="produced_P_total",
                             value=float(sol["P"][g, :, :, :, n].sum())))
            rows.append(dict(scenario=scenario_id, mode=mode, epoch=n + 1,
                             product=g + 1, metric="expired_X_Eq17",
                             value=float(sol["X"][g, :, :, n].sum())))
            rows.append(dict(scenario=scenario_id, mode=mode, epoch=n + 1,
                             product=g + 1, metric="deteriorated_Q_Eq12",
                             value=float(sol["Q"][g, :, :, :, n].sum())))
            for a in range(sol["St"].shape[2]):
                rows.append(dict(scenario=scenario_id, mode=mode, epoch=n + 1,
                                 product=g + 1, metric=f"stock_by_age_a{a}",
                                 value=float(sol["St"][g, d["Jg"][g] - 1, a, n])))
    for e in range(d["E"]):
        for n in range(N):
            rows.append(dict(scenario=scenario_id, mode=mode, epoch=n + 1,
                             product=0, metric=f"idle_capacity_Z_e{e+1}_Eq39",
                             value=float(sol["Z"][e, :, n].sum())))
            rows.append(dict(scenario=scenario_id, mode=mode, epoch=n + 1,
                             product=0, metric=f"assignments_A_e{e+1}",
                             value=float(sol["A"][:, e, :, n].sum())))
            rows.append(dict(scenario=scenario_id, mode=mode, epoch=n + 1,
                             product=0, metric=f"changeovers_C_e{e+1}_Eq36",
                             value=float(sol["Ch"][:, :, e, :, n].sum())))
    for k, v in terms.items():
        rows.append(dict(scenario=scenario_id, mode=mode, epoch=0, product=0,
                         metric=f"cost_{k}", value=float(v)))
    for k in ("objective", "seconds", "mipgap", "n_variables", "n_constraints"):
        rows.append(dict(scenario=scenario_id, mode=mode, epoch=0, product=0,
                         metric=f"solve_{k}", value=float(stats[k])))
    return pd.DataFrame(rows)


def results_summary(sol, d, demand, L0, scenario_id, mode, terms, stats):
    """Une ligne par scenario, memes colonnes que le tableau PPO."""
    SL = service_levels(d, demand, L0, sol["L"])
    return pd.DataFrame([dict(
        scenario=scenario_id, mode=mode,
        policy_cost_J=terms["TotalCost"],
        operating_cost_G=terms["TotalCost"] - terms["ServicePenalty"],
        mean_service_level=float(SL.mean()),
        min_service_level=float(SL.min()),
        mean_backlog=float(sol["L"].mean()),
        expired_units=float(sol["X"].sum()),
        deteriorated_units=float(sol["Q"].sum()),
        mean_utilisation=float(1.0 - sol["Z"].sum()
                               / max(d["Cbar"].sum(), 1e-9)),
        total_release=float(sol["V"].sum()),
        total_fulfilled=float(sol["F"].sum()),
        changeovers=float(sol["Ch"].sum()),
        solve_seconds=stats["seconds"], mipgap=stats["mipgap"],
        status=stats["status"])])


LONG_TOY = results_long(SOL_TOY, d_toy, SC0.demand[:2, :2], L0_toy,
                        "toy", "verification", TERMS_TOY, STATS_TOY)
SUM_TOY  = results_summary(SOL_TOY, d_toy, SC0.demand[:2, :2], L0_toy,
                           "toy", "verification", TERMS_TOY, STATS_TOY)
LONG_TOY.to_csv("cplex_results.csv", index=False)
SUM_TOY.to_csv("cplex_results_summary.csv", index=False)
print(f"cplex_results.csv         : {len(LONG_TOY):,} lignes")
print(f"cplex_results_summary.csv : {len(SUM_TOY)} ligne(s)\n")
display(SUM_TOY.T)

cplex_results.csv         : 91 lignes
cplex_results_summary.csv : 1 ligne(s)



,0
scenario,toy
mode,verification
policy_cost_J,-1531.935
operating_cost_G,-1533.441
mean_service_level,0.549
min_service_level,0.0
mean_backlog,44.375
expired_units,-0.0
deteriorated_units,3.697
mean_utilisation,0.549


In [12]:
"""Cellule 9 (ANCIENNE VERSION) -- CONSERVEE POUR TRACABILITE, NEUTRALISEE.

Cette fonction est celle du notebook d'origine. Elle est gardee ici parce qu'elle
a produit les resultats deja diffuses et qu'un audit doit pouvoir la relire, mais
elle NE DOIT PLUS ETRE APPELEE : elle contient trois defauts qui invalident toute
comparaison avec PPO.

  DEFAUT 1 -- FUITE D'INFORMATION FUTURE.
      scw = dict(avail=scenario.avail[:, :, n0:n0 + W_eff],
                 temp=scenario.temp[:, n0:n0 + W_eff], ...)
      La fenetre recevait les REALISATIONS de avail, temp, hum, dev, Rbar, Wbar et
      material sur tout [n0, n0+W). CPLEX voyait donc une panne, un manque de
      personnel ou une rupture de chaine du froid AVANT qu'ils se produisent.
      Remplace par observed_window_view() -- cellule 9a.

  DEFAUT 2 -- COUT SANS SIGNIFICATION.
      for k, v in tw.items():
          terms_tot[k] = terms_tot.get(k, 0.0) + v / W_eff
      Le cout d'une fenetre ENTIERE divise par sa longueur, puis somme sur des
      fenetres qui se RECOUVRENT. Ni le cout de l'epoque appliquee, ni une moyenne,
      ni rien d'interpretable. Et `objective` valait NaN.
      Remplace par le cout de la trajectoire executee, Eq (42) + Eq (43) --
      cellules 9d et 9f, verifie par le test 4.

  DEFAUT 3 -- AFFECTATION PRECEDENTE OUBLIEE.
      Chaque fenetre repartait d'une machine reputee inactive : le changement de
      serie du premier intervalle n'etait ni facture (kappa) ni impute a la
      capacite (upsilon), soit jusqu'a N x E changements gratuits.
      Remplace par PrevAsg + CHANGEOVER_PREV -- test 3.

Un quatrieme point, moins visible : le plan n'etait jamais ENGAGE. Les Eqs (32)-(34)
etaient declarees "sans objet", ce qui revenait a laisser CPLEX re-optimiser sans
jamais payer la stabilite de planning que le PDF impose (section 3.3.3).
"""


def rolling_horizon(scenario, W=3, timelimit=120, mipgap=5e-3, verbose=False):
    raise NotImplementedError(
        "rolling_horizon() est la version FAUSSE, conservee pour tracabilite.\n"
        "  defaut 1 : realisations futures transmises au solveur\n"
        "  defaut 2 : cout de fenetre divise par W_eff puis somme sur des fenetres\n"
        "             qui se recouvrent -> objectif NaN\n"
        "  defaut 3 : affectation precedente oubliee entre deux fenetres\n"
        "Utilise rolling_horizon_corrected() -- cellule 9f.")


print("Ancienne rolling_horizon() conservee mais NEUTRALISEE.")
print("La version utilisable est rolling_horizon_corrected(), cellule 9f.")

Ancienne rolling_horizon() conservee mais NEUTRALISEE.
La version utilisable est rolling_horizon_corrected(), cellule 9f.


In [13]:
from scipy.stats import gamma as gamma_dist


# --- 1. repere newsvendor classique, Eqs (2)-(7) ------------------------------
def newsvendor_reference(sol, d, scenario, L0):
    """EIP Eq (2), fractile Eq (4), cible Eq (3), lancement Eq (7).
    CALCULE APRES LA RESOLUTION. Ce n'est PAS une contrainte du modele."""
    G, N = d["G"], d["N"]
    rows = []
    for g in range(G):
        jf = d["Jg"][g] - 1
        for n in range(N):
            # Eq (2) : position de stock effective, evaluee au 1er intervalle
            eip = sum(sol["S"][g, jf, a, 0, n] for a in range(d["abar"][g, jf] + 1))
            for j in range(d["Jg"][g] - 1):
                eip += PRM.omega[g, j] * sum(sol["S"][g, j, a, 0, n]
                                             for a in range(d["abar"][g, j] + 1))
            Lp = L0[g] if n == 0 else sol["L"][g, n - 1]
            eip -= Lp
            # Eqs (5)-(6) : les quantites barrees ne sont PAS definies dans le PDF.
            # On reprend le choix du notebook PPO et on le declare.
            u = PRM.beta[g] + PRM.rho[g, n] + PRM.theta_service[g] * Lp
            o = float(np.mean([PRM.mu[g, j, e] for j in range(d["Jg"][g])
                               for e in d["Egj"][g][j]])) \
                + PRM.alpha[g, 0] + PRM.delta[g, jf] - PRM.sigma[g, jf]
            chi = float(np.clip(u / max(u + o, 1e-9), PRM.epsilon, 1 - PRM.epsilon))
            Y = float(scenario.quantile(g, n, chi))                  # Eq (3)
            Vnv, Onv = project_release(max(Y - eip, 0.0), PRM.ell[g], PRM.umax[g],
                                       scenario.material[g, n])      # Eq (7)
            rows.append(dict(product=g + 1, epoch=n + 1, EIP_Eq2=eip,
                             chi_NV_Eq4=chi, Y_NV_Eq3=Y, V_NV_Eq7=Vnv,
                             V_CPLEX_Eq9=sol["V"][g, n],
                             ecart=sol["V"][g, n] - Vnv))
    return pd.DataFrame(rows)


# --- 2. recuperation, Eqs (44)-(45) -------------------------------------------
def recovery_metrics(SL, d, iota0=0, w_conf_grid=(1, 2, 3)):
    """H^rec de l'Eq (45), pour plusieurs fenetres de confirmation.

    w_conf est DEFINI par l'Eq (45) du PDF ; c'est sa VALEUR NUMERIQUE qui n'est
    donnee ni par le PDF ni par le notebook PPO -- "symbolically defined but
    numerically unspecified". On ne peut donc pas rapporter UNE valeur de H^rec,
    seulement une sensibilite. De meme, eta_rec est defini par l'Eq (43) mais non
    chiffre : le terme eta_rec * H^rec reste hors de l'objectif optimise.

    En mode oracle hors-ligne il n'y a ni engagement ni reparation, donc xi = 0 et
    iota0 de l'Eq (44) n'est pas defini : il doit etre fourni de l'exterieur, par
    convention l'epoque de debut de la perturbation du scenario.

    EN MODE ROLLING HORIZON CORRIGE cette limite disparait : les declencheurs de
    l'Eq (32) sont reellement calcules, iota0 est celui de l'Eq (44), et la version
    a utiliser est recovery_metrics_eq45() de la cellule 9j."""
    G, N = d["G"], d["N"]
    ok = np.all(SL >= d["SLmin"][:, None], axis=0)          # (N,) toutes familles
    rows = []
    for w in w_conf_grid:
        rec = None
        for n in range(iota0, N):
            end = min(n + w, N)
            if end > n and ok[n:end].all():
                rec = n - iota0
                break
        rows.append(dict(w_conf=w,
                         H_rec_Eq45=(rec if rec is not None else N - iota0),
                         censure=(rec is None)))
    return pd.DataFrame(rows)


# --- 3. effet de fin d'horizon (audit section 4.5) ----------------------------
def horizon_end_effect(sol, d, k=3):
    """Le stock final n'a aucune valeur de recuperation (Eqs 24 et 27 ne sont pas
    definies a l'epoque terminale) : le modele est donc incite a ne rien produire
    dans les dernieres epoques. On mesure l'ampleur du biais.

    Signe : une valeur NEGATIVE est une chute de production en fin d'horizon,
    c'est le biais attendu. Une valeur positive signale l'inverse et merite d'etre
    expliquee -- typiquement la penalite de capacite inutilisee eta_idle * Z de
    l'Eq (42), qui pousse a produire meme sans demande a servir."""
    N = d["N"]
    if N <= k:
        return dict(production_moyenne=float("nan"),
                    production_k_dernieres=float("nan"),
                    variation_relative_pct=float("nan"),
                    note=f"horizon trop court (N={N} <= k={k})")
    prod = np.array([sol["P"][:, :, :, :, n].sum() for n in range(N)])
    return dict(production_moyenne=float(prod[:-k].mean()),
                production_k_dernieres=float(prod[-k:].mean()),
                variation_relative_pct=float(
                    100 * (prod[-k:].mean() - prod[:-k].mean())
                    / max(prod[:-k].mean(), 1e-9)),
                note="negatif = chute de fin d'horizon (biais attendu)")


NV_TOY = newsvendor_reference(SOL_TOY, d_toy, SC0, L0_toy)
print("Repere newsvendor classique (Eqs 2-7), calcule APRES la resolution :\n")
display(NV_TOY.round(3))
print("\nEcart moyen CPLEX - newsvendor sur le lancement : "
      f"{NV_TOY['ecart'].mean():+.2f} unites")

print("\n\nRecuperation, Eq (45), sensibilite a w_conf")
print("(w_conf est defini par l'Eq (45) mais numeriquement non specifie ;")
print(" eta_rec est defini par l'Eq (43) mais numeriquement non specifie).")
print("Version rolling horizon, avec de VRAIS declencheurs : cellule 9j.\n")
display(recovery_metrics(SL_TOY, d_toy))

print("\n\nEffet de fin d'horizon (audit section 4.5) :")
print("   (sur l'instance JOUET a 2 epoques ce chiffre n'a aucun sens ;")
print("    il devient lisible sur l'instance complete a 28 epoques, k=3)")
for k, v in horizon_end_effect(SOL_TOY, d_toy, k=1).items():
    print(f"   {k:26s} {v if isinstance(v, str) else f'{v:12,.2f}'}")

Repere newsvendor classique (Eqs 2-7), calcule APRES la resolution :



,product,epoch,EIP_Eq2,chi_NV_Eq4,Y_NV_Eq3,V_NV_Eq7,V_CPLEX_Eq9,ecart
0,1,1,44.785,0.750,101.791,57.006,200.0,142.994
1,1,2,155.829,0.775,106.719,0.000,200.0,200.000
2,2,1,36.193,0.750,86.159,49.967,200.0,150.033
3,2,2,-16.135,0.760,79.027,95.162,200.0,104.838



Ecart moyen CPLEX - newsvendor sur le lancement : +149.47 unites


Recuperation, Eq (45), sensibilite a w_conf
(w_conf est defini par l'Eq (45) mais numeriquement non specifie ;
 eta_rec est defini par l'Eq (43) mais numeriquement non specifie).
Version rolling horizon, avec de VRAIS declencheurs : cellule 9j.



,w_conf,H_rec_Eq45,censure
0,1,2,True
1,2,2,True
2,3,2,True




Effet de fin d'horizon (audit section 4.5) :
   (sur l'instance JOUET a 2 epoques ce chiffre n'a aucun sens ;
    il devient lisible sur l'instance complete a 28 epoques, k=3)
   production_moyenne               350.00
   production_k_dernieres           789.60
   variation_relative_pct           125.60
   note                       negatif = chute de fin d'horizon (biais attendu)


In [14]:
"""Configuration Case C et extension ADDITIVE du modèle AMPL.

Le fichier original `model.mod` n'est pas détruit. On construit `model_case_c.mod`
en lui ajoutant seulement :
  * une bibliothèque de plans + un binaire UsePlan[p,h,n] ;
  * une bibliothèque de lancements accessibles + UseRel[g,r,n] ;
  * FIX_RELEASE pour h>0 (K restant, pas de nouveau lancement).
"""
PLAN_COMMITMENT = False
FORECAST_FEATURES = True
ROUTINE_BURST_ENABLED = True
REPAIR_IN_STATE = True
REPAIR_CLIP = 9.0
ROUTINE_BURST_PROB = 0.5
ROUTINE_BURST_LEN = (2, 4)
REPAIR_EST_CV = 0.30

# Référence exacte : case_C_severity (3).ipynb
# >>> SEUL PARAMETRE A CHANGER ENTRE LES TROIS RUNS <<<
SEVERITY = "moderate"   # "mild" | "moderate" | "severe"
SEVERITY_LEVELS = {
    "mild":     dict(k_frac=0.25, wf_frac=0.75, temp_rise=8.0),
    "moderate": dict(k_frac=0.50, wf_frac=0.50, temp_rise=15.0),
    "severe":   dict(k_frac=0.75, wf_frac=0.25, temp_rise=22.0),
}
assert SEVERITY in SEVERITY_LEVELS
SEV = SEVERITY_LEVELS[SEVERITY]

EVAL_SEED = 98765
N_ROUTINE = 24
STRESS_SEED = 424242
DISRUPTION_FAMILIES = ("line_failure", "workforce_shortage", "cold_chain")
N_PER_FAMILY = 60

CHI_GRID_SIZE = 201
CHI_GRID = np.linspace(PRM.epsilon, 1.0 - PRM.epsilon, CHI_GRID_SIZE)
CHI_GRID_STEP = float(CHI_GRID[1] - CHI_GRID[0])
CHI_MAX_ABS_DISCRETIZATION_ERROR = CHI_GRID_STEP / 2.0
LEX_PRIMARY_TOL = 1e-6

RUN_ROUTINE_24 = False
RUN_DISRUPTIONS_180 = False


def build_case_c_model(base_model: str) -> str:
    m = str(base_model)

    anchor = "set GGE dimen 3;                    # (g,gp,e)  : changement de gp vers g possible sur e\n"
    addition = r'''
# --- [CASE C] bibliotheques d'actions finies ---------------------------------
param PCOUNT >= 1 integer default 1;
set PLANS := 1..PCOUNT;
param PlanAsg{GE, PLANS} binary default 0;
param PLAN_LIBRARY_ON binary default 0;

param RCOUNT >= 1 integer default 1;
set RELCANDS := 1..RCOUNT;
param RelCand{PRODUCTS, RELCANDS, EPOCHS} >= 0 default 0;
param RelOpen{PRODUCTS, RELCANDS, EPOCHS} binary default 0;
param RELEASE_LIBRARY_ON binary default 0;
'''
    assert m.count(anchor) == 1
    m = m.replace(anchor, anchor + addition, 1)

    anchor = "param Opn0{PRODUCTS} binary default 0; # O_gn deja engage au debut de l'epoque\n"
    addition = r'''
# [CASE C] h>0 : le lancement n'est plus une decision, K0 est le solde restant.
param FIX_RELEASE binary default 0;
'''
    assert m.count(anchor) == 1
    m = m.replace(anchor, anchor + addition, 1)

    anchor = "var Chg{GGE, INTERVALS, EPOCHS} binary;             # C_gg'ehn changement de serie\n"
    addition = r'''
# [CASE C] un seul plan de la bibliotheque par intervalle
var UsePlan{PLANS, INTERVALS, EPOCHS} binary;
# [CASE C] un seul lancement accessible par produit/epoque a h=0
var UseRel{PRODUCTS, RELCANDS, EPOCHS} binary;
'''
    assert m.count(anchor) == 1
    m = m.replace(anchor, anchor + addition, 1)

    m = m.replace(
        "subject to RELEASE_LO{g in PRODUCTS, n in EPOCHS: REPAIR = 0}:",
        "subject to RELEASE_LO{g in PRODUCTS, n in EPOCHS: REPAIR = 0 and FIX_RELEASE = 0}:"
    )
    m = m.replace(
        "subject to RELEASE_HI{g in PRODUCTS, n in EPOCHS: REPAIR = 0}:",
        "subject to RELEASE_HI{g in PRODUCTS, n in EPOCHS: REPAIR = 0 and FIX_RELEASE = 0}:"
    )
    m = m.replace(
        "subject to RELEASE_MAT{g in PRODUCTS, n in EPOCHS: REPAIR = 0}:",
        "subject to RELEASE_MAT{g in PRODUCTS, n in EPOCHS: REPAIR = 0 and FIX_RELEASE = 0}:"
    )

    anchor = """subject to REPAIR_FIX_OPN{g in PRODUCTS: REPAIR = 1}:
    Opn[g,1] = Opn0[g];
"""
    addition = r'''
# --- [CASE C] lancement fixe apres h=0 ----------------------------------------
subject to FIX_RELEASE_REL{g in PRODUCTS, n in EPOCHS: FIX_RELEASE = 1}:
    Rel[g,n] = K0[g];
subject to FIX_RELEASE_OPN{g in PRODUCTS, n in EPOCHS: FIX_RELEASE = 1}:
    Opn[g,n] = Opn0[g];

# --- [CASE C] lancement accessible via grille de chi -------------------------
subject to REL_GRID_ONE{g in PRODUCTS, n in EPOCHS:
                        RELEASE_LIBRARY_ON = 1 and FIX_RELEASE = 0 and REPAIR = 0}:
    sum{r in RELCANDS} UseRel[g,r,n] = 1;
subject to REL_GRID_VALUE{g in PRODUCTS, n in EPOCHS:
                          RELEASE_LIBRARY_ON = 1 and FIX_RELEASE = 0 and REPAIR = 0}:
    Rel[g,n] = sum{r in RELCANDS} RelCand[g,r,n] * UseRel[g,r,n];
subject to REL_GRID_OPEN{g in PRODUCTS, n in EPOCHS:
                         RELEASE_LIBRARY_ON = 1 and FIX_RELEASE = 0 and REPAIR = 0}:
    Opn[g,n] = sum{r in RELCANDS} RelOpen[g,r,n] * UseRel[g,r,n];
'''
    assert m.count(anchor) == 1
    m = m.replace(anchor, anchor + addition, 1)

    anchor = """subject to FIX_ASG{(g,e) in GE, h in INTERVALS, n in EPOCHS: FixAsg[g,e,h,n] >= 0}:
    Asg[g,e,h,n] = FixAsg[g,e,h,n];
"""
    addition = r'''
# --- [CASE C] exactement un plan admissible par intervalle -------------------
subject to PLAN_ONE{h in INTERVALS, n in EPOCHS: PLAN_LIBRARY_ON = 1}:
    sum{p in PLANS} UsePlan[p,h,n] = 1;
subject to PLAN_LINK{(g,e) in GE, h in INTERVALS, n in EPOCHS:
                     PLAN_LIBRARY_ON = 1}:
    Asg[g,e,h,n] = sum{p in PLANS} PlanAsg[g,e,p] * UsePlan[p,h,n];
'''
    assert m.count(anchor) == 1
    m = m.replace(anchor, anchor + addition, 1)
    return m


MODEL_MOD_CASE_C = build_case_c_model(MODEL_MOD)
with open("model_case_c.mod", "w") as f:
    f.write(MODEL_MOD_CASE_C)

print("model_case_c.mod écrit :", len(MODEL_MOD_CASE_C.splitlines()), "lignes")
print(f"grille chi : {CHI_GRID_SIZE} points, pas={CHI_GRID_STEP:.6f}, "
      f"erreur |chi-chi_grid| <= {CHI_MAX_ABS_DISCRETIZATION_ERROR:.6f}")
print("RUN_ROUTINE_24 =", RUN_ROUTINE_24, "| RUN_DISRUPTIONS_180 =", RUN_DISRUPTIONS_180)


model_case_c.mod écrit : 465 lignes
grille chi : 201 points, pas=0.004900, erreur |chi-chi_grid| <= 0.002450
RUN_ROUTINE_24 = False | RUN_DISRUPTIONS_180 = False


In [15]:
import copy
import hashlib
import itertools


# ============================================================================
# CASE C — générateur EXACT du PPO de référence case_C_severity (3).ipynb
# ============================================================================

def _flatten_ehn_time(x):
    x = np.asarray(x)
    return x.transpose(0, 2, 1).reshape(x.shape[0], -1)


def _unflatten_ehn_time(x, H, N):
    x = np.asarray(x)
    return x.reshape(x.shape[0], N, H).transpose(0, 2, 1)


def build_repair_estimate_case_c(avail, rng, cv=0.30):
    """Copie de la règle PPO Case C.

    Une estimation est tirée UNE FOIS au début d'une panne :
        round(true_duration * (1 + Normal(0, cv)))
    avec un minimum de 1 intervalle, puis elle est décrémentée de 1.
    La valeur brute peut dépasser REPAIR_CLIP ; l'observation, comme dans PPO,
    est plafonnée à REPAIR_CLIP.
    """
    E, H, N = avail.shape
    flat = avail.transpose(0, 2, 1).reshape(E, N * H)
    est = np.zeros_like(flat)
    for e in range(E):
        t = 0
        while t < N * H:
            if flat[e, t] > 0.5:
                t += 1
                continue
            vraie_duree = 1
            while t + vraie_duree < N * H and flat[e, t + vraie_duree] < 0.5:
                vraie_duree += 1
            estimation = max(1, int(round(vraie_duree * (1.0 + rng.normal(0, cv)))))
            for k in range(vraie_duree):
                est[e, t + k] = max(1, estimation - k)
            t += vraie_duree
    return est.reshape(E, N, H).transpose(0, 2, 1)


class CaseCScenarioGenerator:
    """Même générateur que le PPO Case C de référence."""

    def __init__(self, inst, prm, nominal, seed=0):
        self.inst, self.prm = inst, prm
        self.rng = np.random.default_rng(seed)
        # Streams dédiés : exactement ceux du PPO.
        self.burst_rng = np.random.default_rng(seed + 99991)
        self.disr_rng = np.random.default_rng(seed + 77771)

        G, E, N, H = inst.G, inst.E, inst.N, inst.H
        self.base_demand = np.asarray(nominal["demand"], dtype=float)
        self.season_amp = 0.15 + 0.15 * self.rng.random(G)
        self.season_phase = 2 * math.pi * self.rng.random(G)
        self.cv_range = (0.18, 0.32)

        self.Rbar_nominal = float(nominal["Rbar"])
        self.Wbar_nominal = float(nominal["Wbar"])
        self.routine_avail_floor = int(math.ceil(ROUTINE_AVAIL_MIN_FRACTION * E))
        self.Wbar_profile = np.asarray(
            nominal.get("Wbar_profile", np.full(inst.H, self.Wbar_nominal)), dtype=float)
        self.routine_workforce_floor = np.maximum(
            1.0, np.round(0.85 * self.Wbar_profile))
        self.routine_Rbar_floor = 0.85 * self.Rbar_nominal

    def sample_routine(self):
        inst, rng = self.inst, self.rng
        G, E, N, H = inst.G, inst.E, inst.N, inst.H

        n_idx = np.arange(N)
        season = 1.0 + self.season_amp[:, None] * np.sin(
            2 * math.pi * n_idx[None, :] / max(N, 1) * 2.0 + self.season_phase[:, None])
        level = 1.0 + 0.10 * rng.standard_normal((G, 1))
        noise = 1.0 + 0.06 * rng.standard_normal((G, N))
        demand_mean = np.clip(self.base_demand[:, None] * season * level * noise, 5.0, None)
        demand_cv = rng.uniform(*self.cv_range, size=G)

        demand = np.zeros((G, N))
        for g in range(G):
            k = 1.0 / demand_cv[g] ** 2
            demand[g] = rng.gamma(shape=k, scale=demand_mean[g] / k)

        avail = np.ones((E, H, N), dtype=float)
        for h in range(H):
            for n in range(N):
                down = rng.random(E) < 0.06
                if down.sum() > E - self.routine_avail_floor:
                    idx = np.where(down)[0]
                    rng.shuffle(idx)
                    down[idx[E - self.routine_avail_floor:]] = False
                avail[down, h, n] = 0.0

        # Panne persistante Case C : même stream, même acceptation par floor que PPO.
        if ROUTINE_BURST_ENABLED and self.burst_rng.random() < ROUTINE_BURST_PROB:
            e = int(self.burst_rng.integers(E))
            length = int(self.burst_rng.integers(*ROUTINE_BURST_LEN))
            n0 = int(self.burst_rng.integers(0, max(1, N - length + 1)))
            lo, hi = n0, n0 + length
            window = avail[:, :, lo:hi].copy()
            window[e] = 0.0
            if (window.sum(axis=0) >= self.routine_avail_floor).all():
                avail[:, :, lo:hi] = window

        temp = 0.45 * rng.standard_normal((H, N))
        hum = 1.60 * rng.standard_normal((H, N))
        dev = np.zeros((H, N))

        Rbar = self.Rbar_nominal * (1.0 + 0.05 * rng.standard_normal((H, N)))
        Rbar = np.maximum(Rbar, self.routine_Rbar_floor)
        Wbar = np.tile(self.Wbar_profile[:, None], (1, N))
        Wbar -= (rng.random((H, N)) < 0.10).astype(float)
        Wbar = np.maximum(Wbar, self.routine_workforce_floor[:, None])

        material = np.full((G, N), 1e6)

        sc = Scenario("routine", demand_mean, demand_cv, demand, avail,
                      temp, hum, dev, Rbar, Wbar, material)
        sc.repair_est = build_repair_estimate_case_c(avail, self.burst_rng)
        return sc

    def sample_disruption(self, family):
        if family not in DISRUPTION_FAMILIES:
            raise ValueError(family)
        sc = self.sample_routine()
        sc.kind = family
        E, H, N = self.inst.E, self.inst.H, self.inst.N

        n0 = self.disr_rng.integers(N // 4, max(N // 4 + 1, 3 * N // 4))
        span = 2
        window = slice(int(n0), min(N, int(n0) + span))

        if family == "line_failure":
            k = max(1, int(round(SEV["k_frac"] * E)))
            hit = self.disr_rng.permutation(E)[:k]
            sc.avail[np.ix_(hit, range(H), range(window.start, window.stop))] = 0.0
        elif family == "workforce_shortage":
            sc.Wbar[:, window] = np.maximum(
                1.0, np.floor(SEV["wf_frac"] * self.Wbar_profile))[:, None]
        elif family == "cold_chain":
            sc.dev[:, window] = 1.0
            sc.temp[:, window] += SEV["temp_rise"]

        # PPO recalcule repair_est après toute perturbation.
        sc.repair_est = build_repair_estimate_case_c(sc.avail, self.burst_rng)
        return sc

    def assert_routine(self, sc):
        assert sc.kind == "routine"
        assert np.all(sc.dev == 0.0)
        assert np.all(sc.avail.sum(axis=0) >= self.routine_avail_floor)
        assert np.all(sc.Wbar >= self.routine_workforce_floor[:, None])


# ============================================================================
# CASE C — espace d'actions EXACT dérivé des 54 rotations × 2^4 keep bits PPO
# ============================================================================

def enumerate_ppo_rotations_case_c(inst):
    """Copie de `enumerate_plans` du PPO : (R,E,H), valeurs 0=idle, 1..G=produit."""
    plans, seen = [], set()

    def eligible_machines(g, j):
        return [e for e in range(inst.E) if inst.stage_of[g, e] == j]

    route_choices = [[eligible_machines(g, j) for j in range(inst.Jg[g])]
                     for g in range(inst.G)]
    machine_grids = [list(itertools.product(*route_choices[g]))
                     for g in range(inst.G)]

    for off in itertools.product(range(inst.H), repeat=inst.G):
        for machines in itertools.product(*machine_grids):
            base, ok = np.zeros((inst.E, inst.H), dtype=int), True
            for g in range(inst.G):
                for j in range(inst.Jg[g]):
                    e = machines[g][j]
                    h = (j + off[g]) % inst.H
                    if base[e, h] != 0:
                        ok = False
                        break
                    base[e, h] = g + 1
                if not ok:
                    break
            if not ok:
                continue
            empty = [(e, h) for e in range(inst.E) for h in range(inst.H)
                     if base[e, h] == 0]
            choices = [[0] + [g + 1 for g in range(inst.G) if inst.eligible[g, e]]
                       for e, h in empty]
            for combo in itertools.product(*choices):
                P = base.copy()
                for (e, h), v in zip(empty, combo):
                    P[e, h] = v
                key = P.tobytes()
                if key not in seen:
                    seen.add(key)
                    plans.append(P)
    return np.stack(plans)


def build_ppo_full_plan_case_c(r, keep):
    P = PPO_ROTATIONS[int(r)].copy()
    P[np.asarray(keep) == 0, :] = 0
    return P


def _column_to_binary_assignment(col, inst):
    A = np.zeros((inst.G, inst.E), dtype=int)
    for e, v in enumerate(np.asarray(col, dtype=int)):
        if v > 0:
            A[v - 1, e] = 1
    return A


PPO_ROTATIONS = enumerate_ppo_rotations_case_c(INST)
PPO_N_ROT = len(PPO_ROTATIONS)
PPO_FULL_ACTION_COUNT = PPO_N_ROT * (2 ** INST.E)

# Set des affectations réellement exécutables à UN intervalle.
_reachable_by_h = []
_provenance = {}
for h in range(INST.H):
    cols = set()
    for r in range(PPO_N_ROT):
        for keep in itertools.product([0, 1], repeat=INST.E):
            P = build_ppo_full_plan_case_c(r, keep)
            col = tuple(int(x) for x in P[:, h])
            cols.add(col)
            _provenance.setdefault(col, (r, tuple(int(x) for x in keep), h))
    _reachable_by_h.append(cols)

# Pour cette instance, les 3 intervalles ont exactement le même set accessible.
assert _reachable_by_h[0] == _reachable_by_h[1] == _reachable_by_h[2]
PPO_REACHABLE_COLUMNS = sorted(_reachable_by_h[0])
PLAN_LIBRARY = np.stack(
    [_column_to_binary_assignment(col, INST) for col in PPO_REACHABLE_COLUMNS], axis=0)
PLAN_SIGNATURES = PPO_REACHABLE_COLUMNS
PLAN_PROVENANCE = [_provenance[col] for col in PPO_REACHABLE_COLUMNS]
PLAN_LOOKUP = {tuple(A.astype(int).ravel()): p for p, A in enumerate(PLAN_LIBRARY)}


def plan_id_of(A, tol=1e-6):
    B = (np.asarray(A) > 0.5).astype(int)
    if np.max(np.abs(np.asarray(A) - B)) > tol:
        return None
    return PLAN_LOOKUP.get(tuple(B.ravel()))


def _hash_int_array(a):
    return hashlib.sha256(np.ascontiguousarray(np.asarray(a, dtype=np.int8)).tobytes()).hexdigest()


PPO_REFERENCE_ROTATIONS_HASH = "1407b4b085b9554bd8f90dafd9f88f749a8579b967171bae69782c20c9ef098d"
PPO_REFERENCE_REACHABLE_HASH = "def26c8053da3099635ee9f62093f4689b71cc783b9c9bf0ffc2364c4717505c"
PPO_ACTION_SPACE_MATCH = (
    PPO_N_ROT == 54
    and PPO_FULL_ACTION_COUNT == 864
    and len(PLAN_LIBRARY) == 89
    and _hash_int_array(PPO_ROTATIONS) == PPO_REFERENCE_ROTATIONS_HASH
    and _hash_int_array(np.asarray(PPO_REACHABLE_COLUMNS, dtype=np.int8))
        == PPO_REFERENCE_REACHABLE_HASH
)
if not PPO_ACTION_SPACE_MATCH:
    raise AssertionError("L'espace d'actions Case C ne correspond plus au PPO de référence.")


# ============================================================================
# Empreintes des 24 + 180 scenarios du PPO de reference
# Calculees depuis Final_mild / Final_moderate / Final_severe (ScenarioGenerator,
# EVAL_SEED=98765, STRESS_SEED=424242, familles line_failure / workforce_shortage /
# cold_chain, 60 chacune). Les 24 empreintes routine ne dependent pas de la severite.
# La liste severe reproduit exactement la reference du notebook CPLEX severe d'origine.

_ROUTINE_REF = [
    "95d7eed1ceeec2ff076e479ce3a375b6e95b57f28b1269135d57f088c614d9e3",
    "8e2e2c4854c0fd33b69f3938f8b11b262e686f70d374a2b7ffdd41732f8564f8",
    "fe123e42455b9b34f1f41e89e338d4ba287dd17887f4395a2a18a28ae90d0e72",
    "5eee669001856f67e3485c9e6af5d24d7462d3971f227d9b7002f1fc25fd8b0f",
    "549a5721e24ad751cb33ef1dcfb83938ec044312887e3150026cc107ce78a7af",
    "26991f857b626bce7d5a6833600b1fd7d85a3a7462245a7e6bdfac763903c184",
    "b544618c5380bc59b38b31d28cf62de13670ad42e9bef3115497d1b66f098130",
    "15e35dee8ba25e0e51a3d16d7edadb9916e024294fc6369e1c5a1536e08c69be",
    "93eed7d27a59dbbfd1ace404de9658011e96a642f4b3fbfde139bcbc41870f54",
    "ec156eca69f04129bb44198df1fdc1a9b0d98b40f8f6ad932e764745d41a736c",
    "4fa1580c6974c9081e6de0fbcc7a0c7fc1372d33048067f002159f83eba423f8",
    "22fb06a5841eabc6de51ead9464d4a1091e6a096beb943c7274346e2263d168a",
    "5e99c836a54dc5e70aa58a6990997e219fb7476df4a28fa3479e1cddb82c796b",
    "f48dac6784f9d00fa6b9ba327aeed4a77433328bf7007ea014bbda43ba94607a",
    "95195855049c8e8eb934c35433be2d41fe22af17186f4d306fc29a99815d3473",
    "62aee2f33356f2199b513ba66d89826241f6dcb0be11a4c033c033b619cdb2f6",
    "538a100013c5c4e424e59328d6ef16ee4b5d135c4b4bc063cb53869494d9ab4a",
    "5ab8d4dd808fa250ffb629b3865412ab00983a8fcc7dc7a265b1b1207cd7351f",
    "1104ca964997287d764a730dccd8efeed2b1b010be64fc130c42fcda3ad6a66b",
    "e33bdc4793a5023490aeb80c32eb78faa89a5503cc0bf7b3f8fb065adb8fdf32",
    "c39bde5756276f67bc4199fb5711bcdf6c707aee56901043dd7bd2edd1a8c558",
    "0441dab045a16631ce75947929a5cd9c2d448fecfdf83165b1d12a5cf4309219",
    "70cf265247193995f56dab08b5c85cf53b0b45cb04cc9047d8229075313ea436",
    "a077526ffe0d8c1f3db4e6ee185384f59e61d03b5fd5ecb22cb0f4ae78be8ab9"
]

_DISRUPTION_REF = {
  "mild": [
    "1be403e5f5b6bdc47c12ea0f831ef1fbe9f1fb61532a267e030fbadacbdf8b16",
    "656f07f120a651a716408b0b4ae77b2bb364c90a44c29973fb96010508d76f46",
    "93549c346bbcff93ad2a1c3ecfa191e5aee012ae8d69b462cc1342df6687bfd3",
    "209bd7a5663b5fdf587f65f86019939060033a426be2281052384b5f845a115e",
    "713d6ab41d3451d380ad242344f14774f74dc3b663500adcef40ca569d681959",
    "66888e810f4fa5abb0389ed82812962c49a03b0974d58b6dbafc821056869d85",
    "b0f6a5f768b11afa0875be81ece395706ba06030170af78a68cb9447699eeb69",
    "b2052bf723cab74bdcf2c0e37f4822288cfc2a3532d519a3b6b6c0a168af4b72",
    "66361840f873422dcc11d4c85effdb9603b2caacfe0099463a78bcf4dd5daeb6",
    "5109ff543a53591d10dad46a9019179425fd2d19887ad9031cedce5586df4283",
    "4648c0780520c15d553a719a5098605e2d2f883b6d48392ce0c68232804bbdc1",
    "3c29bb66b40e74742510ef8d13523626f47a421097f5a22d2472ae7480632c05",
    "3abdc6692103847debbeea6645f7b1721f9661de28ba3280f5b8af89b4e88e1f",
    "0cdcff2602ae420a6875d6dde07a20867c294b6b11ea0f5e89a0a878215b65f9",
    "5df687181b467b4e10ecfe80dae9566f10a2a5e48494bea85e975b161a54cc5b",
    "33985161f0eb19b1d1e09ec442bda871d392e5001f79f2ae2d839680a259856c",
    "822b87c7035e0cf841fc4d3ef28ea03e3396957d64242224d5cf353568a85c9e",
    "b612a7925853af1c6361bd11463a29c1091e8a319c9ed541cfe883cc0a5500bc",
    "a5ddd7d80eb5e442dd7ae9d8e83e38c58ec07037db4e8ae2156e7b6e5cd6737a",
    "87ccb436ed158e415edb3edaadd096d47c1aa36f7b65c976b9e4f0d0f2883245",
    "a9c85038bb8c3098947f8884aed4407d7313485d175669155fca7944d3e0c443",
    "bff7cd745f7937fcaf92ccc30e9cf358dc638380b0bd77a0a1408b9a34f46c4d",
    "59169860074b3f5b12af1d9ae1547f50043c0d153c68ea4d091656009a4049a8",
    "feecae6cf69dd7de90cfcaa70365d60b126c8659545deffa2d413af3addf632d",
    "6a1709c64d53e996302c987bbb9e44f3dd5a731fe6f428d247e7ddf23973fe74",
    "483f37d850e72ef6c6801d9b7fc37238df90d4cc2349789a08acb3042ddc676c",
    "f1985c3eb491479ec3e9d6658cb0fe36e9f3ee80decd0f90867b05b927b9b789",
    "d315962bcafc3d544ee1333eb59233d0ab9620af9e1caecf114761cfa26bd8ca",
    "d836e3d1f37af18e7334e11e2ded564bafd28f5fe3baa2cb152de87c0a7cde29",
    "b778aea532db533f8023c12cafffa3db658944036350ef3e20b0f9fd82e59eb3",
    "008e9a0c343690ffcb91c74e2bdc25f4bd0d925590d3be3acbaac8d394c6a754",
    "dd463bd9152ef4dbb52179dfe8fd405a8e0e60ccc4929b40119eb57d30d3fdb5",
    "0c0a345fb5053ab889d46a06b37e5d08d3bd4b82decd2a8fe997b2c8864ee698",
    "40d15cb104bfd6b2965056457dac8a16efcb3a7ff1b60b4ac6afa038a0f8d848",
    "ef8c5c618abcb6eb89c159dc8ce0ea8abb25d159321f471836ed962798436a7a",
    "b0a87a6fad33a591e8253dc11efa9da9abe06b87a166cc8f5f298a32f7ed9207",
    "56388ebd5908018a4630029b155fc3f2fd208a60f90a1e704a61dec2c68d8701",
    "1c2c9ce86fc4f2d006760193fa117f00b326c2e6dce78a1e1228842da50f79eb",
    "0dff6848eb007e0b6b8a65716662aaac001f2a7c16c75c14f3ccd2b70862e90f",
    "810bc3b2fd64aeab2d77f3ed4244019fdead7e7d999fe8027fa66c21959ebd81",
    "0dc22ec2b8d8bd993e1f661a81f2b24640f3fac3cb088079a3799571293e5f3b",
    "a03c85d244f3894615bfffcce6c94e22b5a26c46a8597438d70a8e94a4015deb",
    "2fa387e555c8e6c42656617b9168cf1acaa505e381b0b3aa8fbe38e38339bbd2",
    "890c6450f6b827507c198d7b2ab93e2b5575d1c53ed394840bf688708d1e0d6a",
    "70d56e334304138c64bc6961cddec5984bf0860f90f1285239e00d1014fd94ae",
    "e8fe848e5ea6efacc9712676844b3f54d68a290c632a6d2bb6f73fa7ad5d7e86",
    "71e3d7e7a8b8335afb78d888f4f8c0b4390fdbe2d2c89c46afd4cedb01812bb7",
    "fb7b9c3903b4d00af3d1bd054b23c2f82b4b0021e4d85b671bdf20731c378d3b",
    "ad53c9ad5e7c603562dcf7b606a9288b2f1e9937298652765d342f58be119655",
    "287623b693cf67b7fd9c351312feeffece17cdbc66ab74b4877fb63e73ad78d7",
    "9a6d877c1b66cc1ca7bcd319118e0b61774981cb809bbee6a8e48d7429e51de6",
    "94d395a12850bf3161f8b1c91e05b55678150ff5c33a7b34d74b63551a62c8c6",
    "6b3c84ab31d8444f3e38408ee55b3d994fa448743af7dafa49ce00dd9f444c06",
    "f39a32842c876c52712c6ab2cadd492706ca8cefe18856e62d366db89c9132ce",
    "08a453fd4827a58804c16cc539231856b81325ca0f29c77b63432b8ba2e01117",
    "32e6e2d37ccfdc16e19da252762f6b3927d54d6c0bc828424e0fdb2e472895f6",
    "4923b21a645eb3fa9360d47ade5085dcdb85649b454b5ab68dee16202524edfd",
    "58a9cdf27181efc8d1995455815c18a7e1691fc12a5f57504f8bff763c41bf6c",
    "f19984c3e21e76b5e2b434c92155b0b58b173ad8d554450f574bdc9394826638",
    "e5e382c34107eb37aab387c7debb9fc920e89199f7ae2945a93203fa4bb3fe16",
    "165e93ef59e3bf794ae86f3d36d7eafd7bb54b62a157aad768493e51f58af092",
    "99b64e5864f9670e415e6376a76a6e8d8c8c42fddc12bf8d7c8d1597591c964a",
    "f51773b15b0bbe02bd4a8502e9112b8919aeac60cb28c69005d54d19f1264e12",
    "7f45d34b2a9805f98e03d8541b7427ca677214f4536fa263bc9c45bc6ce864b1",
    "dab2cf28e0547c2530e2953f11a45ce41e4fc6293f3995432ed3bf2dd9a525c6",
    "8ec59761c76ae35772493a77fb6cb3d684e7c9088dfbe238ed977045d7396090",
    "763939881b40331ca6978d1bae4eef6c7c3674348c8dad942d3514c8e0ed1d7d",
    "5c62462599641a5d41bfcf5899ce03ee063758225ab4e81ab0c065c688dc8f5b",
    "a51fea98ec8d7f1bf070b8a7aca03c435b5bc7d6cdb8953e8e55ad04b53e9730",
    "a7dea17221d58e7a0dd57160662e9a309f770383f3e8e64f58caed1da3050b36",
    "2e20fc61d58df82a842b24d9adc3369753ed93dbcd737297620b6f3de2d8962a",
    "5331981d5eea520290b85a4c26bc815075a5f599b69d8078471a8e0b9c31655a",
    "0e96bd14070f5da65003b5db90cc547ae4b87ae76318ee9299773d15ea30cd6a",
    "1f8822d0dd6d0132ea7ffbbcf9bfcfe6806568ee32bc5ca1eefd8c8d55f439c3",
    "55c8ac73a4403bdfc5805fe19a73eafe2a8a0d45bf2be2219207cb1243593f85",
    "60a13892df63d752c719c4032ccbc427b0c0721bf751d945eb4633ea1eb7c2b4",
    "e42352c0e9378d4586fff8d4ba476e5e0f0d1aca2b2dc5a6cec659008e0678fc",
    "a67538dc9e72a0adf382844d0239385afd9ccff70cf1cdc3e7a35d3e409200b9",
    "fa8f1574e07a822cc603471a4039f11e66686019f644bf6a638980c0747e66bb",
    "ed6dea03c7582f9a439c77df2700a252da320c182d6e84f53ce670eb68aa5440",
    "9831dda57c7a03ef8f687dec462fd46f9044126bf3a9f3f0ae46894494cfc54b",
    "c123ec6c3250a398e87d31d42af22eb40e73c41e9c553688b1f83e988dac8bae",
    "a6162ce565c7e8cf02c790f0f60b2c183fbdc2d6715742c74ba94c77a33230a3",
    "5557e2056412e73f1ad0ff4074c4983044765fe631ccd82a9be332c89e36f2f7",
    "99603faf0a41e5eade592839bd367a38dab3ffdcc080e8e587033e5f3a3f8ecf",
    "4bd44f91d09400c0cd36b35e54d5174950d553883fa81583cafdd6cb55803972",
    "cfd1d18554e6ceba32e2274efa5258c7376ca7c918b2b03182c8c784b69ae7b3",
    "c49b4e6b8957c79716ec8d73fc47475a8fe8779a83c298d2209b60cab760bb7b",
    "6f3ad6f011dbca4b9e312247d073892a96130b579f327fd99ef52009c1cde689",
    "0588e48d1568934b825fdd245a8b4aeb801cdc9185eb057f359bbe8527f5a5cb",
    "e67a7e2b4ed7b8d8cc8574cf046c3c5235fbd8b1dd76de4b68c9501945395f88",
    "ef3cd8e7f9079eb49154b53d0aa2416ead3d20530bbf57b69b00e64390bc790a",
    "e6d2f9a29ea3db5151a0f54a7f29aeffe314419cb23ac9dac4e240b51d22e4f1",
    "4bd6bf8ff365b2393e23e116803be3ff438d90d07434c64e8fa475bbb1c5ae46",
    "8f46e0380bfb0b1d35b52303e402af8d76f5a0b39fd13317ded1e7e4213bd63c",
    "91f883d21287fa977be3d556f2a476bb1c1e62b20fc6559e14c3d3c8bfe19879",
    "0816280a3c426497ba0c2115a4a905641718e0f9a13b7f7cc55daf186d92b3d2",
    "914c49d65403da71d1ff4a5a47e4d06ff462114efd541d9ae995e7c455190a28",
    "672e5179f3246cb648a956470a3816d2e539095c70382559b00dcf638a3f90fb",
    "1ff4749d222d8fc3be7965349213ffae308a82aadb8941354334b9b74ddf71b5",
    "67072a4dcb085c9adf19ad0e2be88d5b6b3c12b24531a434506b386b8f3fceb8",
    "f9df57b05283d43da21017f171c73e2642e39df47a666f06e5379919b38d821b",
    "21337c3b2da49e01aa4466937a0c8045f7b81823fb4b56554d065b1c6c0e5dc6",
    "93baeef6e86371298f3cd78a7336ef7fcb8af4066466b02d3882f67ea1b3f228",
    "1a24236e2fb44fb5554409c24b58b7ac818421c028c746fc6e41d50156fd3714",
    "29646990d73389e5653558addb5d6e9ff0ca657eadc36cbf51a803f584c1bcd4",
    "cdd26c1f1af02df2bc9446b757967af06108538f6f65001c9cf0ce8d06981383",
    "74339bdfcba437536931dfb7e789fdcdd02d7b92127210c48a778735312629ba",
    "4f3a311c9ce01b7fab2d6ce8d4900288b55c5ceaceb3633405459564cc9e7d28",
    "89dd0fdf526f90574f89b66aa9f689c7c535f50254638a8475576572a6e97e83",
    "a7335854f91d1dcdae6775b3a2d4ab4b5092611962a506bc31e829cb44d4b6ff",
    "35f95cbf441301765065c4a636e6469bc7c4d8074c57260bb770ac94d311b95f",
    "a10533712b6311e7c79c301ca71eaf3c81dae282894f72ec9125ca2885219027",
    "557dfa3343883f35012cdeac3a1bc9ffe2ee34ebb731b5aa5f917b3b5cdaa954",
    "ad7b5743f888c20ce9173fbf54c223d97f4ebb92b841aae4d06087e16509c0e9",
    "2f7d60f962e7090b158aa4fee72f94b0ee37d0661d19fbd2a763eed6410ea426",
    "fe6b0fb6b593998853cbff93ea6ebbab6b62944b84fdb770079c3f005809c0cb",
    "6ee58bcc4b2d1643a39dfdd923f9e3e0b6e9f1bb5225911fa53366384203d4fb",
    "5d4eeec730d798baf5d557038a08510433f18172590e6da200b465a35ad42553",
    "0a40c0a163cb0b3219203eb567849b6fe4c03be8e5181f8051f3f6f0a108a294",
    "126fbce73e23c6869836f6f42f7cbc5026d77f5035b414100e134108def92955",
    "e81fee0b01eb3e7143395733e987afb6236a04274fc9d591cd5cf374777859f1",
    "89eff9b69a5d8d030be947277a24e44ab5fd6971ea2728db36e1bd51ad91155d",
    "eb4138b7eff408609bb160813df42c572e5ebb6efd2da2ad7a3876cf32dc5284",
    "3c861675839a27806c0b1382b3d6650613ee3dd379b1b3d23633c15e736026dd",
    "33be0116e768aa66ec997b49f0670292022006cf5e0934086ff1501437a819c4",
    "adfa1a990d355fe8f4ad1fde83a904eca66daf3be44c277d4cb7a731a76fef93",
    "fad630b9d71b7baaac0a1f76c3f025375b2b8c2b34d443882d374b0a55ccf956",
    "f2a8717b9512ea85dd13b80430275597aa335ef9b600502747e74437bc68257b",
    "152e17dcb3e106bb772040e0ae934a7f263902e3641541f189beb20d5a42246e",
    "8b655cc7af015df0be66e938106d1689323621e7a7981b855c962ddd5f5e0fb7",
    "400c8d1601f9fdad801a355956763c3b749849903c7e7e64cc79e5d80b5d3efe",
    "7d8e2eca164893818c25f54c6499884f4dcb0ed8ec74c342889e025aa68f3558",
    "b38c43d2def9d3a75e6458f0868943c383c526400c531c01b6055e7dd6ddfa84",
    "be5f374be82434e7b085326fd126ae1457ebb5a1a0011f9a4561ab3f378680dd",
    "b41e4daf0eea554f0f07a1ab563322c4fe6bba1b247ae3f5b355671eed17662e",
    "0c4f5b2de5e489f197d1ad99ea322cdc89b56b76e2d814ba16c7c1cda2d0a569",
    "10f03e90bee96354eaf8d90442e1340b87acfcc18b145b7b616b3fc66655be44",
    "7007e9f1e8852014192338efda699c12d9b53a422523d27bed9e6744fc40f92f",
    "5f1c003535394d609837c963c4aa332edc2fbc2f8b7e05183ec1cab81f03fb2b",
    "85921e7fbb7b4f4dbb508aef47bd11ec65fe0b59a2e57248f9ca92f2ce377037",
    "baa717fcdfe83cc2e6008cd1fceb8285ed591301e9006f9d559265145362ab62",
    "82804e7bd8b385e5fe9227c4713fe474890f1ad0ed4fda02273369a62f424805",
    "48bfa8e0d9927d94010d7beef106cd98e816857ac5854bd9e1198c9bc9aaa3ea",
    "822321397c366f3a0f6e47005bfce8858e1afd1d427fc495c168fea316987690",
    "7d74a652a258e65fca6c95443ab2aea2ad6278cbd9d892b1e8e4ea7236bad6ea",
    "8dc4c1b13139f4bd99779ebd51025a4898abea32199265365b8eda7830cb1ac1",
    "a5dff86c17d473598c64990d24a502cc6a1976d8c5c27a54d4c55d975994ade1",
    "d2b2457b3010eb64f9b2d5d72b2a2bbfd1fd752deaf576a31992ad7cf00a435a",
    "9653793094619680fde08ba3b764c0d26350e79d71a184833e56411a4c9c6143",
    "21773b227a1d36d3d5e5a145a4d7597abbc90095d6d54959f2f30db67d484116",
    "c848e30bff5f65ce1c6f28d9e1d5772eb89c96b5414fef39da333d5a2ef53459",
    "1ff6df90f911bf2a83fe50b43073c7f353cc40da38f489449b513a7ae0c16fab",
    "de2bcb387701ecf27536ba746cfeea23cf84e9b3903fede8d1d5c941ba5ea1c5",
    "b1f61d76a5826e1ccd8e0a2a635f24f51dd724d708e9fd597d27f173904e90e2",
    "01d54cec6c0ca9d8271d75673d78240033a1c8fb3cfe53152e3c62454f444cbb",
    "d992a8b78f0086f842001f0005b969aaf79c4bd266d4c26c5194afa641b5ff0f",
    "62f6a65e6989db9ea19f40f25cf616d00f5856279f28282245f231ac9bc28245",
    "a2d0c8f63202be6bb21307d3ef255a12ebb117a7e44361e1ad741d0743de3739",
    "10692a601ec8c38373461896987ad997b908f233ec27344e0ecc521f2474de11",
    "45e33cb200a565231546fb32fc269aecbef43784dde95c6f7808a48bcd3c6587",
    "9e392f878bfb9afd4726be0686f6fa40a80315ad792741e9f1d8669927c69db8",
    "5ff69f4ea463afcabae95254a498980d82d5d9ff7d4772194a3903a9c87c5930",
    "0ff8d0d61b07b6f9e47bbe8ccbf87a17d98a8f0d2abf4cef30490e823602a8a2",
    "37b8ec8984c44f1ad27a7abe988ce69849db20aa52ad5e5bd275029bf42c6cdf",
    "775fb14beb3e7613bee768774344e8b7587733a7e038d80346b469ccfea55fa2",
    "e3dc9e31c568db9c40311ce625386b4b83e241ebbb66fa668b06847ff641082c",
    "4af56a04aedf73afa0c97ed1fa8df6cb35527db909117400d757f83fb09a6145",
    "9eaa45177ebec3e6e2cf2e6036b7a78657f9c49119674ccee06c624bb647142a",
    "1beb42119849afadd829bf8fa964827e5b2d196116b682cd32dc7020411b5154",
    "deda976a1b478963f277ff70c315abd8127240878eaff4c2756d8384b1f8875d",
    "0f7266946bd5750d1cfd99439f4cd16da6c676e31db2084d82c80f79fb2b536e",
    "39ff4f86ce5c8ebfbbb6e0a6991edb977d69004270595fa41cb69678e2aa7a52",
    "3005cc346b0a9c284ee7122546f1f39729d895b88a856e398c9a8091b7aaa880",
    "29787baf4f52fbcf127b25b1f8434a36370de84c641662f38645271804f60231",
    "0d0c5685b00efbf3af9fbf6642a68b69cc85172ff426a4729735ebbf8ca7fc95",
    "f907b395bf9b19b51b85517a70c2d628eea56983ece75572be19dbd9e0aab0aa",
    "22be7d7da1104360183614ac5c8bd4e76476b8825dda51338aa9e9123580f854",
    "e89c9fa4039951c157d5418d3e1fae7077c577361a04f7299e4503674d488ec0",
    "58122ffb23cf527934b918556ef25f4bb66d938ebdd8afb1a48be4e4b45bbd1f"
  ],
  "moderate": [
    "c030fb51b836e4e1c5980dfbad485207a2ede3667f212c11e6f5b7868baac1ea",
    "f33b0b39a236c5e36fe48de475a280006a23622e579856f96f57b63ef5b5437e",
    "f2f5b6623589e33dee46a6db7244b280579aa7e8566ed2905af26563003dd194",
    "d252bdf5f68d7220f83410ff105d38506cf22112bbeb6c5a6b8613b0a1a2082c",
    "ff0e61d1dc69f64abe671d7f385dafbc4017b6ef65212a99397aa8d0a8cbd89e",
    "c3f9a91641d010b303e8ddd98d819622f26971d2916bd8b136e1adca74357e19",
    "6017420cd17cd684978d708f2a8a170bc3e11d126da87ce6d576aaea6d058302",
    "e0d26d89b6111cbbcf8e111543addcddd550675725d9260a375d195cc7d85689",
    "bf7e231f91ee3fcc496fdc9611b4e227fade2adf04dc91c2eceab88d2a395f39",
    "968ffc6f85e6d386db95c56ddaae1964fdfe3d83ac7787a2998968389ff6654d",
    "b0a9dd68bcc5e261835d808f5cd61e70cf4ebfe492807b1ad07d3856f01ad8b1",
    "b189511a3f3072021d70c91c355f9d60f6486cc1819bb93ba6b01b3a00772c04",
    "413999974958e89fb60ffe696b3ad0a625ab9982d0b095cc5e5a19a998bf6878",
    "b8f52f3fb9406ae08ef26d3093e4b0a9a2b2cda8572ae8e92b27059bcaea4dfd",
    "cdf59cb16e431979a21add5ddbd216cfd9a8257325f5e5b36c3dd2338b8e684e",
    "f04c3671b730335444b4569a4b56f9c2f9aedc440c94e6ec534faf06a48131ce",
    "8e34584d3252506114e5a30392d69a05a08985ec69ed0fd8d7de846c249e9680",
    "d4087b706c1c480a555c514120c5851916b3c5cd9d16e43033b550169fdd83c6",
    "e4005bf1aed38c7844d95616b798b6b9eb100a07b446f65fd2a027384c540689",
    "7a01f901f0141ad4908e2cb265258a3131a87ff703f2eae6c043614689f4369b",
    "f55e9d5770b8b7d50b2294147039316b516bf5bf4a3ac01494471dd417d31991",
    "963d323aebc7d23b146798ee0e6645f570bb189da6ebc19f921a46a471f480c0",
    "11b9d0db47d8950f2f8e80108af07fa50eb458fdaf20ab8394b19c35f685f775",
    "a818a702059d11b4977c29323434e7208fe90df5776198510da2b5cae2ce8c51",
    "5693a6686267e92e2314a55a3364bfcdb4e592e902fd1f27167da8eb2fb69f29",
    "b6a459703fec04469444b30e31f312b72868bb0b043fe45515a8c75ec93357f9",
    "bc03512a67946d8fb1ebe6dd47b13e4ae9c218d63d08dce56d34cd4f253a9a1b",
    "4d4e60a5117f876bbb63cd169453850cc152c1ef578113f0a3d2e9b8fdc4a38e",
    "cf8aa75358a2a9312a488eed64df2156e4757e3a95cfd56019b9e3e68e863fba",
    "0aa926daf95fa7dbc26995af563d2d798edf8df260bbeac38595ff2ef997d524",
    "d43d91c0761f27cb3da061338a9ffc31d4ce6104621b586aa16bf8bcd0d557ae",
    "da22f120861312f5262beb9fd07862018f5d46366295d013264fb12b9dff071e",
    "947fb161f06fe74aef2a7bcf769e3e970363ff8ca6578f46b45533d8a155c35d",
    "533bc162c56d7a225ac767b305a9a6b65c72b038f41714c151360bc5b87987d2",
    "40d813fb4d7b3d2a8e340db5bf337ff60c4276aa332538784ae8bb119ffeb229",
    "f08f0c15e86c45e55773982f895d5c2afebdb445c8ca7509f2465679443fe83d",
    "01cbf1da21fb021c047691cb2c9769ab8900ed7ed93ef8a69b08dcfe4a758d56",
    "07ce4af55e9ed8fc75a005f7379e78bfe5dd202b8d33ccc84724443e53982a58",
    "a8b3340f7a7911e80eb276a6f7fdb668476806e845db91ca8debeb311da8a174",
    "79a0d931d3ebbb83479eef670d0a47a2be09f4d1ead960480827cda773bc28fb",
    "2148fff5a2041864f46df651ceb8e7112a85acfdd817055cfd244108f5c56d14",
    "aadd12650d5d32f45f4688b40020f62e9b8279929e25f17c158ae61256cd4233",
    "3ee1db20b5c931bec30eb0a21ace8fb9db35c79330530a91902aaef8cc3df5b8",
    "d7386afcb1db65fdc5b384ccc4a95c40c26a23e0b7e0d209b32866eafc5122e2",
    "bb938c7d6d153da3305a4bc8d5cf17fd07d018d08893b9f41e9cefa26ca18772",
    "f44e95b35d27e3ee98b81c38409a324956c65aa02cb96b3495b149f21a6d267f",
    "6f1f70bbc54625dc24dac80a3594a51d0c59fa41e0bebcf2f3fb0bcb06c640e0",
    "57d466bb4a5f43290762aebd843b38504a628c4672f4a8605275d23f430c2fcb",
    "55371a0c64abcb8ac6bd70c6fa9c83f4a7cbaa4a5954efef411473ee6d9646c8",
    "05e0ecc408e7a3b88dca25b74fe5b38c23c387c03a728be01b5e6dac00a66e2e",
    "7c5a18e993ba5b419a2fc7ef77ebac1a4682c1fbefcdf265a1ec15d08299b087",
    "f8cea0a9252c8e491d29dbdf04559231dbce5b5f4af7784323a28d91592d8055",
    "b2a8e16d78e5d7088204793ed97a2f02e34b2fc628261dd3a6fed1b3e4b0b9d3",
    "9949dbd5b2e87f037d1141e72eebb9b44cf6a363524aabc77650a013387cd0cd",
    "db27411a739f5d6dc5f5d3473b92d5eceb2b25bb4102562b169a9c2c5340333e",
    "e99d76abfaedcfbc1fd6bd39f91a60c4509c396e553378ce06ba2a06c56e1a9f",
    "7be9ecef6cceb539b30c78d7cd591749bf9291248987edb2fb7f9649b3d10b77",
    "ec4d6cfb5168fe7427a8fed87376be68001d47b17927b53f7af9e7adb25380a7",
    "e038198f1f2d509fc20dcb3f4909487de800bd36a13f56dc8c89a77f2407a01d",
    "764e4d2d6580568649141977d8a0de91c36d6a46ff0fa7c7be2c3f3b1b4e4de4",
    "77c2d99291ce1755cb7e0b645e83a6781f39c4d9a7b1e9fb75cd9c3be8c64ed1",
    "40581125b627f1eda105169e36dc1faef570b6efcf0b0d41253ae4db13116948",
    "d3a7e715fd30648466fed95e76d68869d993017cab7e1f4e46c822773a3fcf34",
    "27255f10ad11ea2f1cf9f69b1b8bf8f0e898b3f80b0bfb5a0848e123fd07737b",
    "084e58e14f2cba7538c54f9f47b7cb74f648490ddb4f75de10a172ec75f91131",
    "035d06998d900e2437663d8bc07b70b84f359f4a2b26f3bf237990933eeedb28",
    "b759a70ad70e7c17161e469aebfebea4d8332432a4ca84dc1f3ea5b5ccc4a388",
    "571bfcfe42985ac5cbd803d4d95ad650cd25b60fdd1f9f14f4afa9636be6fbe2",
    "a7cfb2a849f6e4c0f24306dda61da088687ea5c79e0ed6d6108880b04adaf343",
    "e7d327befd065f6f9bb1231d817a286fec45bc24e147c26070299743a4641d4c",
    "9a16385977b2190d33dfbbab897e9e6129b5b03569321bf78c3a03b744814a47",
    "4cc84dc6dfda0edd28144908128a819abb737142787756ad97b1d76f94e54460",
    "f95b75991cb96d4d63a1de925354db50d969ad13fb16788f6b756300eccaac78",
    "f9136eda4ff55594d96d4ce169f9a4dbe9e27e9f8f8771e0b518983bb8d78f2b",
    "0a4cfcbebb876476920d5baa46004ceb5280f4389ea514b519c76f9f89e2e082",
    "02090c16f5fb527e40bbd918849d773aadc7bbead0b22740a678470239019afa",
    "880b2df25f51d7b9a6a354bf9cece07ceafb529fc4458285a2de0ad432c736c6",
    "78c266718451e877369a2a94a1588e86d90cbfc193ef34b842c2f3dc69308915",
    "84546a08814f779e3c2f5daab59dcff3991e1d76514276abab2660b29dfd6816",
    "336c74ae8464449d5b448cbd1e6ed47a82ef5ded7202ba5a6fe704f0fee48230",
    "ac407eb7bff992f5119741f3599445574806631eb425f033f7b512dacba8d7f3",
    "ded7fffee62d4b69bb18fed0534f28fd4e7306ac6d4c774dacfabf64d46e9d8a",
    "0bbbf8d43973dc7315407b792584f641befa3e82d8c7ddeec8027e23a858f9ba",
    "3600faf871587178175823a30390a5fac5977996cecc60797d974f90341edf75",
    "ce7195cfcf4764b3fedecb1f89b9222b700a48911518491e72119f21ebc6f099",
    "f2baf8842b5f3647c0126cce6014715e376d9c3502346807e2c92140419fbe1f",
    "1a221515ae5276aea79d1d2483b3da531b100db9f38d4145384e06049e07ff75",
    "46f497efb4910ca5779e27c9b20d598bb7caa1755e8ae4e719c1f728702ee14f",
    "2a336cf4542532568894a7d51efbe4e2b37722da782ce5f39e4c403b63d8ee0d",
    "d1ac7b014bf3ab8b2e168223c7de41873f6d7a053f207115dee85c85bc13d003",
    "9140bceaf972df0ef9ccc3d0e9e13f9b0eaa52cfa23058e0ab44a177804a3d97",
    "92dd2a04160b04bdbe73191e19eda2c8f55e8d9071959c4771d380c07bce0ddc",
    "00ffd293e7ac99cc7bca1748958cc691bb8865ad6ba15dbeb87f69e6cc6a14ac",
    "86cdff70ee95b71bc238e11025c740c68a2a58c82c4dfb6e7c26eae2baa96c17",
    "860bc78e576381a63926cf8b48ff472fb5566495775f386f9db06e317968b444",
    "2d4342fe7f243107ea73926915900921dc8c53b13968c5ebd1fb4e4e45969df9",
    "6829e8885802fbc7db78675771f41ade43e8d9284e522f778834fc13a8970b4d",
    "cd5f13357b239b7525f53028d27a60a0b9324ac468cc20df828b7549e1380282",
    "f757ccd73536674a2de33431556457a3cf5ad76469c192fb2a6c2e5aa6847730",
    "e5d8dc46c80c574c6509f4a4f2353a66ccb09a371b7b6b929dfc6e8dbfd8d82d",
    "9c0b02adb9dd87a6ddcfc364f52b3a9eab2c2a683313009c15a9a44f26319791",
    "2c09233f32650599e53986d4b7b2dcd8bb699bb78b952128e39283e0be70aee5",
    "07082306441443965dd16c5d6b6aaa59e97cae928449cac221767de7dbde925a",
    "4157f75594854a23d484d3ee3f8a046212beb80db628c5cb847202213d847d46",
    "e4ba1b463a298d7af45d9971478c492adb6c53fd39771cbb40184448f3cf82b2",
    "e666e289ef53213e49bac5bf5768b20764db93ffda7e5dac3568beb9ddef60bf",
    "a13fdef806a71dea8b0d8b4c547f748e930d0c3825ed907fd1c6732df59d5538",
    "32436cd648aea2cb1f77e8d239eeeea6a20557a5e99a85e20afcad0c35f3e446",
    "d92acc1ee081d466ea078bf38869d3b6634ec8d72b13741ac3677e4534a92abc",
    "299c907664787822b506454ac47979898a955dabc4df46dda8aeba1032c9e38b",
    "332f6c64a916f9dcc78f989e802ed61f410acc283065edc667daf225f92379d8",
    "36b4397964a577e8abaf33184dc22a26a0b84541de75fa734f29ea43535032df",
    "98f456d2c4711c77a8a84f6a4f13850a0e0fa1ab963aeb409f0b3a7e782219ba",
    "bfc6a756c0a25f7d4aba42b724ae9b79728791f64b488dbf81fd914c82b8241b",
    "ed59d915a9ddddc2432a994bb1ce79e6928b22095df28bd5df725c9a4abbd3df",
    "6d0190d7b1453f9434deccbbd80f5236fd56af48894923b0db5d045fe5a6726e",
    "04941d0167f775efdfb4ab81dc91abaec5896d1f6a6b659b4e003260f349f449",
    "0ea3ef476b554d28009d7347f8fbf782626ce7a61c4dd2eb470b16786b223953",
    "c983b4e971993795961a68bba9ff26d14475c09a5c0a89695b686da14d4eab9d",
    "09dd637479f44c6c437def3dac4d20c1414262164c2f43f6bd1da509e5222782",
    "b8b600410b941a93c05824886b47bbe1b9af987cf4dbc07d565f0b3689fb4e22",
    "39e5879df4a8010f28d8adf1beb6f9f0d4ce3794bb48772e8c70cfa9e542326e",
    "45a8a6a1c02817480f7ff7afdfc39cdc02ac6b3061e2f5f3fe6a16cc9500b862",
    "be2eb96a5e8b1c99c9960e0a4c218c724b60820f6e82d2a27ef799f10f695e28",
    "1a94777e4f66c2bcc594534f95a6d623eddf861c8ab5d3c662305d6677ca89f3",
    "3ccba4ec66cca72c5fd0ebeef11e6f114b37a437ff2d5bd867ff5209498a6ac3",
    "16e612b8de3b869af2a59784ff5a0f5e1805a4ad046e7b6ead0476d3ffac2f2d",
    "eed0ae37be7d4025abb06bbd6a6cd41520fdc4cdc02b8ccfdb86034193bd3a88",
    "3dc9828479094f8ea2bb40b9498b441934497b22be4733789984e44354a683ff",
    "a9e771ed46176d6e57e9eccb2e6fb6a95142d239b02d1154c512aed63381a807",
    "8a4f6994b27565c9bed4024001cb2ef860166c67ae4e03d9ecebfcbfe1262d2a",
    "50e702558bae5308472e73a509b3c919bf3a61f9ce9873089bda10a7ea00793e",
    "5fb1dd7e2ef1f63ce7a7e0d811d43032356ef4b5896db40ddeffd51e46590a19",
    "637790a4fda6ed51de528428db87714ad78180237af341a322a6a50b63a33277",
    "662c72c8a913f75de5be6dba33a332cf942af5ff0db8575db8d635e631540d41",
    "aecaa550aee1b6593f23faedf96a2f51f0c6df862183ba40cfa50585ce753f03",
    "bb9c876a07e41b4683c27ce632a7b5883eac681f73758f014505e555abb5e478",
    "4e5bf4abf1fdfe08c3d920b16ec2362584aaeb237f1b40987f84121b263a3f79",
    "f7461efcea3439db7201a57c56dc077b85f7b6b0ba45583575830941bfd91099",
    "799d53109cc33d5d1eea6e11a35fbbe5d6947547fca03847ea2ac5a07ef3d2cc",
    "34eebbc7c92b0e43aa54b712cfe74d33b0dad03faf1e3c6377e1822baf762b3f",
    "f2a7799e2fdc328c68b19469b8227b7fdb908dd85a9730f0ba8a744ac5eaf8ba",
    "38ff76ecb60850a28bbc3722caa652cb20c2bc162f9bd20a97723d8ff537acde",
    "a43025ef900b54efdf4f8c7cce5c7177ad07149cad7cff807eae318ffcf1f6b1",
    "4c48d16d770e3be6f7e36a51e814cb3f7d95206782389e33502c04a93cd70b71",
    "23a5a48818edb4437d36124fe2fef90947985d72d8addf8db81e9b2c39cf5d21",
    "eb7eb826b851c21cd1f53bc4ac7bb4151c286e6aeef1bc6739362d4a3e2e533b",
    "acd86c3f8797d5d12a6094f4c00c0ba5ab375802541a7ea5e08fa9a1b49666a5",
    "52bd7ad8a9e1ab3e80292f4eca45da7f9701bbaaf0b6ac723fd6c00cb5d7a602",
    "3fe9fc1de53e7f5375f7975a871a943f166d2e9823120f828802cbb55582dcf4",
    "90a3de905a0e7f06ac953d2f49a5a2924f7fd18a487ce3bc149cd4babd8cc410",
    "3a7b9d03e20f5628fdeca4f28d62c1af1981fa79f6ed556549eecd2391607c6b",
    "30b3e9141f5bd78fa05e71c42dd837507d635333babf734d77cd997690471c64",
    "a0aa7bb77b85acda843dc4574c34a4e6850d5ec374f204b1de00775425c4277c",
    "70abdef25d11b3f6fca5d922a16f8416d2a7c50da4c1c3eb882c33363ad126d0",
    "e64d8be9bc87129d62b3dba54029e74004e295f6be0f243e31fc4fa6585d8fa9",
    "8a30c7ef3bddc3c0282de1991ca7f1be0678e548242a7b0d4fff8b8bc5e53f78",
    "7f2da7b5ba2090c4a7c9d49f1b58928572888b9426839e54803b22f0a8359bb0",
    "e260b90926952033048abfb7da5c26e1ad07a421b89081d5d805ceb83b1c1a89",
    "0b91f54851e139a6876582bffac68409dbdf348bc1e4796f723bb4c82f0a03d0",
    "1da7046a1bbb324a6806931ee47214efaf821289f5dbe1a68600ee047f098fc0",
    "6c5ef7a7fa3a14c15014414a87f1e02b071b29841e04c119c53398b2322f80c3",
    "6278edcb323770674ec9bf9c94f7cfc7f50a38d6bbe4405fb5036506e9dd4295",
    "3910d6f7d0391ff7763607fd2c06c4cb94c19deb3b679972830b608fc033fd17",
    "9c39b715769c2e9ce0ca9c282a5a0e0ddf7a12bf5f04daa6fade6d38154160d2",
    "415a62dee4bdd61b4d9320f4e54b9f70ccb8b02062a20555ddc5c6017ea53649",
    "a78d5de6f13e76fb8e9cf35eaab5c8242c5f3ae5d27b44bd35bb52a71a26e7eb",
    "7ca89cdb187727d40c5659e5d4ed3cc9bdf791fa17f08ac78a8f3a1fba949e8f",
    "49f61b2c2325aa2b0dbaa30ff8730e4cb275d1075b46a62af86bc8b67eea0054",
    "7ff0a1f40daf3851c770d9eb4d89a3431b9e8c8c2b05d8854b6fef6e99b140b8",
    "5175782c31e7ca4a5cf117371b07ce3e06d165cf25a42d33c705ebe4a432df1d",
    "05e79385f9fae9d8613b2b92bbb75385f9e461c1a3ce85b36d4bc79e29f7a795",
    "81aaf224bb219d9477d300197b6b9d6467ca421655ef108f25bf63b624a7aa61",
    "fe0fad629bcabd6070a307f6a1f528359f4d8b0a1fff12e815f704d6001f2133",
    "6149c6059cf7773c88c2b32e9f15ba600368497796ac25a3ef6913355e98393d",
    "b66634943ca9f470a33fd7585791ad5318150a9e46e02e4d765f21d9c957d753",
    "b15d754aa369c1a35e05ab441dadc691f46e387b40c353170b08a06731f83b35",
    "0a9baec22b77627376d4da02a37c894fa1eed6de46501abbe3468c80c55c85bb",
    "66035d42e77789cdd4d0532f72eb96571194f65dcc4fb365037d3818f27c856d",
    "965807d3dd79251b5f83925e619395696ec58e2815b7c50c933dcd2120ad4d54"
  ],
  "severe": [
    "fbb564790b7cf30cf3380409733c1dec2c4b224a95d1e41160f8a8e0ce171ce7",
    "eed453f44941c9906b10d3facc082fe6f736131f193ef235048ad95f7f4a8622",
    "bf3d7fdb637767189ce18e5b6f9b100c75eaea3c4a38f8601d4ef3ff4cf7961d",
    "a51cf36a85dc477cb9e83f735d3fa60da9869a27474a866b01026d6ead9f6eeb",
    "237c70260d7316d47e9305231c97002eb4c4f6cb1470e0feaa44bdf7cf936883",
    "910b3cdfe1baa837ad147d166587ba524203cdf30aa22138170bbaa545804150",
    "21647eb853c4eda76d21c0633c31a16c2518204e6fb33e42cab51bfd5b714fcb",
    "0dabd5ff7e18c37799b98d1df1e58f5e97733190fe5b0985d362f05040a9e30e",
    "2a6204dc18e0ad20874bd0c8eacaae16888502a717d6d33bd595369609744789",
    "2df4c994dab15446cf951ce69e017ed163fbb868f823be6c38bc0099c5328438",
    "5eb4b1e607a66cda35a44b103fe0374464236d9624a35748e24f6cb32f22b627",
    "ed0d2c1d364485409b954c143693ea6c804756f9ca151a9272db6bc2902ee7d2",
    "d7b90e4d99fe11f21ff911ff40351f89cdc2acea3d75ee088788c610b3575bb1",
    "f32979bcb0c44d0cc8e1dbffcd4c0a7f5296bd6359cb53ae43c06b8de7631ba9",
    "f4a5421819f37c8f795290606f2a693bb187eabf244c406f521177c64940fdfd",
    "2f40704f157924cc18290c1f27cd53f1ead0bd6b45209d4a4146a6b57751e252",
    "2d0314b0fe9681503db291475710df5242e118747d92fecac33a89dbbdcb600b",
    "72e1af0c86e5846972dc08dad91fe354e53d3a74904d1c236bc10bb05a420ada",
    "87f28415331eb58494eaf8557cfa29f4dcd3185e2946dc36869488e4076457dc",
    "98609b0743bc23cf6ee1572024376dd2a004b84ae6a42ebb916faa2d75332731",
    "ce306772c030372261dd4c84c199f388e7f8ff452c5f0fea95a26a4fd5019b76",
    "c541aa4d60aefd117378466e3bfefd068185187464629bd134fe590f896c019e",
    "b9d897459b309c19f9d104db3bc184ac583735ce0c8899a7ac1ac2bcf3e65090",
    "eccd41139cafc3a7b6705d0782a0d05ac05bee1ee789c092ab5ea692e2ac6217",
    "5742c0a560525b7f79549246e46e89807d63b6d4a2db3894dd00b89c0f897bf5",
    "18daf89cc35885d607733a00b9010f0e52d53045ec3afa30d3b74137cfc6b644",
    "8c830494b99ef6557ba9343db09275c928e46cba669606fe2b2a7aff659dd48d",
    "5b020723823406376bec6a5a26ceb5d0535638198737db3aeb4ed2834bbf9dd0",
    "96394e6c7ceaec8753bbbb953ccf47d3126ce388e76bad3fbaad2cad535fb9c6",
    "636ee26bc22a73f1da3a5a8e4f0491b4ed1c6b202bfe014462e1422c91befe90",
    "240ae68396059190c241cd7e9ae026c9ba0cd9769212be319e7ba7f1bf839035",
    "3f1a35d0e517b64cf72be0e3cf3307caddd0924144e3612ace67299b06a69641",
    "99e1ed67fe3be463eb41b2c8c840dd1424ee2daa7e7f79214e9cc10ce0b3cbe6",
    "6c26966ff260a6e4450ce7054321ab4297c57401c1eb6e604de1886a25ed4a9a",
    "cb5c6346e4571d1ea120fd16c57778d532d9296e4cdf7929804af2356533d865",
    "a20e86f65adac0d32e5dee2875fddcc1f9d45250bc87e1e2556271424b0ea89f",
    "4453906b19336bf41205e7774d2370fbbe5957ecb2cd4f8cced451d1884c464a",
    "3f1a658f4fb8c8838d244b41f250fa0d3176316039b6579787dee9a909b1bbb8",
    "52b04bd10aa82e9a2fe0a62dbb77667ce951bed2cba3069331969a6118c1f989",
    "0560d64f42269c1557f3e5a590f2a52efeb6ec7a41d5857c2c4c56b3366a0465",
    "ee47609c52340e8628eab0e0136d04d338759486154bd7ce22b23d9715fe1f62",
    "512854840b41b5d6cfa1bd876cd00406a8c39eb116feaec8b48081b4f3796c0a",
    "034b977ed9d78fa444780773dcee44948edca9a7284803bc9971f1465e78dee9",
    "99953ec6d7407f50201d5db81e532c0e399ab040ae6f45f22266cd27751082e8",
    "3b0d5fd43aacf365b1be8ca3706c5aab9ef46a2ab6d3c7d63366100301698a76",
    "cb181c6fd21cd9f69201d538e6b2746dc88bf28f60d6f270cf9f087465425bc9",
    "cc1a427e9a00281fbf66de93429d805b3a250f96a07398f11855554599b9a260",
    "9efa0df8177729a1f66d79edd9ff6fb4c6d1784d37874d754151417ec39a04a7",
    "22130530057dc8568bc2114a522475234e647c0e50d8389b68e342a0ec7abf3f",
    "18905127131d6aec6521779ba0ad0c18d35114b1b161a9acf5114f154aeceede",
    "ef4eae8080dca2eb0345c6577e58aa1f0223968a1f813d007d5e47392101e9bf",
    "8f0676965289377dd8e8acfcc013adb540d14ac613204324a85b759e9bf31773",
    "0386f7be59ddb99a392f71a92180b507ce02510f54b64413008b5bf85f907501",
    "4d0dc401dc1c368cb8d6721567a8a844311e2a6a174ac63e9da028d2054be56e",
    "853bb9e9b0e2a82f233daab13ae806dbbc2ddb5cb8bd6fa810c450c745e47bdc",
    "995c6b0fa860636e4e0228b27220353349a4a799abeb3107b5649371e7928be2",
    "9638758ab008c8340e34f67c3f042c9c37508b9de35452b469704f658b673c5f",
    "ba1269b8e4e3ae5729c7ea881a336fd8f0a79be985e056b701893f70de8c6a5d",
    "d91e5733d545ab131c1f5e2c6d79bed4a21ccbb77787fbac9dcf0c7415bd9106",
    "bf51a6861755d579e556c24c7f2010413fd76fd89517ccbc7f8c48fd59cb83cc",
    "8f8fe662d18c0055a756f4b18eeb0ad47859135e9e64139ace479768d524ccf5",
    "d343d3049a6b1f18c7f40bc999548e66a2c0cdfbc7b190c4bae50b2d09adf1ac",
    "5b14baff6f7d91fb16f7e510a83a26254d7dc496e5310b1e26952376f79031b3",
    "1f166a2d59262fdfb9fcee3db9cba5b76bd080c76c99c7bb218a189bf15ebd21",
    "61441bbc0328ed8f0fe88976077dbbd1df1d04321e882e7c9a8deb63f82d6ac2",
    "a7ddff86ebce47b39aa48f341b0563a91f0e5e410248e92c01a267c2a165e350",
    "9ef19d98c6a6594a05df6f11ba26be3babd37994ba200287089c8b78abab4728",
    "0a9596d90922b1d4cdf4281eec6a38e6958f884648339593b3af33cf9b49c8bf",
    "8cefed028309acbad327a91ee6239df325a9d3b933ddf2f651a8c171a10abab0",
    "fbf89e17156cf3db6f3599e9c84c8c5446f45c633b2fb72ec4cb8fdc8519ecf0",
    "51f811cb631caccd00ae3d487a1f8bbc6a2d595a3d7273842ba1c02c15ba93ff",
    "929b2a02622533176aa402ed665180919fecd9de208487557926a4729e79608d",
    "20940b130b374c44bfca50beb7c310695fdcb1253dee540f56f7f6f4d47209c9",
    "4bf9e1fa8c5224b690cb61027fd4e48ecdfd58fd5d0068d6b7b28aa6ea871a96",
    "442c07e8629edd0fe2a1339a121d2285bff0fcd0af0088e1df3cefff44a74600",
    "b1c5276a83fcb40e199b58a685566e25916014993b2bf537464b748872d54fcf",
    "7395590418f2bd99172775056d8032184142234f2bb8bc63d71cecf7b708e3ef",
    "90d5ddcec33d30dc836cc9e4d70860018c6462d4684d90b020deca63d6a2bf24",
    "6b6f659204cd3bfc2f9a75fcb059150d758b34e5eb0dc7684cf0b5abe66b18d4",
    "3afc3099fa946bc4cb4a0e773c4c910d8461db4e5dee8db0fab3fcb80434888a",
    "566469cb820b9828113490b1e84de059cf93d3e46cf6bbb4f86283164241a865",
    "12861e5add3480fdbce3251f5e8d93ce241bd62fe02f68d447a93fcd2f059527",
    "16de64f785b69b8821bf91620f5cb4f170394d8941ff9eba4d855cf1715d0253",
    "5d502a8ec4dd0d6129594a49e6522557756aeb55089273dcb7f40d063a07f7ab",
    "2389b442d66cad16779da4f6925536306da0f0315ebd6a5f415532a8c06c110a",
    "957e01afa6662e77567ea0bfaf88419d34ac6af27228ea8c9fe1d6b6bc1472e2",
    "d1b18eeb04be54ef112b34b1dfb5d50f962276a7e7ae2c9abb89eda07f4a5871",
    "eb94185d78ec27894c36dfc6ce2f5eed34d269475985acda1e43af02f69d268b",
    "aed0581f03cd286b59c74c4553b5819b38d58026175d3a53a3a80c51a92a7d05",
    "ae232ffb2fa959427c2dd2a73e8e6c5bffeb46fda9342145cb8b435068b05b0d",
    "020ad5e0500ed022194addacd7dffb8312315600060bc072c8b22d1ad16d8be9",
    "8a1932f808f11d69758c436ecf98a526e379b91fcf0ade4bf870380cf183cdcf",
    "b894737943b3ebaa7e9a5a73d12a53bfbcf100853365b4c1010bbfdea7186094",
    "1f2164d3720a94498a1776492a90399cc7d2d1bd884f23961e54e9922f01b688",
    "eaedaaf52aae827fc2cf5b9e98eb9aa4dd3de4c5b3e8e87bd94374b50080983f",
    "ab9e86e9b59f2555b2e1554f4a542b37f313261fa0e580d93146b5ea46186241",
    "54913139601aa34edd8d346256fb5729289ded4bad258a97a02bfa2a726684d9",
    "f37d5813a7de8478377984deeb7668acb48ce56ca9734b781c5557975e88ede2",
    "6e2b3f05e1b76755868a583177eb0e3853acd229e966226925d199f6ad4b4f0d",
    "3d996771b6bfd64a3db585b98afbbf42422b8169fcd86d44854ab32ee1158704",
    "c12dc9ba829efb6aa26e1cbcc72b8e4cc18a01cf3d8cd0ac18dace5f2c2c9c1f",
    "ce465a6a7b71a67338852d816a6659723997185c1b4dac9c21767b5aedc9a3d7",
    "e673cf8740d17aece4c1a2a07cb4b4b8d26ed4d997409de6a8048b8dbde039f5",
    "f18b852393b21c61d3ddff53479f339af431d9848875a9e2cd156f18b6d00c2f",
    "bd1f1d0a19cb5bb9b7ee96a06b223ac2cb8a7ef0363c6ee2a49aa3e5a6dbb51c",
    "cd7a55c7b62c01a21eeffd8bd23e760456d550dd1fce1833ac162ed9ff535f05",
    "02f00e54f6986b0a11a546ecb9a8550b245f9c63daad85bc61903e4f7e0e2f36",
    "0339eb769b314d9596e49c02cf65df02c55528c4e555d6980e8c952e6f9a79dc",
    "33563441e7f5ea11af7dd06a8d5ed603682999dae222ef00f7fa099d7d170c30",
    "f6f866d2b7215bf444386590d3850de6e651cccdfc6cd850c27f398ebd851d66",
    "d88614a22f21f800213a11f95f2597cdf1438cfac9c41d829b4aa160db2b17e6",
    "5b09a89f44dd69a2e4da862cb28bfe4f9fd299eef2a91a6ec3474608cb5571ee",
    "1e78635f590a732b177296b74ef0fe75af992274b6e7bca4befce96917e84623",
    "ea5c2c277b594d7c52bdde5160a6061a6f0ff08fa2d5988eb94fd93e9a1f9d78",
    "75e688e8699d522f87f70fbdf0c6e2a6e3472e8c2e39c8d00b1c0e753eeae07b",
    "98cd2797b99cd98456aa9b7716e404c436146aa3aa0f11753a137c08c00e3c7a",
    "06f986137303f99adfde2b389e57a557c96b4f764a9e2da7b36598d76b678267",
    "2ae013be4d13a7f6e26396ea801a95fac0d6c04e0f208eb67fe48d0ce6803628",
    "b0355b20abbbc28f8719833b3830c6720fcbc41760470b64daecddf5257a106a",
    "6a64294b66566d9d43f135c2b3285b5d656a456dc303b72246cd08582e1050f4",
    "d077aecedce45461f4190bf962a4c95e4e9270a5b0211aeec8f3f3ff2bdcae18",
    "e0cb739294a6c326e37fd37c5d6aa63791d2913943ee65391b6e0fbe72f6777e",
    "6a8da773f7d1bfd3f8d10c4afee2037e3f0b0098c220b4f6229307e52f5426e2",
    "4ad41eee9ffff4e48dbdbe4ea3ed520897c545fbad97c64d15a8e23a05919888",
    "d2d484c03913d939f9c822821c36cb53a74d0e8c2276331d938ce7d229d28132",
    "25c54853aa73973a4657ba85996cce2144a39207197d19f7181ef90f1615e56f",
    "9ee2086a63bec4d33a6e2cadec34c20af84fd8f3ea717bc2b28d273e17c2e4fa",
    "c10327e787649b2d944a8729a2df0bcb5f57188dffa49889faf57ae63ae636be",
    "280dd88e7d33195df8c56b4223fe6540f02e81a32fd26887884285d19f09b6f5",
    "8b2dbb9ef8a52c2d9fd5619bd1791916b3c282303938bdf9feac0fa797e02d3f",
    "f246d6d0e73121b32f30ac172a0adb4655ea07e34981e98a390bfdc9c022b6c9",
    "fda6177c5e2263febfce5a0cf9ff33f3ce123292ba1d2aa606da7a8f4adafcf6",
    "86780dfea9ed5a6fe5caff513e6d3179e16fefad163ce96dbb0954699ce804a2",
    "52ec8b9c7d367a39546810ba30bbd6482177ae90d3082d845458ac75ca9f829f",
    "c1393b6b0af3079f1512c5d5c82da7e1520dbccef417348936c21d40a9630bfc",
    "2a48e55d0453a707cc0ba4429ce35666fe71b61a999345acc62cebae30aef7f8",
    "3692100cda3a2639792a8edef4a1b5387cbd52a44c64da3932741969533037ab",
    "e20fef5798cd7a198a9ebffbc5833cbf945b9d0b38353d74cc70d597d269a0c9",
    "5c91b68f1e46daf66aa12a659d3d459c316ed103ad17aac3e05b55874d8f7a85",
    "16182c4eec493935e0b2efcee25d4fbbb5dd5457f26d27782ecf6519fb667f93",
    "5792ffe91aa60aff3f797d558b0a3576bd7d60d5dd07cb331c35c9dd5bc592ae",
    "05f989d9e4e3767a8ba9c897b4908178fe42e38e8a9c837991537b04de1388e9",
    "4a4188a51f1a107fad6a095f14b7ba91bc2effb7fb36527d38c4c07d6f2ffdb4",
    "4aa939738a915aa9f7ed04883872074077590d429ebd93a2c54727502e545b07",
    "2b698533a55ae746558a6639a8ea72751229818309ecbb05dacc5149243e1021",
    "f62e40efecf0adabaeff985ef4928029e85d34be0b9b61d11c2e4a29df0db2f3",
    "88b51d4f41b9a350ce98520a21c5c1be73a864565a630bf57a9bb48a7420cb71",
    "ef8d853307a987424136ffe75da998bfccab7a5b7b5c21cd802900b72a26ba2a",
    "f339727adb49954b06e2669390b20f0df7463b5d13d354dfb7622bcf17554d72",
    "30a9cf9babfdb9944e522e010cf8a8dbac34ac062aab8ec6e99aa08b20a23acc",
    "c6e790ce5c688973cee691c249e68188ef0b70980f0fcf36ac4edacee170d9a0",
    "163fa234911cfad54cb0eaf79bffdb3e91b555a8d8d7b619ff8cda24a5c0c1ac",
    "366bf3cfe8b8107c9d745240068ac88d05de14992fe81e0589a283c279556c88",
    "2fb68e2956752dea9e5262ddbed616f055b4c77c51d714f3035fb337ed0a249b",
    "863ec3049b758e157bb84d1ec692b3c780a0f6370c7b256891a1b6feb1b45067",
    "453cdcaabe12617bff910ee418c0ac1e6e2670020b2cf90a8becc72998c271ed",
    "b9be07298b7b6f2b374b3bde98071be2e045cdff12fe8a0d0a57718faa35fbcb",
    "7f01d294c710fd6d1bae545bfcef95af992a64d049a5f5363b52b78dd4703405",
    "4321db95e39ef0a26d9a0ad11068202a7e46141372411becaa3a89c94b7ea63a",
    "231483991932a9265795d8c5ca90b6ea2dc3c8ba28284b760ca989b34dec9c2c",
    "58bafb886b2473d533bf9234b1fe8ad2e1c0cc41ae7a504187f69f04710b5632",
    "0c0fd21d2b965dc014f4006ae4c09b88a46c6c99516b6add6fc4c51817b3fdb8",
    "48605a0fd2850b3d9e8efd4d33bc3357db0406b5c08611e642e7ce80feab3e16",
    "2828fc4bfcf78d3f106d786f883acef3be436aadfe64324cfc0262f2366aa1d6",
    "549486b4a4330d463b4c8c5e4489c926e6341097c737bd91a3913326e9cfea9a",
    "5b064a51b72dad212103a1614e3b5cd95b5d16f9e05cb331e6854be161fab73f",
    "bc55f6c1b38f16b36762fbf0971e8999e039a2f85eba2844eeffd3627bf57629",
    "a0225efaeff4d6b3c1a071545d0ea30aba6c6e99b3d2f5e2f89eb6d59d3e1e93",
    "1996f29b1e62831f915be990a73911c3a6f85b6e3e2e243376fbe0669c39813c",
    "95e3bf10c0233a045c76da4476f67082c1244557dc178ba2b23d9d47ef3784f9",
    "47d0eac0ca6005bdc33d4033356e3dae454be74241de08c09b4f7a5bc5046299",
    "9431fbe43d33bf7da02a99aba7cb64bc62f35de836c07085bbf1b5a2577f808d",
    "44b0a297ed17ccda32fa69a7dfb120e0beb766403b0a15c4a7344d941c1e8343",
    "2991837dbe1e4c49e6748065d749da7266de99764904eac8e463f47080f427b8",
    "46d8921e012f7cb0467b924f670829457f130bceb03894a62f675aa177eb6f49",
    "c23e249641c3fac9e2f1e29948d19477a65cb36701782f88db631c5b071f0165",
    "211731c3e3408eb51472b075f1e92fa532b4f284cf10b8f1ca60ae5789cd29b8",
    "a748de8717deabb5a6186f1838c882cbf03f09db6b7eab1b3f45df68ea31be86",
    "437c91d6698e8a0a8210620287bc6dc5c2369d9048ea1aadad89edc333126097",
    "758d13959ba9c6dd422a8b39ed9fdee8cf7443255eae433407ab24ab0aa0014d"
  ],
}

if SEVERITY not in _DISRUPTION_REF:
    raise ValueError(f"SEVERITY inconnue : {SEVERITY}")

PPO_REFERENCE_ROUTINE_HASHES    = _ROUTINE_REF
PPO_REFERENCE_DISRUPTION_HASHES = _DISRUPTION_REF[SEVERITY]
print(f"reference PPO chargee pour SEVERITY = {SEVERITY}")


def scenario_fingerprint_case_c(sc, decimals=8):
    h = hashlib.sha256()
    h.update(str(sc.kind).encode())
    for name in ("demand_mean", "demand_cv", "demand", "avail", "temp", "hum",
                 "dev", "Rbar", "Wbar", "material", "repair_est"):
        arr = np.asarray(getattr(sc, name))
        if np.issubdtype(arr.dtype, np.floating):
            arr = np.round(arr.astype(np.float64), decimals)
        h.update(name.encode() + b"\0")
        h.update(str(arr.shape).encode() + b"\0")
        h.update(np.ascontiguousarray(arr).tobytes())
    return h.hexdigest()


EVAL_GEN_CASE_C = CaseCScenarioGenerator(INST, PRM, NOMINAL, EVAL_SEED)
CASE_C_EVAL_SCENARIOS = [EVAL_GEN_CASE_C.sample_routine() for _ in range(N_ROUTINE)]
for _s in CASE_C_EVAL_SCENARIOS:
    EVAL_GEN_CASE_C.assert_routine(_s)

_routine_hashes = [scenario_fingerprint_case_c(s) for s in CASE_C_EVAL_SCENARIOS]
PPO_REFERENCE_ROUTINE_MATCH = (_routine_hashes == PPO_REFERENCE_ROUTINE_HASHES)

_stress_check_gen = CaseCScenarioGenerator(INST, PRM, NOMINAL, STRESS_SEED)
_stress_hashes = []
for _fam in DISRUPTION_FAMILIES:
    for _ in range(N_PER_FAMILY):
        _stress_hashes.append(
            scenario_fingerprint_case_c(_stress_check_gen.sample_disruption(_fam)))
PPO_REFERENCE_STRESS_MATCH = (_stress_hashes == PPO_REFERENCE_DISRUPTION_HASHES)
PPO_REFERENCE_SCENARIOS_MATCH = PPO_REFERENCE_ROUTINE_MATCH and PPO_REFERENCE_STRESS_MATCH

if not PPO_REFERENCE_SCENARIOS_MATCH:
    raise AssertionError(
        "Les scénarios générés ne correspondent pas aux empreintes du PPO "
        "case_C_severity (3).ipynb. Ne pas lancer le benchmark.")


# ============================================================================
# Lancement via la même chaîne chi -> quantile Gamma -> EIP -> projection -> V
# ============================================================================

def effective_inventory_position_case_c(d, S, L):
    eip = np.zeros(d["G"])
    for g in range(d["G"]):
        jf = d["Jg"][g] - 1
        eip[g] = sum(S[g, jf, a] for a in range(d["abar"][g, jf] + 1))
        for j in range(d["Jg"][g] - 1):
            eip[g] += PRM.omega[g, j] * sum(
                S[g, j, a] for a in range(d["abar"][g, j] + 1))
        eip[g] -= float(L[g])
    return eip


def release_grid_case_c(sc, n, d, S, L, material, chi_grid=CHI_GRID):
    eip = effective_inventory_position_case_c(d, S, L)
    R = len(chi_grid)
    cand = np.zeros((d["G"], R))
    opn = np.zeros((d["G"], R), dtype=int)
    for g in range(d["G"]):
        for r, chi in enumerate(chi_grid):
            Y = sc.quantile(g, n, float(chi))
            v, o = project_release(max(Y - eip[g], 0.0),
                                   PRM.ell[g], PRM.umax[g], material[g])
            cand[g, r] = v
            opn[g, r] = o
    gaps = np.array([
        np.max(np.diff(np.unique(cand[g]))) if len(np.unique(cand[g])) > 1 else 0.0
        for g in range(d["G"])
    ])
    return dict(chi=np.asarray(chi_grid).copy(), V=cand, O=opn, EIP=eip,
                max_release_grid_gap=gaps,
                chi_max_abs_error=CHI_MAX_ABS_DISCRETIZATION_ERROR)


print(f"PPO rotations exactes          : {PPO_N_ROT}")
print(f"PPO actions rotation×keep      : {PPO_FULL_ACTION_COUNT}")
print(f"Affectations intervalle uniques: {len(PLAN_LIBRARY)}")
print("Action-space PPO match         :", PPO_ACTION_SPACE_MATCH)
print("24 scénarios routine match     :", PPO_REFERENCE_ROUTINE_MATCH)
print("180 scénarios stress match     :", PPO_REFERENCE_STRESS_MATCH)
print(f"erreur max discrétisation chi  : {CHI_MAX_ABS_DISCRETIZATION_ERROR:.6f}")


reference PPO chargee pour SEVERITY = moderate
PPO rotations exactes          : 54
PPO actions rotation×keep      : 864
Affectations intervalle uniques: 89
Action-space PPO match         : True
24 scénarios routine match     : True
180 scénarios stress match     : True
erreur max discrétisation chi  : 0.002450


In [16]:
def case_c_observation_view(sc, nominal, INST, n, h):
    """Vue non anticipative de [h,H) dans l'époque n."""
    E, H = INST.E, INST.H
    Hr = H - h

    obs_avail = np.asarray(sc.avail[:, h, n], dtype=float).copy()
    obs_temp = float(sc.temp[h, n])
    obs_hum = float(sc.hum[h, n])
    obs_dev = float(sc.dev[h, n])
    obs_R = float(sc.Rbar[h, n])
    obs_W = float(sc.Wbar[h, n])
    obs_mat = np.asarray(sc.material[:, n], dtype=float).copy()
    repair_now = np.minimum(np.asarray(sc.repair_est[:, h, n], dtype=float), REPAIR_CLIP).copy()

    avail = np.ones((E, Hr, 1), dtype=float)
    for e in range(E):
        if obs_avail[e] < 0.5:
            predicted_down = max(1, int(math.ceil(max(repair_now[e], 0.0))))
            avail[e, :min(Hr, predicted_down), 0] = 0.0
    avail[:, 0, 0] = obs_avail

    temp = np.zeros((Hr, 1)); hum = np.zeros((Hr, 1)); dev = np.zeros((Hr, 1))
    temp[0, 0], hum[0, 0], dev[0, 0] = obs_temp, obs_hum, obs_dev
    Rbar = np.full((Hr, 1), float(nominal["Rbar"]))
    Wprof = np.asarray(nominal["Wbar_profile"], dtype=float)
    Wbar = Wprof[h:H, None].copy()
    Rbar[0, 0], Wbar[0, 0] = obs_R, obs_W
    material = obs_mat[:, None].copy()

    return dict(avail=avail, temp=temp, hum=hum, dev=dev, Rbar=Rbar, Wbar=Wbar,
                material=material, repair_est_obs=repair_now,
                position=np.array([n, h, n * H + h], dtype=int))


def case_c_remaining_dims(INST, PRM, n, h):
    d = build_dims(INST, PRM, N=1, H=INST.H - h)
    d["Cbar"] = PRM.Cbar[:, h:INST.H, n:n+1].copy()
    d["Delta"] = PRM.Delta[h:INST.H, n:n+1].copy()
    d["rho"] = PRM.rho[:, n:n+1].copy()
    d["n_lo"], d["h_lo"] = n, h
    return d


def forecast_current_epoch(sc, n, G):
    desc = sc.forecast_descriptor(n).reshape(-1, 5)
    return desc[:G, 3:4].copy()


def nonanticipative_payload(view, demand_forecast, S, L, K, prev_asg):
    return dict(
        S=np.asarray(S).round(12).tolist(),
        L=np.asarray(L).round(12).tolist(),
        K=np.asarray(K).round(12).tolist(),
        prev=np.asarray(prev_asg).round(12).tolist(),
        demand=np.asarray(demand_forecast).round(12).tolist(),
        avail=np.asarray(view["avail"]).round(12).tolist(),
        repair=np.asarray(view["repair_est_obs"]).round(12).tolist(),
        temp=np.asarray(view["temp"]).round(12).tolist(),
        hum=np.asarray(view["hum"]).round(12).tolist(),
        dev=np.asarray(view["dev"]).round(12).tolist(),
        Rbar=np.asarray(view["Rbar"]).round(12).tolist(),
        Wbar=np.asarray(view["Wbar"]).round(12).tolist(),
        position=np.asarray(view["position"]).tolist(),
    )


def write_dat_case_c(path, d, view, demand, S0, L0, prev_asg,
                     plan_library, release_grid=None, fixed_K=None, fixed_O=None):
    """Réutilise le writer historique puis ajoute seulement les objets Case C."""
    write_dat(path, d, view, demand, S0, L0, prev_asg=prev_asg,
              repair=0, K0=None, Opn0=None, fix_asg=None)

    P = int(plan_library.shape[0])
    lines = ["", "# ===== CASE C FAIR ACTION LIBRARIES =====",
             f"param PCOUNT := {P};", "param PLAN_LIBRARY_ON := 1;"]
    rows = []
    for p in range(P):
        A = plan_library[p]
        for g in range(d["G"]):
            for e in range(d["E"]):
                if d["eligible"][g, e]:
                    rows.append(f"{g+1} {e+1} {p+1} {int(A[g,e])}")
    lines.append("param PlanAsg := " + " ".join(rows) + ";")

    if release_grid is not None:
        cand = np.asarray(release_grid["V"], dtype=float)
        opn = np.asarray(release_grid["O"], dtype=int)
        R = cand.shape[1]
        lines += [f"param RCOUNT := {R};", "param RELEASE_LIBRARY_ON := 1;",
                  "param FIX_RELEASE := 0;"]
        rc, ro = [], []
        for g in range(d["G"]):
            for r in range(R):
                for nn in range(d["N"]):
                    rc.append(f"{g+1} {r+1} {nn+1} {_fmt(cand[g,r])}")
                    ro.append(f"{g+1} {r+1} {nn+1} {int(opn[g,r])}")
        lines.append("param RelCand := " + " ".join(rc) + ";")
        lines.append("param RelOpen := " + " ".join(ro) + ";")
    else:
        assert fixed_K is not None and fixed_O is not None
        lines += ["param RELEASE_LIBRARY_ON := 0;", "param FIX_RELEASE := 1;",
                  "param K0 := " + " ".join(f"{g+1} {_fmt(fixed_K[g])}" for g in range(d["G"])) + ";",
                  "param Opn0 := " + " ".join(f"{g+1} {int(round(float(fixed_O[g])))}" for g in range(d["G"])) + ";"]

    with open(path, "a") as f:
        f.write("\n".join(lines) + "\n")
    return path


def _primary_cost_expr_ampl():
    return ("CostProcessing + CostChangeover + CostHolding + CostBacklog "
                        "+ CostDeterioration + CostDisposal - Revenue - TerminalValue + CostIdle + ServicePenalty")


# Tolérance numérique AMPL uniquement. Elle est très inférieure aux tolérances
# économiques / de décision et évite les faux "infeasible" de presolve à ~1e-14.
AMPL_PRESOLVE_EPS = 1e-10


class FairCaseCPlanner:
    def __init__(self, model_path="model_case_c.mod", timelimit=90, mipgap=1e-6, verbose=False):
        self.model_path = model_path
        self.timelimit = timelimit
        self.mipgap = mipgap
        self.verbose = verbose
        self.n_solves = 0

    def solve_decision(self, d, view, demand, S0, L0, prev_asg,
                       plan_library, release_grid=None, fixed_K=None, fixed_O=None,
                       tag=""):
        data_path = "data_case_c.dat"
        write_dat_case_c(data_path, d, view, demand, S0, L0, prev_asg,
                         plan_library, release_grid, fixed_K, fixed_O)
        a = load(data_path, self.model_path)
        # AMPL presolve peut parfois déduire deux bornes qui diffèrent seulement
        # de quelques 1e-14 (arrondi flottant). 1e-10 reste négligeable ici et
        # empêche que ce bruit numérique soit interprété comme une infeasibility.
        try:
            a.option["presolve_eps"] = AMPL_PRESOLVE_EPS
        except Exception:
            try:
                a.setOption("presolve_eps", AMPL_PRESOLVE_EPS)
            except Exception:
                pass
        try:
            a.option["solver_msg"] = 1 if self.verbose else 0
        except Exception:
            pass

        st1 = solve(a, timelimit=self.timelimit, mipgap=self.mipgap,
                    verbose=self.verbose, label=tag + " | primary")
        self.n_solves += 1
        if "solved" not in str(st1["status"]).lower():
            raise RuntimeError(f"résolution primaire non résolue : {st1['status']}")

        sol1 = extract(a, d)
        unused1 = float(sol1["V"].sum() - sum(sol1["P"][g, 0, :, :, :].sum()
                                              for g in range(d["G"])))
        J1 = float(st1["objective"])

        expr = _primary_cost_expr_ampl()
        tol = max(LEX_PRIMARY_TOL, 10.0 * self.mipgap * abs(J1))
        a.eval(f"""
            subject to CASEC_PRIMARY_BOUND:
                {expr} <= {J1 + tol:.17g};
            minimize CASEC_UNUSED_RELEASE:
                sum{{g in PRODUCTS, n in EPOCHS}}
                    (Rel[g,n] - sum{{(gg,jj,e) in GJE, h in INTERVALS:
                                     gg = g and jj = 1}} Prod[gg,jj,e,h,n]);
            objective CASEC_UNUSED_RELEASE;
        """)
        a.option["cplex_options"] = (f"timelimit={self.timelimit} mipgap={self.mipgap} "
                                     f"return_mipgap=3")
        # Réappliquer explicitement la tolérance avant la 2e résolution : c'est
        # précisément ici que les deux bornes de coût peuvent amplifier un bruit
        # flottant de l'ordre de 1e-14 dans le presolve AMPL.
        try:
            a.option["presolve_eps"] = AMPL_PRESOLVE_EPS
        except Exception:
            pass
        t0 = time.time(); a.solve(); dt2 = time.time() - t0
        self.n_solves += 1
        stat2 = str(a.get_value("solve_result"))
        if "solved" not in stat2.lower():
            raise RuntimeError(
                f"résolution lexicographique non résolue : {stat2}. "
                f"presolve_eps={AMPL_PRESOLVE_EPS:g}")

        J2 = float(a.get_objective("TotalCost").value())
        unused2 = float(a.get_objective("CASEC_UNUSED_RELEASE").value())
        if abs(J2 - J1) >  tol + 1e-7:
            raise AssertionError(f"lexicographie a modifié le coût principal: {J1} -> {J2}")

        sol = extract(a, d)
        terms = cost_terms(a)

        # IMPORTANT — lire directement le plan binaire choisi par le MILP.
        # `Asg` est lié exactement à `UsePlan` dans AMPL, mais CPLEX peut rendre
        # les binaires avec un petit résidu numérique (tolérance d'intégralité).
        # Reconstruire l'ID depuis Asg avec un seuil 1e-6 peut donc produire un
        # faux "hors bibliothèque" alors que le plan MILP est bien admissible.
        up_df = a.get_variable("UsePlan").get_values().to_pandas()
        chosen = []
        for idx, row in up_df.iterrows():
            key = idx if isinstance(idx, tuple) else (idx,)
            if len(key) >= 3 and int(key[1]) == 1 and int(key[2]) == 1:
                chosen.append((int(key[0]) - 1, float(row.iloc[0])))
        if not chosen:
            raise AssertionError("aucun UsePlan disponible pour l'intervalle courant")
        pid, use_val = max(chosen, key=lambda z: z[1])
        active = [p for p, v in chosen if v > 0.5]
        if use_val <= 0.5 or len(active) != 1:
            raise AssertionError(
                f"UsePlan non one-hot: max={use_val:.12g}, actifs={active}")

        A0_raw = sol["A"][:, :, 0, 0].copy()
        A0 = np.asarray(plan_library[pid], dtype=float).copy()
        plan_link_err = float(np.max(np.abs(A0_raw - A0)))
        # Tolérance de diagnostic seulement ; A0 exécuté est le plan exact de
        # la bibliothèque sélectionné par UsePlan. Cela ne change pas l'optimum.
        if plan_link_err > 1e-4:
            raise AssertionError(
                f"PLAN_LINK incohérent: plan={pid}, erreur max={plan_link_err:.3e}")

        release_index = None
        if release_grid is not None:
            # Meme correctif que pour UsePlan : on lit le binaire UseRel choisi par
            # le MILP au lieu de reconstruire l'indice depuis Rel avec un seuil,
            # que la tolerance d'integralite CPLEX (1e-5) peut faire echouer.
            release_index = []
            ur_df = a.get_variable("UseRel").get_values().to_pandas()
            picks = {}
            for idx, row in ur_df.iterrows():
                key = idx if isinstance(idx, tuple) else (idx,)
                if int(key[2]) != 1:
                    continue
                g0, r0 = int(key[0]) - 1, int(key[1]) - 1
                v = float(row.iloc[0])
                if g0 not in picks or v > picks[g0][1]:
                    picks[g0] = (r0, v)
            for g in range(d["G"]):
                if g not in picks or picks[g][1] <= 0.5:
                    raise AssertionError(f"UseRel non one-hot, produit {g+1}")
                r_sel = picks[g][0]
                ecart = abs(float(release_grid["V"][g][r_sel]) - float(sol["V"][g, 0]))
                if ecart > 1e-6:
                    print(f"  [snap] {tag} produit {g+1}: ecart Rel/grille = {ecart:.3e}",
                          flush=True)
                release_index.append(int(r_sel))
                sol["V"][g, 0] = float(release_grid["V"][g][r_sel])
                sol["O"][g, 0] = float(release_grid["O"][g][r_sel])

        try:
            nvar = int(a.get_value("_nvars")); ncon = int(a.get_value("_ncons"))
        except Exception:
            nvar = ncon = -1
        try:
            a.close()
        except Exception:
            pass

        stats = dict(status=stat2, primary_J_before=J1, primary_J_after=J2,
                     primary_delta=J2-J1, unused_release_before=unused1,
                     unused_release_after=unused2,
                     seconds=float(st1["seconds"]) + float(dt2),
                     n_variables=nvar, n_constraints=ncon, plan_id=int(pid))
        return dict(V=sol["V"][:, 0].copy(), O=sol["O"][:, 0].copy(),
                    A_current=A0, plan_id=int(pid), release_index=release_index,
                    sol=sol, stats=stats, plan_terms=terms)


FAIR_PLANNER = FairCaseCPlanner()
print("FairCaseCPlanner prêt : lancement à h=0, plan réoptimisé à chaque intervalle.")


FairCaseCPlanner prêt : lancement à h=0, plan réoptimisé à chaque intervalle.


In [17]:
from scipy.optimize import linprog
TOL = 1e-9


def _maxflow(ub, group_of, group_cap, psi, Rres, projection="lexmax"):
    """max sum(P) s.c. 0<=P_e<=ub_e, somme par groupe <= cap, sum(psi*P) <= Rres.
    Renvoie P (m,), lexicographiquement maximal dans l'ordre des indices."""
    m = len(ub)
    if m == 0:
        return np.zeros(0)
    ub = np.maximum(np.asarray(ub, dtype=float), 0.0)
    psi = np.asarray(psi, dtype=float)

    # --- voie rapide : glouton dans l'ordre des indices ----------------------
    rem = {k: float(v) for k, v in group_cap.items()}
    P = np.zeros(m)
    for e in range(m):
        k = group_of[e]
        take = min(ub[e], max(rem[k], 0.0))
        P[e] = take
        rem[k] -= take
    if projection == "greedy" or Rres is None or float(psi @ P) <= Rres + 1e-9:
        return P                      # chaque P_e est deja a son maximum propre

    # --- cas general : LP du debit total, puis raffinement lexicographique ----
    #  Le raffinement REDUIT le probleme a chaque etape (les machines deja fixees
    #  sortent du LP et leur consommation est retranchee des capacites) au lieu
    #  d'ajouter des egalites serrees, qui rendaient le LP numeriquement infaisable.
    keys = list(group_cap.keys())

    def _lp(idx, caps, Rrem, Tmin, obj_e):
        n = len(idx)
        rows, rhs = [], []
        for k in keys:
            r = np.array([1.0 if group_of[i] == k else 0.0 for i in idx])
            if r.any():
                rows.append(r)
                rhs.append(max(float(caps[k]), 0.0))
        rows.append(np.array([psi[i] for i in idx]))
        rhs.append(max(float(Rrem), 0.0))
        if Tmin is not None:
            rows.append(-np.ones(n))
            rhs.append(-max(float(Tmin), 0.0))
        c = (-np.ones(n) if obj_e is None
             else np.array([-1.0 if i == obj_e else 0.0 for i in idx]))
        return linprog(c, A_ub=np.array(rows), b_ub=np.array(rhs),
                       bounds=[(0.0, float(ub[i])) for i in idx], method="highs")

    r0 = _lp(list(range(m)), group_cap, Rres, None, None)
    if not r0.success:
        return np.zeros(m)
    T = float(-r0.fun)
    if T <= TOL:
        return np.zeros(m)

    caps = {k: float(v) for k, v in group_cap.items()}
    Rrem, Trem = float(Rres), T
    P = np.zeros(m)
    for e in range(m):
        idx = list(range(e, m))
        r = None
        for eps in (0.0, 1e-9, 1e-7, 1e-5):          # relachement progressif
            r = _lp(idx, caps, Rrem, Trem - eps * max(1.0, abs(T)), e)
            if r.success:
                break
        if r is None or not r.success:
            r = _lp(idx, caps, Rrem, None, e)
        P[e] = min(max(float(r.x[0]), 0.0), float(ub[e])) if r.success else 0.0
        caps[group_of[e]] -= P[e]
        Rrem -= psi[e] * P[e]
        Trem -= P[e]
    return np.clip(P, 0.0, None)


def project_throughput(d, A_h, S_h, Gam_h, K_rem, Cbar_h, avail_h, Rbar_h, chg_time,
                       projection="lexmax"):
    """Debit maximal realisable a un intervalle, affectation A_h (G,E) donnee.

    Renvoie (P (G,Jm,E), U (G,Jm,Am)) ou U est le retrait au stade j-1 qui
    alimente le stade j (Eq 18), preleve en FEFO (le plus vieux d'abord)."""
    G, E = d["G"], d["E"]
    Jm, Am = max(d["Jg"]), int(np.max(d["abar"])) + 1
    tau, psi_p, stage_of = d["tau"], d["psi"], d["stage_of"]

    machines, ub, psis, group_of = [], [], [], []
    group_cap = {}
    for e in range(E):
        gs = [g for g in range(G) if A_h[g, e] > 0.5]
        if not gs:
            continue
        g = gs[0]                                     # Eq (31) : au plus un produit
        j = int(stage_of[g, e])
        if j < 0:
            continue
        t_avail = max(0.0, float(Cbar_h[e]) * float(avail_h[e]) - float(chg_time[e]))
        machines.append((e, g, j))
        ub.append(t_avail / tau[g, j, e])
        psis.append(psi_p[g, j, e])
        group_of.append((g, j))
        if (g, j) not in group_cap:
            group_cap[(g, j)] = (float(K_rem[g]) if j == 0
                                 else float(np.dot(Gam_h[g, j - 1, :], S_h[g, j - 1, :])))

    P = np.zeros((G, Jm, E))
    U = np.zeros((G, Jm, Am))
    if not machines:
        return P, U

    vals = _maxflow(np.array(ub), group_of, group_cap, np.array(psis), float(Rbar_h),
                    projection=projection)
    for (e, g, j), v in zip(machines, vals):
        P[g, j, e] = max(0.0, float(v))

    # --- Eq (18) : retraits FEFO au stade amont ------------------------------
    for (g, j) in set(group_of):
        if j == 0:
            continue
        need = float(P[g, j, :].sum())
        for a in range(int(d["abar"][g, j - 1]), -1, -1):      # a grand = plus vieux
            if need <= TOL:
                break
            avail_a = Gam_h[g, j - 1, a] * S_h[g, j - 1, a] - U[g, j - 1, a]
            take = min(max(avail_a, 0.0), need)
            U[g, j - 1, a] += take
            need -= take
    return P, U


def _changeover(d, A_h, prev_asg):
    """Eq (36) sur l'affectation EXECUTEE. prev_asg (G,E) binaire de l'intervalle
    immediatement precedent (0 partout = machine inactive => aucun changement)."""
    G, E = d["G"], d["E"]
    C = np.zeros((G, G, E))
    t = np.zeros(E)
    cost = 0.0
    for e in range(E):
        gs = [g for g in range(G) if A_h[g, e] > 0.5]
        ps = [g for g in range(G) if prev_asg[g, e] > 0.5]
        if not gs or not ps:
            continue                              # intervalle a vide : pas de changement
        g, gp = gs[0], ps[0]
        if g == gp:
            continue
        C[g, gp, e] = 1.0
        t[e] += float(d["upsilon"][g, gp, e])
        cost += float(d["kappa"][g, gp, e])
    return C, t, cost


def _mask_assignment(d, A_in, avail_h, Wbar_h):
    """Corrections de faisabilite appliquees par l'ENVIRONNEMENT (Penalty de
    l'Eq (46)) : Eq (31) machine indisponible, Eq (38) main-d'oeuvre.
    Ordre d'abandon deterministe : indices machine decroissants."""
    A = A_in.copy()
    G, E = d["G"], d["E"]
    ncorr = 0
    for e in range(E):
        if avail_h[e] < 0.5 and A[:, e].sum() > 0.5:
            ncorr += int(A[:, e].sum())
            A[:, e] = 0.0
    wcap = int(np.floor(float(Wbar_h) + 1e-9))
    while A.sum() > wcap:
        for e in range(E - 1, -1, -1):
            if A[:, e].sum() > 0.5:
                A[:, e] = 0.0
                ncorr += 1
                break
        else:
            break
    return A, ncorr


def _age(d, S_h, P_h, U_h, Gam_h):
    """Eqs (15)-(16) : etat des stocks au debut de l'intervalle suivant."""
    S_next = np.zeros_like(S_h)
    for g in range(d["G"]):
        for j in range(d["Jg"][g]):
            S_next[g, j, 0] = float(P_h[g, j, :].sum())                    # Eq (15)
            for a in range(int(d["abar"][g, j])):                          # Eq (16)
                S_next[g, j, a + 1] = max(
                    0.0, Gam_h[g, j, a] * S_h[g, j, a] - U_h[g, j, a])
    return S_next


def recompute_cost_from_trajectory(d, sol, demand, L0, rho):
    """RECALCUL INDEPENDANT du cout a partir des seules decisions executees.

    Chemin de code volontairement different de epoch_cost() : vectorise sur tout
    l'horizon, et re-derive SL de l'Eq (29) a partir de L et de la demande
    realisee au lieu de le recevoir. Sert au test 4."""
    G, E, N, H = d["G"], d["E"], d["N"], d["H"]
    Jg, abar, Egj = d["Jg"], d["abar"], d["Egj"]

    proc = sum(d["mu"][g, j, e] * sol["P"][g, j, e, :, :].sum()
               for g in range(G) for j in range(Jg[g]) for e in Egj[g][j])
    chg = sum(d["kappa"][g, gp, e] * sol["Ch"][g, gp, e, :, :].sum()
              for g in range(G) for gp in range(G) if gp != g for e in range(E)
              if d["eligible"][g, e] and d["eligible"][gp, e])
    hold = sum(d["alpha"][g, a] * sol["S"][g, j, a, :, :].sum()
               for g in range(G) for j in range(Jg[g]) for a in range(abar[g, j] + 1))
    det = sum(d["delta"][g, j] * sol["Q"][g, j, a, :, :].sum()
              for g in range(G) for j in range(Jg[g]) for a in range(abar[g, j] + 1))
    disp = sum(d["delta"][g, j] * sol["X"][g, j, :, :].sum()
               for g in range(G) for j in range(Jg[g]))
    back = float((d["beta"][:G, None] * sol["L"]).sum())
    rev = float((np.asarray(rho)[:G, :N] * sol["F"]).sum())
    idle = float(d["eta_idle"] * sol["Z"].sum())

    SL = np.zeros((G, N))
    for g in range(G):
        for n in range(N):
            Lp = L0[g] if n == 0 else sol["L"][g, n - 1]
            SL[g, n] = 1.0 - max(sol["L"][g, n] - Lp, 0.0) / (demand[g, n] + d["eps0"])
    serv = float(d["eta_serv"] * np.maximum(d["SLmin"][:G, None] - SL, 0.0).sum())

    G_tot = float(proc + chg + hold + back + det + disp - rev + idle)
    return dict(CostProcessing=float(proc), CostChangeover=float(chg),
                CostHolding=float(hold), CostBacklog=back,
                CostDeterioration=float(det), CostDisposal=float(disp),
                Revenue=rev, CostIdle=idle, operating_cost_G=G_tot,
                service_penalty=serv, policy_cost_J=G_tot + serv), SL


def _empty_solution(d):
    G, E, N, H = d["G"], d["E"], d["N"], d["H"]
    Jm, Am = max(d["Jg"]), int(np.max(d["abar"])) + 1
    return dict(V=np.zeros((G, N)), O=np.zeros((G, N)), K=np.zeros((G, H, N)),
                P=np.zeros((G, Jm, E, H, N)), A=np.zeros((G, E, H, N)),
                Ch=np.zeros((G, G, E, H, N)), S=np.zeros((G, Jm, Am, H, N)),
                U=np.zeros((G, Jm, Am, H, N)), Q=np.zeros((G, Jm, Am, H, N)),
                X=np.zeros((G, Jm, H, N)), St=np.zeros((G, Jm, Am, N)),
                Ud=np.zeros((G, Am, N)), F=np.zeros((G, N)), L=np.zeros((G, N)),
                Z=np.zeros((E, H, N)), DL=np.zeros((G, N)), Sh=np.zeros((G, N)))


# ============================================================================
# CASE C FINAL — débit séquentiel EXACT de l'environnement PPO
# ============================================================================

def project_throughput_ppo_case_c(d, A_h, S_h, Gam_h, K_rem,
                                  Cbar_h, avail_h, Rbar_h, chg_time):
    """Copie fonctionnelle de Env._project_throughput du PPO.

    A_h est déjà corrigée par `_mask_assignment`. L'ordre est important :
    j = 0..Jmax-1, puis e = 0..E-1. C'est le tie-break implicite du PPO.
    """
    G, E = d["G"], d["E"]
    Jm, Am = max(d["Jg"]), int(np.max(d["abar"])) + 1

    cap_time = np.maximum(
        np.asarray(Cbar_h, dtype=float) * np.asarray(avail_h, dtype=float)
        - np.asarray(chg_time, dtype=float), 0.0)

    surv = np.asarray(Gam_h, dtype=float) * np.asarray(S_h, dtype=float)
    # Les cases hors domaine ne doivent jamais alimenter un stade.
    for g in range(G):
        for j in range(Jm):
            if j >= d["Jg"][g]:
                surv[g, j, :] = 0.0
            else:
                ab = int(d["abar"][g, j])
                surv[g, j, ab + 1:] = 0.0

    P = np.zeros((G, Jm, E))
    U = np.zeros((G, Jm, Am))
    rem_shared = float(Rbar_h)
    K_work = np.asarray(K_rem, dtype=float).copy()

    for j in range(Jm):                       # upstream first — PPO exact
        for e in range(E):                    # fixed machine-index order — PPO exact
            gs = [g for g in range(G) if A_h[g, e] > 0.5]
            if not gs:
                continue
            g = gs[0]
            if int(d["stage_of"][g, e]) != j:
                continue

            tau_ = float(d["tau"][g, j, e])
            if tau_ <= 0.0 or tau_ > 1e5:
                continue

            cand = cap_time[e] / tau_
            if j == 0:
                cand = min(cand, K_work[g])
            else:
                cand = min(cand, float(surv[g, j - 1].sum()))

            psi_ = float(d["psi"][g, j, e])
            if psi_ > 0.0:
                cand = min(cand, rem_shared / psi_)

            Pv = max(float(cand), 0.0)
            if Pv <= EPS_NUM:
                continue

            P[g, j, e] = Pv
            cap_time[e] -= tau_ * Pv
            rem_shared -= psi_ * Pv

            if j == 0:
                K_work[g] -= Pv
            else:
                need = Pv
                ab_prev = int(d["abar"][g, j - 1])
                for a in range(ab_prev, -1, -1):       # FEFO — oldest first
                    if need <= EPS_NUM:
                        break
                    take = min(float(surv[g, j - 1, a]), need)
                    U[g, j - 1, a] += take
                    surv[g, j - 1, a] -= take
                    need -= take

    if RUNTIME_CHECKS:
        assert rem_shared >= -1e-6, "PPO shared-resource rule violated"

    return P, U, np.maximum(K_work, 0.0)


In [18]:
def execute_interval_case_c(d_step, S, K, prev_asg, A_plan, obs_h,
                            projection="ppo_exact"):
    """Exécute l'intervalle courant avec la règle exacte du PPO.

    `projection` est conservé uniquement pour compatibilité d'interface ;
    le benchmark Case C utilise toujours le débit séquentiel PPO.
    """
    G, E = d_step["G"], d_step["E"]
    Jm, Am = max(d_step["Jg"]), int(np.max(d_step["abar"])) + 1

    temp = np.array([float(obs_h["temp"])])
    hum = np.array([float(obs_h["hum"])])
    dev = np.array([float(obs_h["dev"])])
    Gam = survival_array(
        d_step, temp, hum, dev,
        np.array([d_step["Delta"][0, 0]]))[:, :, :, 0]

    # Même correction que PPO : disponibilité, puis dépassement workforce
    # en gardant les machines actives d'indice le plus faible.
    A_exec, ncorr = _mask_assignment(
        d_step, np.asarray(A_plan, dtype=float),
        np.asarray(obs_h["avail"], dtype=float),
        float(obs_h["Wbar"]))

    C_h, tchg, _ = _changeover(d_step, A_exec, prev_asg)

    P_h, U_h, K_next = project_throughput_ppo_case_c(
        d_step, A_exec, S, Gam, K,
        d_step["Cbar"][:, 0, 0],
        np.asarray(obs_h["avail"], dtype=float),
        float(obs_h["Rbar"]), tchg)

    Q = np.zeros((G, Jm, Am))
    X = np.zeros((G, Jm))
    Z = np.zeros(E)

    for g in range(G):
        for j in range(d_step["Jg"][g]):
            for a in range(d_step["abar"][g, j] + 1):
                Q[g, j, a] = (1.0 - Gam[g, j, a]) * S[g, j, a]
            ab = int(d_step["abar"][g, j])
            X[g, j] = max(
                0.0, Gam[g, j, ab] * S[g, j, ab] - U_h[g, j, ab])

    for e in range(E):
        proc = float(sum(
            d_step["tau"][g, j, e] * P_h[g, j, e]
            for g in range(G)
            for j in range(d_step["Jg"][g])
            if e in d_step["Egj"][g][j]))
        Z[e] = max(
            0.0,
            float(d_step["Cbar"][e, 0, 0]) * float(obs_h["avail"][e])
            - proc - float(tchg[e]))

    tr = dict(
        A=A_exec, C=C_h, P=P_h, U=U_h, Q=Q, X=X, Z=Z,
        S=S.copy(), K=K.copy(), Gam=Gam, ncorr=int(ncorr))
    return tr, K_next



def interval_cost_case_c(d, tr):
    G, E = d["G"], d["E"]
    proc = chg = hold = det = disp = 0.0
    for g in range(G):
        for j in range(d["Jg"][g]):
            for e in d["Egj"][g][j]:
                proc += d["mu"][g,j,e] * tr["P"][g,j,e]
            for a in range(d["abar"][g,j] + 1):
                hold += d["alpha"][g,a] * tr["S"][g,j,a]
                det += d["delta"][g,j] * tr["Q"][g,j,a]
            disp += d["delta"][g,j] * tr["X"][g,j]
        for gp in range(G):
            if gp == g: continue
            for e in range(E):
                if d["eligible"][g,e] and d["eligible"][gp,e]:
                    chg += d["kappa"][g,gp,e] * tr["C"][g,gp,e]
    idle = float(d["eta_idle"] * np.sum(tr["Z"]))
    return dict(CostProcessing=float(proc), CostChangeover=float(chg),
                CostHolding=float(hold), CostDeterioration=float(det),
                CostDisposal=float(disp), CostIdle=idle)


def close_epoch_case_c(d_ep, S_last, tr_last, L_in, D_real, rho_n):
    """Eqs (22)-(29), après le dernier intervalle seulement."""
    G = d_ep["G"]
    Jm, Am = max(d_ep["Jg"]), int(np.max(d_ep["abar"])) + 1
    Spre = np.zeros((G, Jm, Am)); Gam = tr_last["Gam"]
    for g in range(G):
        for j in range(d_ep["Jg"][g]):
            Spre[g,j,0] = float(tr_last["P"][g,j,:].sum())
            for a in range(d_ep["abar"][g,j]):
                Spre[g,j,a+1] = max(0.0, Gam[g,j,a] * S_last[g,j,a] - tr_last["U"][g,j,a])

    F = np.zeros(G); L = np.zeros(G); Udem = np.zeros((G,Am)); S_out = np.zeros_like(S_last)
    for g in range(G):
        jf = d_ep["Jg"][g] - 1; ab = int(d_ep["abar"][g,jf])
        stock = Spre[g,jf,:ab+1].copy()
        F[g] = min(float(L_in[g] + D_real[g]), float(stock.sum()))
        L[g] = max(0.0, float(L_in[g] + D_real[g] - F[g]))
        need = F[g]
        for a in range(ab, -1, -1):
            take = min(stock[a], need); Udem[g,a] = take; need -= take
            if need <= TOL: break
        for j in range(d_ep["Jg"][g]):
            for a in range(d_ep["abar"][g,j] + 1):
                S_out[g,j,a] = max(0.0, Spre[g,j,a] - (Udem[g,a] if j == jf else 0.0))

    SL = 1.0 - np.maximum(L - L_in, 0.0) / (np.asarray(D_real) + d_ep["eps0"])
    return S_out, dict(Spre=Spre, Udem=Udem, F=F, L=L, SL=SL,
                       CostBacklog=float(np.dot(d_ep["beta"][:G], L)),
                       Revenue=float(np.dot(np.asarray(rho_n), F)),
                       service_penalty=float(d_ep["eta_serv"] * np.maximum(d_ep["SLmin"][:G]-SL,0.0).sum()))


COST_KEYS_CASE_C = ("CostProcessing","CostChangeover","CostHolding","CostBacklog",
                    "CostDeterioration","CostDisposal","Revenue","CostIdle",
                    "operating_cost_G","service_penalty","policy_cost_J")


def rolling_horizon_case_c(sc, planner=FAIR_PLANNER, n_epochs=None,
                           projection="ppo_exact", verbose=False):
    N = INST.N if n_epochs is None else int(n_epochs)
    d_full = build_dims(INST, PRM, N=N, H=INST.H)
    d_full["Cbar"] = PRM.Cbar[:,:,:N].copy(); d_full["Delta"] = PRM.Delta[:,:N].copy(); d_full["rho"] = PRM.rho[:,:N].copy()
    G, E, H = d_full["G"], d_full["E"], d_full["H"]
    S,_ = initial_state(d_full, sc); L = np.zeros(G); prev = np.zeros((G,E))
    sol = _empty_solution(d_full)
    totals = {k:0.0 for k in COST_KEYS_CASE_C}
    journal=[]; plans_by_interval=[]; release_audit=[]; total_seconds=0.0; corrections=0

    for n in range(N):
        V_epoch=None; O_epoch=None; K=np.zeros(G); L_next=L.copy()
        for h in range(H):
            d_rem = case_c_remaining_dims(INST, PRM, n, h)
            view = case_c_observation_view(sc, NOMINAL, INST, n, h)
            Df = forecast_current_epoch(sc, n, G)
            if h == 0:
                rg = release_grid_case_c(sc, n, d_rem, S, L, view["material"][:,0])
                dec = planner.solve_decision(d_rem,view,Df,S,L,prev,PLAN_LIBRARY,
                                             release_grid=rg,tag=f"n={n+1} h={h+1}")
                V_epoch=dec["V"].copy(); O_epoch=dec["O"].copy(); K=V_epoch.copy()
                release_audit.append(dict(epoch=n+1,EIP=rg["EIP"].copy(),V=V_epoch.copy(),
                                          release_index=list(dec["release_index"]),
                                          max_release_grid_gap=rg["max_release_grid_gap"].copy(),
                                          chi_max_abs_error=float(rg["chi_max_abs_error"]),
                                          lex=dict(dec["stats"])))
            else:
                dec = planner.solve_decision(d_rem,view,Df,S,L,prev,PLAN_LIBRARY,
                                             fixed_K=K,fixed_O=O_epoch,tag=f"n={n+1} h={h+1}")
            total_seconds += float(dec["stats"]["seconds"])

            obs_h = dict(avail=sc.avail[:,h,n].copy(),temp=float(sc.temp[h,n]),
                         hum=float(sc.hum[h,n]),dev=float(sc.dev[h,n]),
                         Rbar=float(sc.Rbar[h,n]),Wbar=float(sc.Wbar[h,n]),
                         repair_est=sc.repair_est[:,h,n].copy())
            d_step = build_dims(INST,PRM,N=1,H=1)
            d_step["Cbar"] = PRM.Cbar[:,h:h+1,n:n+1].copy(); d_step["Delta"] = PRM.Delta[h:h+1,n:n+1].copy(); d_step["rho"] = PRM.rho[:,n:n+1].copy()
            tr,K_next = execute_interval_case_c(d_step,S,K,prev,dec["A_current"],obs_h,projection)
            for k,v in interval_cost_case_c(d_step,tr).items(): totals[k] += float(v)
            corrections += tr["ncorr"]

            sol["K"][:,h,n]=K; sol["P"][:,:,:,h,n]=tr["P"]; sol["A"][:,:,h,n]=tr["A"]
            sol["Ch"][:,:,:,h,n]=tr["C"]; sol["S"][:,:,:,h,n]=tr["S"]; sol["U"][:,:,:,h,n]=tr["U"]
            sol["Q"][:,:,:,h,n]=tr["Q"]; sol["X"][:,:,h,n]=tr["X"]; sol["Z"][:,h,n]=tr["Z"]

            pid_exec=plan_id_of(tr["A"])
            plans_by_interval.append(dict(epoch=n+1,interval=h+1,iota=n*H+h,
                                          chosen_plan_id=int(dec["plan_id"]),
                                          executed_plan_id=None if pid_exec is None else int(pid_exec),
                                          release_decision=bool(h==0),K_before=K.copy(),K_after=K_next.copy(),
                                          repair_est=np.asarray(obs_h["repair_est"]).copy(),
                                          primary_delta=float(dec["stats"]["primary_delta"]),
                                          unused_release_before=float(dec["stats"]["unused_release_before"]),
                                          unused_release_after=float(dec["stats"]["unused_release_after"])))

            if h < H-1:
                S_next = _age(d_step,S,tr["P"],tr["U"],tr["Gam"])
            else:
                d_ep=build_dims(INST,PRM,N=1,H=H); d_ep["Cbar"]=PRM.Cbar[:,:,n:n+1].copy(); d_ep["Delta"]=PRM.Delta[:,n:n+1].copy(); d_ep["rho"]=PRM.rho[:,n:n+1].copy()
                S_next,end=close_epoch_case_c(d_ep,S,tr,L,sc.demand[:G,n],PRM.rho[:G,n])
                totals["CostBacklog"] += end["CostBacklog"]; totals["Revenue"] += end["Revenue"]; totals["service_penalty"] += end["service_penalty"]
                sol["St"][:,:,:,n]=end["Spre"]; sol["Ud"][:,:,n]=end["Udem"]; sol["F"][:,n]=end["F"]; sol["L"][:,n]=end["L"]
                sol["DL"][:,n]=np.maximum(end["L"]-L,0.0); sol["Sh"][:,n]=np.maximum(d_ep["SLmin"][:G]-end["SL"],0.0)
                L_next=end["L"].copy()

            journal.append(dict(epoch=n+1,interval=h+1,plan_id=int(dec["plan_id"]),
                                release_decision=bool(h==0),executed_plan_id=pid_exec,
                                repair_est_max=float(np.max(obs_h["repair_est"])),
                                lex_primary_delta=float(dec["stats"]["primary_delta"]),
                                lex_unused_before=float(dec["stats"]["unused_release_before"]),
                                lex_unused_after=float(dec["stats"]["unused_release_after"]),
                                corrections=int(tr["ncorr"])))
            S,K,prev=S_next,K_next,tr["A"].copy()

        sol["V"][:,n]=V_epoch; sol["O"][:,n]=O_epoch; L=L_next

    totals["operating_cost_G"] = float(totals["CostProcessing"]+totals["CostChangeover"]+totals["CostHolding"]+totals["CostBacklog"]+totals["CostDeterioration"]+totals["CostDisposal"]+totals["CostIdle"]-totals["Revenue"])
    totals["policy_cost_J"] = float(totals["operating_cost_G"]+totals["service_penalty"])
    stats=dict(label="CPLEX Fair Baseline Case C",status="executed",seconds=float(total_seconds),
               decisions=int(N*H),releases=int(N),plan_reoptimizations=int(N*H),
               env_corrections=int(corrections),plans_by_interval=plans_by_interval,
               release_audit=release_audit)
    return sol,totals,stats,d_full,pd.DataFrame(journal)


In [19]:
def make_future_twin_case_c(sc):
    """Copie identique à (n=0,h=0), différente uniquement dans le futur."""
    B = copy.deepcopy(sc)
    E,H,N = B.avail.shape
    flat_a = _flatten_ehn_time(B.avail); flat_r = _flatten_ehn_time(B.repair_est)
    flat_a[:,1:] = 1.0; flat_a[0,1:] = 0.0
    flat_r[:,1:] = 0.0; flat_r[0,1:] = REPAIR_CLIP
    B.avail = _unflatten_ehn_time(flat_a,H,N); B.repair_est = _unflatten_ehn_time(flat_r,H,N)
    tt=B.temp.T.reshape(-1); hh=B.hum.T.reshape(-1); dd=B.dev.T.reshape(-1)
    rr=B.Rbar.T.reshape(-1); ww=B.Wbar.T.reshape(-1)
    tt[1:]+=15.0; hh[1:]+=20.0; dd[1:]=1.0; rr[1:]*=0.4; ww[1:]=1.0
    B.temp=tt.reshape(N,H).T; B.hum=hh.reshape(N,H).T; B.dev=dd.reshape(N,H).T
    B.Rbar=rr.reshape(N,H).T; B.Wbar=ww.reshape(N,H).T
    B.demand[:,1:] *= 3.0
    return B


def test_nonanticipativity_structural(sc):
    d=case_c_remaining_dims(INST,PRM,0,0); S0,_=initial_state(d,sc)
    L0=np.zeros(d["G"]); K0=np.zeros(d["G"]); prev=np.zeros((d["G"],d["E"]))
    B=make_future_twin_case_c(sc)
    p1=nonanticipative_payload(case_c_observation_view(sc,NOMINAL,INST,0,0),forecast_current_epoch(sc,0,d["G"]),S0,L0,K0,prev)
    p2=nonanticipative_payload(case_c_observation_view(B,NOMINAL,INST,0,0),forecast_current_epoch(B,0,d["G"]),S0,L0,K0,prev)
    return json.dumps(p1,sort_keys=True) == json.dumps(p2,sort_keys=True)


def test_repair_est_structural():
    """Vérifie la sémantique PPO : entier >=1 pendant la panne, 0 sinon,
    et observation plafonnée à REPAIR_CLIP."""
    avail = np.ones((1, 3, 4), dtype=float)
    # six intervalles consécutifs en panne dans l'ordre n->h
    flat = avail.transpose(0, 2, 1).reshape(1, -1)
    flat[0, 2:8] = 0.0
    avail = flat.reshape(1, 4, 3).transpose(0, 2, 1)
    est = build_repair_estimate_case_c(avail, np.random.default_rng(123), cv=0.30)

    up_ok = np.all(est[avail > 0.5] == 0.0)
    down = est[avail < 0.5]
    integer_ok = np.all(down >= 1.0) and np.allclose(down, np.round(down))

    sc = copy.deepcopy(CASE_C_EVAL_SCENARIOS[0])
    sc.avail[0, 0, 0] = 0.0
    sc.repair_est[0, 0, 0] = REPAIR_CLIP + 5.0
    v = case_c_observation_view(sc, NOMINAL, INST, 0, 0)
    clip_ok = abs(v["repair_est_obs"][0] - REPAIR_CLIP) <= 1e-12
    influence_ok = np.all(v["avail"][0, :min(INST.H, int(REPAIR_CLIP)), 0] == 0.0)

    return bool(up_ok and integer_ok and clip_ok and influence_ok)



def recompute_changeovers_case_c(d,sol):
    prev=np.zeros((d["G"],d["E"])); worst=0.0
    for n in range(d["N"]):
        for h in range(d["H"]):
            C,_,_=_changeover(d,sol["A"][:,:,h,n],prev)
            worst=max(worst,float(np.max(np.abs(C-sol["Ch"][:,:,:,h,n]))))
            prev=sol["A"][:,:,h,n]
    return worst


def run_case_c_tests(sc_routine, sol_r, tot_r, st_r, d_r,
                     sc_dis=None, sol_d=None, tot_d=None, st_d=None, d_d=None):
    results = {}

    results["1_no_future_information"] = bool(
        test_nonanticipativity_structural(sc_routine))

    pbi = st_r["plans_by_interval"]
    results["2_plan_each_interval"] = bool(
        len(pbi) == d_r["N"] * d_r["H"]
        and st_r["plan_reoptimizations"] == d_r["N"] * d_r["H"])

    expected = [(i % d_r["H"]) == 0 for i in range(len(pbi))]
    results["3_release_only_epoch_start"] = bool(
        [x["release_decision"] for x in pbi] == expected)

    results["4_repair_est_used"] = bool(test_repair_est_structural())

    # Le test 5 n'est plus "CPLEX respecte sa propre library".
    # Il vérifie la library CONTRE le PPO de référence.
    executed_in_library = all(
        x["executed_plan_id"] is not None for x in pbi)
    chosen_in_library = all(
        0 <= x["chosen_plan_id"] < len(PLAN_LIBRARY) for x in pbi)
    results["5_plan_library"] = bool(
        PPO_ACTION_SPACE_MATCH
        and PPO_N_ROT == 54
        and PPO_FULL_ACTION_COUNT == 864
        and len(PLAN_LIBRARY) == 89
        and executed_in_library
        and chosen_in_library)

    results["6_changeovers"] = bool(
        recompute_changeovers_case_c(d_r, sol_r) <= 1e-9)

    rec, _ = recompute_cost_from_trajectory(
        d_r, sol_r,
        sc_routine.demand[:d_r["G"], :d_r["N"]],
        np.zeros(d_r["G"]),
        PRM.rho[:d_r["G"], :d_r["N"]])
    cost_err = max(abs(rec[k] - tot_r[k]) for k in rec)
    results["7_cost_recomputed_1e-6"] = bool(cost_err <= 1e-6)

    accessible = True
    for a in st_r["release_audit"]:
        _tol = 1.0
        if abs(a["lex"]["primary_delta"]) > _tol + 1e-7:
            accessible = False
        if a["lex"]["unused_release_after"] > a["lex"]["unused_release_before"] + 1e-6:
            accessible = False
    results["8_release_grid_and_lexicographic_V"] = bool(
        accessible and len(st_r["release_audit"]) == d_r["N"])

    finite_r = (
        all(np.isfinite(a).all() for a in sol_r.values()
            if isinstance(a, np.ndarray))
        and all(np.isfinite(float(v)) for v in tot_r.values()))
    finite_d = True
    if sol_d is not None:
        finite_d = (
            all(np.isfinite(a).all() for a in sol_d.values()
                if isinstance(a, np.ndarray))
            and all(np.isfinite(float(v)) for v in tot_d.values()))
    results["9_no_nan"] = bool(finite_r and finite_d)

    # Audits PPO↔CPLEX indépendants des solves.
    results["reference_ppo_actions_match"] = bool(PPO_ACTION_SPACE_MATCH)
    results["reference_ppo_routine24_match"] = bool(PPO_REFERENCE_ROUTINE_MATCH)
    results["reference_ppo_stress180_match"] = bool(PPO_REFERENCE_STRESS_MATCH)
    results["cost_recompute_max_error"] = float(cost_err)

    numbered_ok = all(v for k, v in results.items() if k[:1].isdigit())
    reference_ok = (
        results["reference_ppo_actions_match"]
        and results["reference_ppo_routine24_match"]
        and results["reference_ppo_stress180_match"])
    results["all_pass"] = bool(numbered_ok and reference_ok)
    return results



RUN_CASE_C_PREFLIGHT = True
PREFLIGHT_EPOCHS = 2
PREFLIGHT_RESULTS = None
CASE_C_TEST_RESULTS = None

if RUN_CASE_C_PREFLIGHT:
    print("="*78); print("PREFLIGHT CASE C — 1 routinier + 1 perturbation,",PREFLIGHT_EPOCHS,"époques chacun"); print("="*78)
    SC_PREF_R=CASE_C_EVAL_SCENARIOS[0]
    SOL_PREF_R,TOT_PREF_R,ST_PREF_R,D_PREF_R,JOU_PREF_R=rolling_horizon_case_c(SC_PREF_R,n_epochs=PREFLIGHT_EPOCHS,verbose=False)
    _stress_pre=CaseCScenarioGenerator(INST,PRM,NOMINAL,STRESS_SEED)
    SC_PREF_D=_stress_pre.sample_disruption("line_failure")
    SOL_PREF_D,TOT_PREF_D,ST_PREF_D,D_PREF_D,JOU_PREF_D=rolling_horizon_case_c(SC_PREF_D,n_epochs=PREFLIGHT_EPOCHS,verbose=False)
    CASE_C_TEST_RESULTS=run_case_c_tests(SC_PREF_R,SOL_PREF_R,TOT_PREF_R,ST_PREF_R,D_PREF_R,SC_PREF_D,SOL_PREF_D,TOT_PREF_D,ST_PREF_D,D_PREF_D)
    print(pd.Series(CASE_C_TEST_RESULTS))
    if not CASE_C_TEST_RESULTS["all_pass"]:
        raise AssertionError("Au moins un test Case C a échoué. Ne pas lancer les campagnes.")
    PREFLIGHT_RESULTS=dict(routine_cost=float(TOT_PREF_R["policy_cost_J"]),disruption_cost=float(TOT_PREF_D["policy_cost_J"]))
else:
    print("Préflight désactivé; aucun solve Case C lancé.")


def summary_case_c(sol,d,sc,tot,stats,scenario_id):
    demand=sc.demand[:d["G"],:d["N"]]; SL=service_levels(d,demand,np.zeros(d["G"]),sol["L"])
    return pd.DataFrame([dict(scenario=scenario_id,kind=sc.kind,policy_cost_J=float(tot["policy_cost_J"]),
                              operating_cost_G=float(tot["operating_cost_G"]),service_penalty=float(tot["service_penalty"]),
                              mean_service_level=float(np.mean(SL)),min_service_level=float(np.min(SL)),
                              total_release=float(sol["V"].sum()),total_fulfilled=float(sol["F"].sum()),
                              total_backlog=float(sol["L"].sum()),expired_units=float(sol["X"].sum()),
                              deteriorated_units=float(sol["Q"].sum()),changeovers=float(sol["Ch"].sum()),
                              plan_reoptimizations=int(stats["plan_reoptimizations"]),cplex_seconds=float(stats["seconds"]))])


def long_case_c(sol,d,sc,tot,stats,scenario_id):
    rows=[]
    for n in range(d["N"]):
        for g in range(d["G"]):
            rows.append(dict(scenario=scenario_id,kind=sc.kind,epoch=n+1,product=g+1,
                             release=float(sol["V"][g,n]),fulfilled=float(sol["F"][g,n]),
                             backlog=float(sol["L"][g,n]),expired=float(sol["X"][g,:,:,n].sum()),
                             deteriorated=float(sol["Q"][g,:,:,:,n].sum()),policy_cost_J=float(tot["policy_cost_J"])))
    return pd.DataFrame(rows)


def run_routine_24_case_c():
    if not RUN_ROUTINE_24:
        print("RUN_ROUTINE_24=False — campagne routinière non lancée."); return None,None
    if CASE_C_TEST_RESULTS is not None and not CASE_C_TEST_RESULTS["all_pass"]: raise RuntimeError("Tests Case C non verts.")
    sums=[]; longs=[]
    for i,sc in enumerate(CASE_C_EVAL_SCENARIOS):
        print(f"routine {i+1}/{N_ROUTINE}")
        sol,tot,st,d,_=rolling_horizon_case_c(sc,verbose=False); sid=f"routine_{i:02d}"
        sums.append(summary_case_c(sol,d,sc,tot,st,sid)); longs.append(long_case_c(sol,d,sc,tot,st,sid))
        pd.concat(sums,ignore_index=True).to_csv("cplex_case_c_results_summary.csv",index=False)
        pd.concat(longs,ignore_index=True).to_csv("cplex_case_c_results_long.csv",index=False)
    return pd.concat(sums,ignore_index=True),pd.concat(longs,ignore_index=True)


def run_disruptions_180_case_c():
    if not RUN_DISRUPTIONS_180:
        print("RUN_DISRUPTIONS_180=False — campagne perturbations non lancée."); return None,None
    if CASE_C_TEST_RESULTS is not None and not CASE_C_TEST_RESULTS["all_pass"]: raise RuntimeError("Tests Case C non verts.")
    gen=CaseCScenarioGenerator(INST,PRM,NOMINAL,STRESS_SEED); sums=[]; longs=[]
    for fam in DISRUPTION_FAMILIES:
        for k in range(N_PER_FAMILY):
            print(f"{fam} {k+1}/{N_PER_FAMILY}")
            sc=gen.sample_disruption(fam); sol,tot,st,d,_=rolling_horizon_case_c(sc,verbose=False); sid=f"{fam}_{k:02d}"
            sums.append(summary_case_c(sol,d,sc,tot,st,sid)); longs.append(long_case_c(sol,d,sc,tot,st,sid))
            pd.concat(sums,ignore_index=True).to_csv("cplex_case_c_disruptions_summary.csv",index=False)
            pd.concat(longs,ignore_index=True).to_csv("cplex_case_c_disruptions_long.csv",index=False)
    return pd.concat(sums,ignore_index=True),pd.concat(longs,ignore_index=True)


print("RUN_ROUTINE_24 =",RUN_ROUTINE_24)
print("RUN_DISRUPTIONS_180 =",RUN_DISRUPTIONS_180)
print("Aucune campagne 24/180 n'est lancée par défaut.")


PREFLIGHT CASE C — 1 routinier + 1 perturbation, 2 époques chacun
CPLEX 22.2.0:   lim:time = 90
  mip:gap = 9.9999999999999995e-07
  mip:return_gap = 3

suffix absmipgap OUT;
suffix relmipgap OUT;
CPLEX 22.2.0:   lim:time = 90
  mip:gap = 9.9999999999999995e-07
  mip:return_gap = 3
CPLEX 22.2.0:   lim:time = 90
  mip:gap = 9.9999999999999995e-07
  mip:return_gap = 3

suffix absmipgap OUT;
suffix relmipgap OUT;
CPLEX 22.2.0:   lim:time = 90
  mip:gap = 9.9999999999999995e-07
  mip:return_gap = 3
CPLEX 22.2.0:   lim:time = 90
  mip:gap = 9.9999999999999995e-07
  mip:return_gap = 3

suffix absmipgap OUT;
suffix relmipgap OUT;
CPLEX 22.2.0:   lim:time = 90
  mip:gap = 9.9999999999999995e-07
  mip:return_gap = 3
CPLEX 22.2.0:   lim:time = 90
  mip:gap = 9.9999999999999995e-07
  mip:return_gap = 3

suffix absmipgap OUT;
suffix relmipgap OUT;
CPLEX 22.2.0:   lim:time = 90
  mip:gap = 9.9999999999999995e-07
  mip:return_gap = 3
CPLEX 22.2.0:   lim:time = 90
  mip:gap = 9.9999999999999995e-07
 

In [20]:
RUN_DISRUPTIONS_180 = True

DISR_SUMMARY, DISR_LONG = run_disruptions_180_case_c()

display(DISR_SUMMARY)

line_failure 1/60
CPLEX 22.2.0:   lim:time = 90
  mip:gap = 9.9999999999999995e-07
  mip:return_gap = 3

suffix absmipgap OUT;
suffix relmipgap OUT;
CPLEX 22.2.0:   lim:time = 90
  mip:gap = 9.9999999999999995e-07
  mip:return_gap = 3
CPLEX 22.2.0:   lim:time = 90
  mip:gap = 9.9999999999999995e-07
  mip:return_gap = 3

suffix absmipgap OUT;
suffix relmipgap OUT;
CPLEX 22.2.0:   lim:time = 90
  mip:gap = 9.9999999999999995e-07
  mip:return_gap = 3
CPLEX 22.2.0:   lim:time = 90
  mip:gap = 9.9999999999999995e-07
  mip:return_gap = 3

suffix absmipgap OUT;
suffix relmipgap OUT;
CPLEX 22.2.0:   lim:time = 90
  mip:gap = 9.9999999999999995e-07
  mip:return_gap = 3
CPLEX 22.2.0:   lim:time = 90
  mip:gap = 9.9999999999999995e-07
  mip:return_gap = 3

suffix absmipgap OUT;
suffix relmipgap OUT;
CPLEX 22.2.0:   lim:time = 90
  mip:gap = 9.9999999999999995e-07
  mip:return_gap = 3
CPLEX 22.2.0:   lim:time = 90
  mip:gap = 9.9999999999999995e-07
  mip:return_gap = 3

suffix absmipgap OUT;
suffi

,scenario,kind,policy_cost_J,operating_cost_G,service_penalty,mean_service_level,min_service_level,total_release,total_fulfilled,total_backlog,expired_units,deteriorated_units,changeovers,plan_reoptimizations,cplex_seconds
0,line_failure_00,line_failure,1142.537,1128.618,13.919,0.798,8.717e-09,8644.603,7790.934,4077.296,0.000,154.262,155.0,84,13.941
1,line_failure_01,line_failure,25717.682,25701.390,16.292,0.772,6.649e-09,9306.963,7813.016,7847.437,0.000,187.651,141.0,84,14.480
2,line_failure_02,line_failure,14136.120,14123.539,12.581,0.821,4.783e-09,9940.892,8374.980,6198.328,0.000,195.646,169.0,84,14.210
3,line_failure_03,line_failure,148681.587,148655.792,25.795,0.646,5.582e-09,12230.521,8159.182,27506.527,224.646,418.146,138.0,84,15.845
4,line_failure_04,line_failure,3497.675,3486.250,11.425,0.835,6.269e-09,8575.744,7703.425,3553.493,2.618,178.311,162.0,84,14.488
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
175,cold_chain_55,cold_chain,-787.801,-798.436,10.635,0.841,4.938e-09,8600.725,7946.155,2999.740,0.000,242.433,175.0,84,12.790
176,cold_chain_56,cold_chain,3737.954,3726.868,11.086,0.833,6.323e-09,9827.239,8729.335,4361.884,0.000,245.922,178.0,84,13.333
177,cold_chain_57,cold_chain,-261.547,-269.421,7.874,0.869,6.113e-09,9400.219,8538.435,3522.003,0.000,247.605,172.0,84,15.295
178,cold_chain_58,cold_chain,7639.685,7625.095,14.589,0.791,6.306e-09,9978.216,8636.464,5048.799,0.000,251.101,155.0,84,14.675


In [21]:
sol, tot, st, d, _ = rolling_horizon_case_c(CASE_C_EVAL_SCENARIOS[16], verbose=False)
for k, v in tot.items():
    print(f"{k:25s} {v:15,.1f}")
print("-" * 42)
print(f"{'TOTAL':25s} {sum(tot.values()):15,.1f}")

CPLEX 22.2.0:   lim:time = 90
  mip:gap = 9.9999999999999995e-07
  mip:return_gap = 3

suffix absmipgap OUT;
suffix relmipgap OUT;
CPLEX 22.2.0:   lim:time = 90
  mip:gap = 9.9999999999999995e-07
  mip:return_gap = 3
CPLEX 22.2.0:   lim:time = 90
  mip:gap = 9.9999999999999995e-07
  mip:return_gap = 3

suffix absmipgap OUT;
suffix relmipgap OUT;
CPLEX 22.2.0:   lim:time = 90
  mip:gap = 9.9999999999999995e-07
  mip:return_gap = 3
CPLEX 22.2.0:   lim:time = 90
  mip:gap = 9.9999999999999995e-07
  mip:return_gap = 3

suffix absmipgap OUT;
suffix relmipgap OUT;
CPLEX 22.2.0:   lim:time = 90
  mip:gap = 9.9999999999999995e-07
  mip:return_gap = 3
CPLEX 22.2.0:   lim:time = 90
  mip:gap = 9.9999999999999995e-07
  mip:return_gap = 3

suffix absmipgap OUT;
suffix relmipgap OUT;
CPLEX 22.2.0:   lim:time = 90
  mip:gap = 9.9999999999999995e-07
  mip:return_gap = 3
CPLEX 22.2.0:   lim:time = 90
  mip:gap = 9.9999999999999995e-07
  mip:return_gap = 3

suffix absmipgap OUT;
suffix relmipgap OUT;
C

In [22]:
print(DISR_SUMMARY[["policy_cost_J", "mean_service_level"]].mean())

import shutil
shutil.copy("cplex_case_c_disruptions_summary.csv", f"cplex_disruptions_{SEVERITY}_summary.csv")
shutil.copy("cplex_case_c_disruptions_long.csv",    f"cplex_disruptions_{SEVERITY}_long.csv")
print("saved:", f"cplex_disruptions_{SEVERITY}_summary.csv")

policy_cost_J         11217.552
mean_service_level        0.805
dtype: float64
saved: cplex_disruptions_moderate_summary.csv
